# Zotero PDF Data Extraction Pipeline

This notebook implements an end-to-end workflow for extracting structured nanomaterial, physicochemical, and toxicity data from PDFs stored in a Zotero collection. It retrieves PDFs, repairs extracted text when needed, identifies assay materials, extracts figures/tables and captions, selects evidence candidates, runs LLM-assisted extraction, consolidates outputs, and supports batch execution with reruns.

**Before running:** complete the required user inputs in Step 1, especially `selected_collection`, `project_name`, Zotero library credentials, and the OpenAI API key. Do not commit real API keys or private credentials to a public repository.


# Step 1: Runtime configuration and project setup

Initialize the notebook timer, set project metadata, define credentials, configure model settings, and create output directories.

## Step 1.1: Notebook run timer

Track overall runtime and per-section elapsed time for reproducibility and reporting.

In [ ]:
# Step 1.1: Define notebook runtime tracking helpers
import time as _pipeline_time_mod
from datetime import datetime as _pipeline_datetime_cls
from pytz import timezone as _pipeline_timezone

_PIPELINE_TIMER_TZ = _pipeline_timezone("Asia/Seoul")

# Overall run start time
_PIPELINE_RUN_START_DT = _pipeline_datetime_cls.now(_PIPELINE_TIMER_TZ)
_PIPELINE_RUN_START_TS = _pipeline_time_mod.perf_counter()

# Per-section start times and results
_PIPELINE_SECTION_RESULTS = []

def _pipeline_timer_print_summary():
    """
    Print a summary of the overall run and each timed section.
    """
    run_end_dt = _pipeline_datetime_cls.now(_PIPELINE_TIMER_TZ)
    run_end_ts = _pipeline_time_mod.perf_counter()
    run_elapsed_sec = run_end_ts - _PIPELINE_RUN_START_TS

    run_h = int(run_elapsed_sec // 3600)
    run_m = int((run_elapsed_sec % 3600) // 60)
    run_s = run_elapsed_sec % 60

    print("\n===== Pipeline Runtime Summary =====")
    print("Overall start :", _PIPELINE_RUN_START_DT.strftime("%Y-%m-%d %H:%M:%S %Z"))
    print("Overall end   :", run_end_dt.strftime("%Y-%m-%d %H:%M:%S %Z"))
    print(
        "Overall elapsed : "
        f"{run_h:02d}:{run_m:02d}:{run_s:06.3f} "
        f"({round(run_elapsed_sec, 3)} sec)"
    )

    if not _PIPELINE_SECTION_RESULTS:
        print("No section timing records found.")
        return

    print("\n===== Section Runtime Summary =====")
    for row in _PIPELINE_SECTION_RESULTS:
        sec = row["elapsed_sec"]
        hh = int(sec // 3600)
        mm = int((sec % 3600) // 60)
        ss = sec % 60

        print(
            f"{row['section_label']}\n"
            f"  Start   : {row['start_dt'].strftime('%Y-%m-%d %H:%M:%S %Z')}\n"
            f"  End     : {row['end_dt'].strftime('%Y-%m-%d %H:%M:%S %Z')}\n"
            f"  Elapsed : {hh:02d}:{mm:02d}:{ss:06.3f} ({round(sec, 3)} sec)"
        )

print("[Pipeline Timer] Overall start :", _PIPELINE_RUN_START_DT.strftime("%Y-%m-%d %H:%M:%S %Z"))

## Step 1.2: Project metadata

Set user-facing project metadata used in output folder names and run logs.

In [ ]:
# Step 1.2: Define project metadata for output folders and logs
# Example: user_name = "User"
user_name = "User"

# Short project label used in generated output paths.
# Example: project_name = "Nanotoxicity_Zotero_Extraction"
project_name = "GPT-5.4"

## Step 1.3: Imports, credentials, model settings, and paths

Load dependencies and define user-provided credentials, model options, and local directories. Replace placeholder values before execution.

### Required user inputs

| Variable | What to enter | Example placeholder | Notes |
|---|---|---|---|
| `selected_collection` | Exact Zotero collection name to process | `"Oxide_A2018_5_5"` | Use the collection name as it appears in the target Zotero library. |
| `ZOTERO_USER_ID` | Zotero library ID used by Pyzotero | `"1234567"` | This notebook is currently configured for a **group** library in Step 2.1, so enter the Zotero **group library ID**, not the group name. For a personal library, use your personal userID and adapt the library type in Step 2.1 from `"group"` to `"user"`. |
| `ZOTERO_API` | Zotero API key | `"xxxxxxxxxxxxxxxxxxxxxxxx"` | Create a key with read access to the target library. File/PDF access must be allowed if the PDFs are private or stored in Zotero storage. |
| `OPENAI_API_KEY` | OpenAI API key | `"sk-..."` | For public repositories, prefer loading this from an environment variable instead of committing a real key. |

Security note: leave the placeholders empty in committed code and provide credentials only during local execution.


In [ ]:
# Step 1.3: Import dependencies and configure credentials
import os
import json
import re
import fitz  # PyMuPDF
import pandas as pd
from pyzotero import zotero
from openai import OpenAI
from datetime import datetime
from pytz import timezone

# Step 1.3a: Set the target Zotero collection name
# Use the exact collection name as it appears in the target Zotero library.
# Example: selected_collection = "Oxide_A2018_5_5"
selected_collection = ""

# Step 1.3b: Set Zotero library credentials
# This notebook currently connects to a Zotero GROUP library in Step 2.1:
#     zotero.Zotero(ZOTERO_USER_ID, "group", ZOTERO_API)
# Therefore, enter the Zotero group library ID here, not the group name.
# Example: ZOTERO_USER_ID = "1234567"
#
# For a personal Zotero library, enter your personal userID here and adapt
# the Step 2.1 Pyzotero call from library_type="group" to library_type="user".
ZOTERO_USER_ID = ""

# Zotero API key with read access to the target library.
# File/PDF access is required when downloading private or Zotero-stored PDFs.
# Example: ZOTERO_API = "xxxxxxxxxxxxxxxxxxxxxxxx"
ZOTERO_API = ""

# Step 1.3c: Set the OpenAI API key
# Example: OPENAI_API_KEY = "sk-..."
OPENAI_API_KEY = ""


# LLM model options
CHAT_MODEL_NAME = "gpt-5.4-2026-03-05"
#CHAT_MODEL_NAME = "claude-sonnet-4-6"
#CHAT_MODEL_NAME = "gemini-3.1-pro-preview"
#CHAT_MODEL_NAME = "grok-4.20-0309-reasoning"

CHAT_TEMPERATURE = 0

# Path and folder options
# All save/read/output paths use the variables below.
now = datetime.now(timezone("Asia/Seoul")).strftime("%Y%m%d_%H%M%S")

pdf_global = "../PDF/"
output_dir = os.path.join("../output", f"{selected_collection}_{project_name}_{now}")
pdf_folder = os.path.abspath(os.path.join(pdf_global, selected_collection.replace(" ", "_")))
img_dir = os.path.join("../image/", f"{selected_collection}_{project_name}_{now}")
tmp_dir = os.path.join("../tmp/", f"{selected_collection}_{project_name}_{now}")

os.makedirs(pdf_folder, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)
os.makedirs(img_dir, exist_ok=True)
os.makedirs(tmp_dir, exist_ok=True)

client = OpenAI(api_key=OPENAI_API_KEY)

# Step 2: Zotero PDF retrieval and text preprocessing

Connect to Zotero, download one PDF per eligible item, extract text, repair suspected text artifacts, and isolate the Methods section.

## Step 2.1: Retrieve PDFs from Zotero

Fetch parent items from the selected Zotero collection and download a single PDF attachment for each eligible paper.

The Pyzotero connection below is configured for a Zotero **group** library. In this notebook, `ZOTERO_USER_ID` is used as the Zotero group library ID. If adapting the notebook to a personal library, change the Pyzotero library type from `"group"` to `"user"` and set `ZOTERO_USER_ID` to the personal userID.


In [ ]:
# Step 2.1: Retrieve and download Zotero PDFs
def get_all_pdfs_from_zotero(collection_name):
    # Current workflow setting: Zotero group library.
    # ZOTERO_USER_ID should contain the group library ID when library_type is "group".
    # For a personal library, adapt the second argument to "user" and provide the personal userID.
    zot = zotero.Zotero(ZOTERO_USER_ID, "group", ZOTERO_API)

    collections = {c["data"]["name"]: c for c in zot.collections()}
    if collection_name not in collections:
        raise ValueError(f"Collection not found: {collection_name}")

    collection_key = collections[collection_name]["key"]

    # Retrieve all items from the selected collection
    items = zot.everything(zot.collection_items(collection_key))

    papers = []

    for item in items:
        # Skip attachment items themselves
        if item["data"].get("itemType") == "attachment":
            continue

        title = item["data"].get("title", "No Title")

        try:
            children = list(zot.children(item["key"]))
            pdfs = [c for c in children if c["data"].get("contentType") == "application/pdf"]

            # Skip items without PDFs
            if len(pdfs) == 0:
                print(f"[SKIP] No PDF: {title}")
                continue

            # Skip items with more than one PDF (preserve the original workflow rule)
            if len(pdfs) != 1:
                print(f"[SKIP] Multiple PDFs found: {title}")
                continue

            pdf_item = pdfs[0]
            pdf_key = pdf_item["key"]
            pdf_path = os.path.join(pdf_folder, f"{pdf_key}.pdf")

            # Skip download if the file already exists
            already_exists = os.path.exists(pdf_path)
            if already_exists:
                print(f"[SKIP DOWNLOAD] Already exists: {pdf_path}")
            else:
                zot.dump(pdf_key, pdf_path, pdf_folder)
                print(f"[DOWNLOADED] {pdf_path}")

            papers.append({
                "item_key": item["key"],
                "title": title,
                "pdf_key": pdf_key,
                "pdf_path": pdf_path,
                "already_exists": already_exists,
            })

        except Exception as e:
            print(f"[SKIP] {title} / {e}")

    if not papers:
        raise FileNotFoundError("No PDF papers found in the collection.")

    print(f"Collected {len(papers)} PDF papers from collection: {collection_name}")
    return papers

## Step 2.2: Extract full text from PDFs

Read each PDF page with PyMuPDF and preserve page-level markers for downstream grounding.

In [ ]:
# Step 2.2: Extract page-level full text from PDFs
def extract_full_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    page_texts = []

    for i, page in enumerate(doc):
        txt = page.get_text("text")
        if txt:
            txt = txt.replace("\x00", " ").strip()
            txt = re.sub(r"[ \t]+", " ", txt)
            page_texts.append(f"\n[Page {i+1}]\n{txt}\n")

    full_text = "\n".join(page_texts).strip()
    return full_text

## Step 2.3: Repair suspected text-extraction artifacts

Detect suspicious text chunks and apply minimal LLM-assisted repair only where needed.

In [ ]:
# Step 2.3: Repair suspected text-extraction artifacts
import unicodedata
from typing import List, Dict, Tuple

TEXT_REPAIR_SCHEMA = {
    "name": "suspect_text_repair",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "repairs": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "span_id": {"type": "integer"},
                        "repaired_text": {"type": "string"},
                        "changed": {
                            "type": "string",
                            "enum": ["Yes", "No"]
                        },
                        "confidence": {
                            "type": "string",
                            "enum": ["High", "Medium", "Low"]
                        }
                    },
                    "required": ["span_id", "repaired_text", "changed", "confidence"]
                }
            }
        },
        "required": ["repairs"]
    }
}


def normalize_text_deterministic(text: str) -> str:
    """
    Apply safe normalization only when it does not change meaning.
    """
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "\x00": " ",
        "\xad": "",   # soft hyphen
        "ﬁ": "fi",
        "ﬂ": "fl",
        "ﬀ": "ff",
        "ﬃ": "ffi",
        "ﬄ": "ffl",
        "−": "-",
        "–": "-",
        "—": "-",
        "μ": "µ",     # Standardize the micro sign
        "㎍": "µg",
        "㎖": "mL",
    }

    for src, dst in replacements.items():
        text = text.replace(src, dst)

    # Join line-ending hyphenation
    text = re.sub(r"-\s*\n\s*", "", text)

    # Normalize whitespace
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def split_full_text_pages_for_repair(full_text: str) -> List[Dict]:
    """
    Split full text into pages using [Page n] markers.
    """
    pattern = re.compile(
        r"\[Page\s+(\d+)\]\s*(.*?)(?=(?:\n\[Page\s+\d+\])|\Z)",
        re.DOTALL
    )

    pages = []
    for m in pattern.finditer(full_text):
        pages.append({
            "page": int(m.group(1)),
            "text": m.group(2).strip()
        })
    return pages


def segment_page_text_for_repair(page_text: str, max_chars: int = 1400) -> List[str]:
    """
    Chunk page text around paragraphs.
    If a paragraph is too long, split it further into sentence blocks.
    """
    page_text = page_text.strip()
    if not page_text:
        return []

    paras = [p.strip() for p in re.split(r"\n\s*\n+", page_text) if p.strip()]

    # Fall back to sentence blocks when paragraph boundaries are sparse
    if len(paras) <= 1:
        sents = re.split(r"(?<=[\.\?\!])\s+(?=[A-Z0-9])", page_text)
        sents = [s.strip() for s in sents if s.strip()]
        paras = []

        buf = []
        buf_len = 0
        for s in sents:
            add_len = len(s) + (1 if buf else 0)
            if buf and buf_len + add_len > max_chars:
                paras.append(" ".join(buf).strip())
                buf = [s]
                buf_len = len(s)
            else:
                buf.append(s)
                buf_len += add_len
        if buf:
            paras.append(" ".join(buf).strip())

    # Force-split chunks that are still too long
    final_chunks = []
    for p in paras:
        if len(p) <= max_chars:
            final_chunks.append(p)
        else:
            start = 0
            while start < len(p):
                end = min(start + max_chars, len(p))
                if end < len(p):
                    cut = p.rfind(". ", start, end)
                    if cut != -1 and cut > start + 200:
                        end = cut + 1
                final_chunks.append(p[start:end].strip())
                start = end

    return [c for c in final_chunks if c]


def suspect_score_and_reasons(text: str) -> Tuple[int, List[str]]:
    """
    Heuristically determine whether a suspect chunk needs LLM repair.
    """
    if not text.strip():
        return 0, []

    score = 0
    reasons = []

    # Replacement, private-use, or mojibake-like characters
    if re.search(r"[�]", text):
        score += 5
        reasons.append("replacement_char")

    if re.search(r"[\uE000-\uF8FF]", text):
        score += 5
        reasons.append("private_use_char")

    # Suspected corruption in p-values or comparison symbols
    if re.search(r"\bp\s+[bBqQ]\s*0\.\d+", text):
        score += 5
        reasons.append("broken_comparison_symbol")

    if re.search(r"\bp\s+[bBqQ]\s*\d", text):
        score += 4
        reasons.append("broken_pvalue_symbol")

    # Domain contexts where < and > are often corrupted
    if re.search(r"\b(significant|significantly|increase|decrease|greater|less)\b", text, re.I) and \
       re.search(r"\b[bBqQ]\s*\d", text):
        score += 2
        reasons.append("suspicious_operator_near_result_text")

    # Unusual single-character patterns near zeta, assay, or unit terms
    domain_keywords = [
        "zeta", "potential", "TEM", "SEM", "DLS", "NTA", "BET",
        "hydrodynamic", "surface area", "surface charge",
        "DMEM", "RPMI", "FBS", "serum-free",
        "µg/mL", "ug/mL", "mV", "nm", "Fig.", "Figure", "Table"
    ]
    has_domain = any(k.lower() in text.lower() for k in domain_keywords)

    one_char_tokens = re.findall(r"\b[a-zA-Z]\b", text)
    if has_domain and len(one_char_tokens) >= 6:
        score += 1
        reasons.append("many_single_char_tokens")

    # Known corruption candidates
    suspicious_patterns = [
        r"z[\-\~\? ]potential",
        r"\bS[FP]\b medium",
        r"\bµg\/m[Ll]\b",
        r"\bm[Vv]\b",
    ]
    for pat in suspicious_patterns:
        if re.search(pat, text, re.I):
            score += 1
            reasons.append(f"pattern:{pat}")

    return score, reasons


def collect_suspect_chunks(full_text_raw: str, max_chunk_chars: int = 1400) -> Tuple[List[Dict], List[Dict]]:
    """
    Split full text into pages/chunks and collect suspect chunks.
    return:
      page_chunks: [
        {
          "page": 1,
          "chunks": [{"chunk_id": 0, "text": "..."} , ...]
        }, ...
      ]
      suspects: [
        {
          "span_id": 1,
          "page": 1,
          "chunk_id": 0,
          "text": "...",
          "reasons": [...]
        }, ...
      ]
    """
    pages = split_full_text_pages_for_repair(full_text_raw)

    page_chunks = []
    suspects = []
    span_id = 1

    for page_obj in pages:
        page_num = page_obj["page"]
        page_text_norm = normalize_text_deterministic(page_obj["text"])
        chunks = segment_page_text_for_repair(page_text_norm, max_chars=max_chunk_chars)

        page_chunk_obj = {
            "page": page_num,
            "chunks": []
        }

        for chunk_idx, chunk in enumerate(chunks):
            page_chunk_obj["chunks"].append({
                "chunk_id": chunk_idx,
                "text": chunk
            })

            score, reasons = suspect_score_and_reasons(chunk)
            if score >= 3:
                suspects.append({
                    "span_id": span_id,
                    "page": page_num,
                    "chunk_id": chunk_idx,
                    "text": chunk,
                    "reasons": reasons,
                    "score": score
                })
                span_id += 1

        page_chunks.append(page_chunk_obj)

    return page_chunks, suspects


def repair_suspect_chunks_with_llm(
    suspects: List[Dict],
    client,
    model: str,
    temperature: float = 0,
    batch_size: int = 8
) -> Dict[int, Dict]:
    """
    Run minimal LLM repair only on suspect chunks in batches.
    return:
      {
        span_id: {
          "repaired_text": "...",
          "changed": "Yes"/"No",
          "confidence": "High"/"Medium"/"Low"
        }
      }
    """
    if not suspects:
        return {}

    repair_map = {}

    for i in range(0, len(suspects), batch_size):
        batch = suspects[i:i+batch_size]

        system_prompt = """
You are repairing small corrupted spans extracted from scientific PDF text.

Goal:
- minimally repair corrupted characters only
- preserve numbers, units, abbreviations, and scientific meaning
- do NOT paraphrase
- do NOT summarize
- do NOT rewrite stylistically
- do NOT add missing scientific content unless the local correction is nearly certain

Rules:
1. Make the smallest possible character-level or token-level edits.
2. Preserve all numeric values exactly unless a visible corruption makes the number impossible.
3. Preserve domain terms such as DMEM, RPMI 1640, FBS, TEM, SEM, DLS, BET, zeta-potential, µg/mL, mV, nm.
4. If uncertain, keep the text unchanged.
5. Typical repairs include:
   - broken comparison symbols in phrases like "p < 0.05"
   - corrupted Greek/special characters
   - damaged unit symbols
   - mojibake or broken glyph substitutions
6. Return JSON only.
"""

        user_payload = []
        for item in batch:
            user_payload.append({
                "span_id": item["span_id"],
                "page": item["page"],
                "reasons": item["reasons"],
                "text": item["text"]
            })

        resp = client.chat.completions.create(
            model=model,
            temperature=temperature,
            response_format={
                "type": "json_schema",
                "json_schema": TEXT_REPAIR_SCHEMA
            },
            messages=[
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": (
                        "Repair only if needed. Keep unchanged if uncertain.\n\n"
                        f"Suspect spans:\n{json.dumps(user_payload, ensure_ascii=False, indent=2)}"
                    )
                },
            ],
        )

        result = json.loads(resp.choices[0].message.content)

        for r in result.get("repairs", []):
            repair_map[r["span_id"]] = {
                "repaired_text": r["repaired_text"],
                "changed": r["changed"],
                "confidence": r["confidence"]
            }

    return repair_map


def rebuild_full_text_with_repairs(
    page_chunks: List[Dict],
    suspects: List[Dict],
    repair_map: Dict[int, Dict]
) -> Tuple[str, List[Dict]]:
    """
    Apply repaired_text only to suspect chunks and reconstruct the full text.
    """
    suspect_lookup = {
        (s["page"], s["chunk_id"]): s for s in suspects
    }

    repair_log = []
    rebuilt_pages = []

    for page_obj in page_chunks:
        page_num = page_obj["page"]
        rebuilt_chunks = []

        for chunk in page_obj["chunks"]:
            key = (page_num, chunk["chunk_id"])
            original_text = chunk["text"]

            if key in suspect_lookup:
                s = suspect_lookup[key]
                repair_info = repair_map.get(s["span_id"], None)

                if repair_info is not None:
                    repaired_text = repair_info["repaired_text"].strip()
                    changed = repair_info["changed"]
                    confidence = repair_info["confidence"]

                    if repaired_text == "":
                        repaired_text = original_text
                        changed = "No"

                    rebuilt_chunks.append(repaired_text)

                    repair_log.append({
                        "page": page_num,
                        "chunk_id": chunk["chunk_id"],
                        "span_id": s["span_id"],
                        "score": s["score"],
                        "reasons": s["reasons"],
                        "original_text": original_text,
                        "repaired_text": repaired_text,
                        "changed": changed,
                        "confidence": confidence
                    })
                else:
                    rebuilt_chunks.append(original_text)
            else:
                rebuilt_chunks.append(original_text)

        page_text = "\n\n".join([c for c in rebuilt_chunks if c.strip()])
        rebuilt_pages.append(f"[Page {page_num}]\n{page_text}")

    full_text_clean = "\n\n".join(rebuilt_pages).strip()
    return full_text_clean, repair_log


def repair_full_text_selectively(
    full_text_raw: str,
    client,
    model: str,
    temperature: float = 0,
    batch_size: int = 8,
    max_chunk_chars: int = 1400
) -> Tuple[str, List[Dict]]:
    """
    raw full text -> deterministic normalize -> suspect chunk detection -> LLM minimal repair
    """
    page_chunks, suspects = collect_suspect_chunks(
        full_text_raw=full_text_raw,
        max_chunk_chars=max_chunk_chars
    )

    if not suspects:
        # If no suspect chunks are found, return the deterministically normalized text only
        rebuilt_pages = []
        for page_obj in page_chunks:
            page_text = "\n\n".join([c["text"] for c in page_obj["chunks"] if c["text"].strip()])
            rebuilt_pages.append(f"[Page {page_obj['page']}]\n{page_text}")
        full_text_clean = "\n\n".join(rebuilt_pages).strip()
        return full_text_clean, []

    repair_map = repair_suspect_chunks_with_llm(
        suspects=suspects,
        client=client,
        model=model,
        temperature=temperature,
        batch_size=batch_size
    )

    full_text_clean, repair_log = rebuild_full_text_with_repairs(
        page_chunks=page_chunks,
        suspects=suspects,
        repair_map=repair_map
    )

    return full_text_clean, repair_log

## Step 2.4: Extract the Methods section

Identify Materials and Methods-like headings and truncate at the next major section boundary.

In [ ]:
# Step 2.4: Extract the Materials and Methods section from full text

SECTION_START_PATTERNS = [
    # Highest priority: Materials and Methods variants
    r"(?im)^(?P<heading>\s*materials?\s*(?:and|&)\s*methods?\s*)$",
    r"(?im)^(?P<heading>\s*materials?\s*(?:and|&)\s*methodology\s*)$",
    r"(?im)^(?P<heading>\s*experimental\s+methods?\s*)$",
    r"(?im)^(?P<heading>\s*experimental\s+section\s*)$",

    # Fallback: standalone Methods / Methodology headings
    r"(?im)^(?P<heading>\s*methods?\s*)$",
    r"(?im)^(?P<heading>\s*methodology\s*)$",

    # Fallback when headings appear mid-line rather than at line start
    r"(?i)(?P<heading>materials?\s*(?:and|&)\s*methods?)(?=\s*\n)",
    r"(?i)(?P<heading>materials?\s*(?:and|&)\s*methodology)(?=\s*\n)",
    r"(?i)(?P<heading>experimental\s+methods?)(?=\s*\n)",
    r"(?i)(?P<heading>experimental\s+section)(?=\s*\n)",
    r"(?i)(?P<heading>methods?)(?=\s*\n)",
    r"(?i)(?P<heading>methodology)(?=\s*\n)",
]

SECTION_END_PATTERNS = [
    # Most common section end points
    r"(?im)^(?P<heading>\s*results?\s*(?:and\s*discussion)?\s*)$",
    r"(?im)^(?P<heading>\s*discussion\s*)$",
    r"(?im)^(?P<heading>\s*conclusions?\s*)$",
    r"(?im)^(?P<heading>\s*summary\s*)$",
    r"(?im)^(?P<heading>\s*acknowledg?ments?\s*)$",
    r"(?im)^(?P<heading>\s*references\s*)$",
    r"(?im)^(?P<heading>\s*bibliography\s*)$",

    # Fallback when end headings appear mid-line
    r"(?i)(?P<heading>results?\s*(?:and\s*discussion)?)(?=\s*\n)",
    r"(?i)(?P<heading>discussion)(?=\s*\n)",
    r"(?i)(?P<heading>conclusions?)(?=\s*\n)",
    r"(?i)(?P<heading>summary)(?=\s*\n)",
    r"(?i)(?P<heading>acknowledg?ments?)(?=\s*\n)",
    r"(?i)(?P<heading>references)(?=\s*\n)",
    r"(?i)(?P<heading>bibliography)(?=\s*\n)",
]


def _find_first_heading_match(text: str, patterns):
    """
    Return the earliest match among multiple heading patterns.
    """
    candidates = []

    for pat in patterns:
        for m in re.finditer(pat, text):
            if "heading" in m.groupdict():
                start = m.start("heading")
            else:
                start = m.start()
            candidates.append((start, m))

    if not candidates:
        return None

    candidates.sort(key=lambda x: x[0])
    return candidates[0][1]


def _find_first_heading_match_after(text: str, patterns, start_pos: int):
    """
    Return the earliest heading match after start_pos.
    """
    sub = text[start_pos:]
    m = _find_first_heading_match(sub, patterns)
    if m is None:
        return None

    # Convert the position in the substring back to the original text coordinate
    abs_start = start_pos + (m.start("heading") if "heading" in m.groupdict() else m.start())
    abs_end = start_pos + (m.end("heading") if "heading" in m.groupdict() else m.end())

    class _SimpleMatch:
        def __init__(self, s, e):
            self._s = s
            self._e = e
        def start(self):
            return self._s
        def end(self):
            return self._e

    return _SimpleMatch(abs_start, abs_end)


def extract_methods_section_from_full_text(full_text: str) -> str:
    """
    Extract only the Methods section from repaired full_text.
    - Start: Materials and Methods / Methods / Experimental section, etc.
    - End: Results / Results and Discussion / Discussion / Conclusions /
            Acknowledgments / References, etc.
    - Return 'None' if extraction fails.
    """
    if not isinstance(full_text, str) or not full_text.strip():
        return "None"

    text = full_text

    start_match = _find_first_heading_match(text, SECTION_START_PATTERNS)
    if start_match is None:
        return "None"

    if "heading" in start_match.groupdict():
        start_pos = start_match.start("heading")
    else:
        start_pos = start_match.start()

    end_match = _find_first_heading_match_after(text, SECTION_END_PATTERNS, start_pos + 1)

    if end_match is not None and end_match.start() > start_pos:
        methods_text = text[start_pos:end_match.start()].strip()
    else:
        # If no end heading is found, use text until the end
        methods_text = text[start_pos:].strip()

    # Treat very short extraction results as failures
    if len(methods_text) < 50:
        return "None"

    return methods_text

# Step 3: Extract assayed nanomaterials

Use full-paper text to identify nanomaterials that were actually used in biological or cytotoxicity assays.

## Step 3.1: Define the material extraction schema

The schema constrains the LLM output to material, composition, medium, and supporting evidence fields.

In [ ]:
# Step 3.1: Define the material extraction schema
MATERIAL_SCHEMA = {
    "name": "assay_material_extraction",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "materials": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "material_name": {
                            "type": "string",
                            "description": "Exact nanomaterial name used in the paper."
                        },
                        "composition": {
                            "type": "string",
                            "description": "Abbreviation only, e.g. TiO2, ZnO, SiO2, CeO2. Use 'None' if not explicit."
                        },
                        "media": {
                            "type": "string",
                            "description": "Concise normalized medium/vehicle label. Example: 'RPMI 1640 (10% FBS)', 'DMEM', 'PBS', 'Serum-free DMEM', 'None'. Do not output full sentences or long additive lists."
                        }
                    },
                    "required": ["material_name", "composition", "media"]
                }
            }
        },
        "required": ["materials"]
    }
}

## Step 3.2: Extract materials from each paper

Apply the schema to each repaired full-text paper record and return one material-medium object per assayed nanomaterial.

In [ ]:
# Step 3.2: Extract assayed nanomaterials from each paper
def extract_assay_materials_from_full_paper(full_text):
    system_prompt = """
You are an expert curator in nanotoxicology.

Your task is to extract only the nanomaterials actually used in the experiments of the paper.

Interpret 'materials used in assay' as:
- only nanomaterials directly exposed to cells in biological/cytotoxicity assays.
- exclude characterization-only materials.

Exclude:
- assay reagents (MTT, CCK-8, WST-1, dyes, stains, buffers, solvents)
- background/reference materials not actually tested
- discussion-only materials from prior literature
- general chemicals unless they are nanomaterials being tested

Rules:
- One object per material × medium.
- If the same material appears in multiple media, split into multiple objects.
- Keep material_name exactly as written in the paper.
- Composition must be abbreviation only (TiO2, ZnO, SiO2, etc.) or 'None'.
- Do not return the umbrella label separately unless the paper clearly treats that umbrella label as its own independent assay arm.

Media normalization rules:
- Do NOT copy the full medium sentence verbatim.
- Output a short normalized label only.
- Keep the basal medium name and the serum percentage if explicitly stated.
- If a named medium typically used serum-free (e.g. LHC-9, BEGM) is given for the exposure condition without explicit serum/FBS, normalize it as "<medium> (Serum-free)".
- Format examples:
  - "RPMI 1640 medium supplemented with 10% heat inactivated foetal bovine serum (FBS), L-glutamine (2 mM), streptomycin (100 μg/ml) and penicillin (100 U/ml)"
    -> "RPMI 1640 (10% FBS)"
  - "DMEM containing 1% FBS and antibiotics"
    -> "DMEM (1% FBS)"
  - "serum-free DMEM"
    -> "DMEM (Serum-free)"
  - "LHC-9"
    -> "LHC-9 (Serum-free)"
  - "PBS"
    -> "PBS"
- Omit non-essential additives such as penicillin, streptomycin, glutamine, HEPES, sodium pyruvate, heat-inactivated, and catalog/manufacturer details.
- Include only the most important distinguishing condition in a compact form.
- If no medium/vehicle is explicitly linked to the material, return "None".

Additional media resolution rules:
- Never output generic labels such as "SF medium", "serum-free medium", "culture medium", "complete medium", or "growth medium" as final media.
- Resolve such generic labels to the actual basal medium used for the cells in the assay.
- If the paper states the cells were cultured in a basal medium (e.g. DMEM, RPMI 1640) and later says the nanoparticles were prepared/tested in "SF medium" or "serum-free medium", normalize this as "Serum-free <basal medium>".
- Example:
  - cells were grown in DMEM with 10% FBS
  - nanoparticles were prepared in SF culture medium
  -> output "DMEM (Serum-free)"
- Use the assay exposure medium, not characterization-only dispersants such as ultrapure water or PBS, unless the particles were actually exposed to cells in that vehicle.
- If multiple media appear, prefer the medium directly used for cell exposure/treatment.

- If nothing is found, return {"materials": []}.
"""

    user_prompt = f"""
Extract all nanomaterials actually used in the study from the full paper text below.

Return JSON only.

FULL PAPER TEXT:
{full_text}
"""

    resp = client.chat.completions.create(
        model=CHAT_MODEL_NAME,
        temperature=CHAT_TEMPERATURE,
        response_format={
            "type": "json_schema",
            "json_schema": MATERIAL_SCHEMA
        },
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    content = resp.choices[0].message.content
    return json.loads(content)

# Step 4: Extract figures, tables, and captions

Detect figure/table regions, merge nearby caption blocks, crop assets, and save the extracted visual evidence.

In [ ]:
# Step 4.1: Initialize the document layout model
import csv
import cv2
import layoutparser as lp

from pdf2image import convert_from_path

model = lp.AutoLayoutModel(
    "lp://detectron2/PubLayNet/mask_rcnn_X_101_32x8d_FPN_3x",
    extra_config=["MODEL.DEVICE", "cuda"]
)

In [ ]:
# Step 4.2: Define figure/table caption matching helpers
import math

def center_distance(box1, box2):
    """
    box = (x0, y0, x1, y1)
    Euclidean distance between the center points of two boxes.
    """
    c1x = (box1[0] + box1[2]) / 2
    c1y = (box1[1] + box1[3]) / 2
    c2x = (box2[0] + box2[2]) / 2
    c2y = (box2[1] + box2[3]) / 2
    return math.sqrt((c2x - c1x) ** 2 + (c2y - c1y) ** 2)


def _normalize_extracted_caption_text(text: str) -> str:
    if text is None:
        return ""

    text = str(text)

    # Core requirement
    text = text.replace("\x04", "-")

    # Basic whitespace cleanup
    text = text.replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)

    return text.strip()

    
def extract_figures_tables_with_caption_merging(
    pdf_file_path, key, img_dir, model, dpi=300, margin=20, max_height_ratio=0.9, debug=False
):
    save_dir = os.path.join(img_dir, key)
    os.makedirs(save_dir, exist_ok=True)

    caption_pat = re.compile(r"^\s*(Figure|Fig\.?|Table)\s+S?\d+", re.IGNORECASE)
    table_pat = re.compile(r"^\s*Table\s+S?\d+", re.IGNORECASE)
    figure_pat = re.compile(r"^\s*(Figure|Fig\.?)\s+S?\d+", re.IGNORECASE)
    footer_pat = re.compile(
        r"^(Page\s*\d+|https?://|content/|Copyright|All rights reserved|Springer Nature)",
        re.IGNORECASE,
    )

    pages = convert_from_path(pdf_file_path, dpi=dpi)
    doc = fitz.open(pdf_file_path)

    results = []

    for page_idx, (page, fitz_page) in enumerate(zip(pages, doc)):
        img_path = os.path.join(save_dir, f"{key}_p{page_idx+1}.png")
        page.save(img_path, "PNG")
        image = cv2.imread(img_path)
        img_h, img_w = image.shape[:2]
        pdf_w, pdf_h = fitz_page.rect.width, fitz_page.rect.height

        # 1. Detect Figure/Table bounding boxes with layoutparser
        layout = model.detect(image)
        figures = [b for b in layout if b.type == 4 or str(b.type).lower() == "figure"]
        tables  = [b for b in layout if b.type == 3 or str(b.type).lower() == "table"]

        # 2. Merge caption blocks from PyMuPDF text blocks
        blocks = fitz_page.get_text("blocks")

        merged_caption_blocks = []
        i = 0
        while i < len(blocks):
            text = _normalize_extracted_caption_text(blocks[i][4].strip())

            if caption_pat.match(text):
                caption_full = text
                x0, y0, x1, y1 = blocks[i][0], blocks[i][1], blocks[i][2], blocks[i][3]
                j = i + 1
                block_merge_cnt = 0

                while j < len(blocks):
                    next_txt = _normalize_extracted_caption_text(blocks[j][4].strip())

                    if footer_pat.match(next_txt):
                        break

                    if (
                        (blocks[j][1] - y1 < 40)
                        and (len(next_txt) > 0)
                        and (blocks[j][0] - blocks[i][0] < 30)
                        and block_merge_cnt < 4
                    ):
                        caption_full += " " + next_txt
                        x0 = min(x0, blocks[j][0])
                        y0 = min(y0, blocks[j][1])
                        x1 = max(x1, blocks[j][2])
                        y1 = max(y1, blocks[j][3])
                        j += 1
                        block_merge_cnt += 1
                    else:
                        break

                caption_full = _normalize_extracted_caption_text(caption_full)

                merged_caption_blocks.append({
                    "x0": x0,
                    "y0": y0,
                    "x1": x1,
                    "y1": y1,
                    "text": caption_full
                })
                i = j
            else:
                i += 1

        # 3. Match the nearest caption for each Figure/Table and crop the image
        for box_list, label_str in [(figures, "figure"), (tables, "table")]:
            for j, obj in enumerate(box_list):
                x1, y1, x2, y2 = map(int, obj.coordinates)

                if debug:
                    debug_img = image.copy()
                    cv2.rectangle(debug_img, (x1, y1), (x2, y2), (0, 0, 255), 2)
                    debug_img_path = os.path.join(save_dir, f"{key}_p{page_idx+1}_{label_str}{j+1}_box.png")
                    cv2.imwrite(debug_img_path, debug_img)

                crop = image[y1:y2, x1:x2]
                crop_img_path = os.path.join(save_dir, f"{key}_p{page_idx+1}_{label_str}{j+1}.png")
                cv2.imwrite(crop_img_path, crop)

                # Convert the object box to PDF coordinates
                obj_box_pdf = (
                    x1 * (pdf_w / img_w),
                    y1 * (pdf_h / img_h),
                    x2 * (pdf_w / img_w),
                    y2 * (pdf_h / img_h),
                )

                # Select the same-type caption with the shortest center-point distance
                cap_candidates = []
                for b in merged_caption_blocks:
                    cap_text_raw = _normalize_extracted_caption_text(b["text"])

                    is_match_type = (
                        table_pat.match(cap_text_raw) if label_str == "table"
                        else figure_pat.match(cap_text_raw)
                    )

                    if not is_match_type:
                        continue

                    cap_box_pdf = (b["x0"], b["y0"], b["x1"], b["y1"])
                    dist = center_distance(obj_box_pdf, cap_box_pdf)
                    cap_candidates.append((dist, cap_text_raw))

                cap_text = min(cap_candidates, key=lambda x: x[0])[1] if cap_candidates else ""
                cap_text = _normalize_extracted_caption_text(cap_text)

                results.append((crop_img_path, cap_text, label_str, page_idx + 1))

    df = pd.DataFrame(results, columns=["image_path", "caption", "image_property", "page"])

    # Apply one final normalization pass so CSV and downstream steps use identical captions
    if not df.empty:
        df["caption"] = df["caption"].map(_normalize_extracted_caption_text)

    csv_path = os.path.join(save_dir, f"{key}_cap.csv")
    df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8",
        quoting=csv.QUOTE_MINIMAL,
        doublequote=True,
        escapechar="\\"
    )
    return df

# Step 5: Select physicochemical figure/table candidates

Use material names and captions to select figures/tables likely to contain physicochemical characterization data.

In [ ]:
# Step 5.1: Define the PChem candidate selection schema
import json
import pandas as pd

PCHEM_SELECTOR_SCHEMA = {
    "name": "physchem_figure_table_selector",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "materials": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "material": {"type": "string"},
                        "selected_items": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "additionalProperties": False,
                                "properties": {
                                    "row_id": {"type": "integer"},
                                    "page": {"type": "integer"},
                                    "image_property": {"type": "string"},
                                    "caption": {"type": "string"}
                                },
                                "required": ["row_id", "page", "image_property", "caption"]
                            }
                        }
                    },
                    "required": ["material", "selected_items"]
                }
            }
        },
        "required": ["materials"]
    }
}

In [ ]:
# Step 5.2: Prepare PChem caption candidates
def prepare_pchem_caption_candidates(df_images: pd.DataFrame) -> pd.DataFrame:
    df = df_images.copy().reset_index(drop=True)
    df["row_id"] = df.index.astype(int)

    keep_cols = ["row_id", "page", "image_property", "caption"]
    for col in keep_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column in df_images: {col}")

    return df[keep_cols]

In [ ]:
# Step 5.3: Select PChem figure/table candidates by material
def select_pchem_figures_tables_from_captions(
    df_materials: pd.DataFrame,
    df_images: pd.DataFrame,
    client,
    model,
    temperature: float = 0
) -> dict:
    material_records = (
        df_materials[["material", "material_name", "composition", "media"]]
        .drop_duplicates()
        .fillna("None")
        .to_dict(orient="records")
    )

    candidates_df = prepare_pchem_caption_candidates(df_images)
    candidates = candidates_df.to_dict(orient="records")

    system_prompt = """
You are an expert in nanomaterial physicochemical data curation.

Your task is NOT to extract numeric values yet.
Your task is to decide which figures or tables are the best candidates for extracting physicochemical properties
for each target material.

Target properties:
- core_size
- core_size_measurement_method
- core_size_source
- hydrodynamic_size
- hydrodynamic_size_measurement_method
- hydrodynamic_size_source
- surface_charge
- surface_charge_measurement_method
- surface_charge_source
- surface_area
- surface_area_measurement_method
- surface_area_source

Prefer captions mentioning:
- TEM, SEM, AFM, XRD
- DLS, NTA, hydrodynamic size, particle size distribution
- zeta potential, electrophoretic mobility, ELS
- BET, surface area
- physicochemical characterization
- particle characterization
- tables listing properties of nanoparticles
- supplementary tables/figures with size, charge, surface area

Do NOT prefer captions mainly about:
- cytotoxicity only
- cell viability only
- apoptosis only
- uptake only
- inflammation only
- histology only

Rules:
- Work only from the provided captions and metadata.
- One material may map to multiple figures/tables.
- Return up to 5 selected items per material.
- If no suitable figure/table exists, return an empty selected_items list.
- Focus on candidates likely to contain the target physicochemical properties for the exact material.
"""

    user_prompt = f"""
Target materials:
{json.dumps(material_records, ensure_ascii=False, indent=2)}

Figure/Table candidates:
{json.dumps(candidates, ensure_ascii=False, indent=2)}

Return JSON only.
"""

    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        response_format={
            "type": "json_schema",
            "json_schema": PCHEM_SELECTOR_SCHEMA
        },
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    return json.loads(resp.choices[0].message.content)

In [ ]:
# Step 5.4: Convert PChem selector results to a DataFrame
def pchem_selector_result_to_df(selector_result: dict) -> pd.DataFrame:
    rows = []
    for mat_obj in selector_result.get("materials", []):
        material = mat_obj["material"]
        for item in mat_obj.get("selected_items", []):
            rows.append({
                "material": material,
                "row_id": item["row_id"],
                "page": item["page"],
                "image_property": item["image_property"],
                "caption": item["caption"],
            })
    return pd.DataFrame(rows)

In [ ]:
# Step 5.5: Add image path metadata to PChem candidates
def build_pchem_candidates_df(df_materials: pd.DataFrame, df_images: pd.DataFrame, client, model) -> pd.DataFrame:
    selector_result = select_pchem_figures_tables_from_captions(
        df_materials=df_materials,
        df_images=df_images,
        client=client,
        model=model,
        temperature=0
    )

    df_pchem_selector = pchem_selector_result_to_df(selector_result)

    if df_pchem_selector.empty:
        return pd.DataFrame(columns=["material", "row_id", "page", "image_property", "caption", "image_path"])

    df_images_idx = df_images.reset_index().rename(columns={"index": "row_id"})

    df_out = df_pchem_selector.merge(
        df_images_idx,
        on="row_id",
        how="left",
        suffixes=("_sel", "")
    )

    # Clean up duplicate selector and source page/caption columns
    if "page_sel" in df_out.columns:
        df_out["page"] = df_out["page"].fillna(df_out["page_sel"]) if "page" in df_out.columns else df_out["page_sel"]
        df_out.drop(columns=["page_sel"], inplace=True, errors="ignore")

    if "image_property_sel" in df_out.columns:
        df_out["image_property"] = (
            df_out["image_property"].fillna(df_out["image_property_sel"])
            if "image_property" in df_out.columns else df_out["image_property_sel"]
        )
        df_out.drop(columns=["image_property_sel"], inplace=True, errors="ignore")

    if "caption_sel" in df_out.columns:
        # Prefer the source caption; use the selector caption only when the source caption is missing
        if "caption" in df_out.columns:
            df_out["caption"] = df_out["caption"].fillna(df_out["caption_sel"])
        else:
            df_out["caption"] = df_out["caption_sel"]
        df_out.drop(columns=["caption_sel"], inplace=True, errors="ignore")

    return df_out

# Step 6: Extract physicochemical data

Harvest physicochemical claims from selected visual evidence and text context, then consolidate values per material.

In [ ]:
# Step 6.1: Shared PChem extraction helpers and image utilities
import io
import base64
from PIL import Image
def insert_col_after(df: pd.DataFrame, after_col: str, new_col: str, values) -> pd.DataFrame:
    df = df.copy()

    if new_col in df.columns:
        df.drop(columns=[new_col], inplace=True)

    if after_col not in df.columns:
        df[new_col] = values
        return df

    insert_idx = df.columns.get_loc(after_col) + 1
    df.insert(insert_idx, new_col, values)
    return df


def split_full_text_by_page(full_text: str) -> dict:
    page_dict = {}
    pattern = re.compile(r"\[Page\s+(\d+)\]\s*(.*?)(?=(?:\n\[Page\s+\d+\])|\Z)", re.DOTALL)

    for match in pattern.finditer(full_text):
        page_num = int(match.group(1))
        page_text = match.group(2).strip()
        page_dict[page_num] = page_text

    return page_dict


def normalize_match_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.replace("\x00", " ")
    text = text.replace("\xad", "")
    text = re.sub(r"-\s*\n\s*", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()


def build_material_query_terms(material_name: str, composition: str = None, media: str = None):
    stopwords = {
        "nanoparticle", "nanoparticles", "particle", "particles",
        "np", "nps", "the", "and", "with", "for", "in", "of"
    }

    mat_norm = normalize_match_text(material_name)
    comp_norm = normalize_match_text(composition) if composition else ""
    media_norm = normalize_match_text(media) if media else ""

    material_terms = [
        t for t in re.split(r"[^a-zA-Z0-9%]+", mat_norm)
        if len(t) >= 2 and t not in stopwords
    ]

    composition_terms = [
        t for t in re.split(r"[^a-zA-Z0-9%]+", comp_norm)
        if len(t) >= 2 and t not in stopwords and t != "none"
    ]

    media_terms = []
    if media_norm and media_norm != "none":
        media_terms.append(media_norm)

        if "rpmi 1640" in media_norm:
            media_terms.append("rpmi 1640")
            media_terms.append("rpmi")
        if "dmem" in media_norm:
            media_terms.append("dmem")
        if "pbs" in media_norm:
            media_terms.append("pbs")
        if "water" in media_norm:
            media_terms.append("water")

        if "fbs" in media_norm:
            media_terms.append("fbs")
            m = re.search(r"(\d+)\s*%\s*fbs", media_norm)
            if m:
                media_terms.append(f"{m.group(1)}% fbs")

    material_terms = list(dict.fromkeys(material_terms))
    composition_terms = list(dict.fromkeys(composition_terms))
    media_terms = list(dict.fromkeys(media_terms))

    return material_terms, composition_terms, media_terms


def split_page_into_paragraphs(page_text: str):
    page_text = page_text.replace("\x00", " ").strip()

    paras = re.split(r"\n\s*\n+", page_text)

    if len(paras) <= 2:
        paras = re.split(r"(?<=[\.\?\!])\s+(?=[A-Z0-9])", page_text)

    cleaned = []
    for p in paras:
        p = p.strip()
        if len(p) >= 40:
            cleaned.append(p)

    return cleaned


def get_page_context(full_text: str, center_page: int, window: int = 1, max_chars: int = 5000) -> str:
    page_dict = split_full_text_by_page(full_text)

    pages = []
    for p in range(center_page - window, center_page + window + 1):
        if p in page_dict:
            pages.append(f"[Page {p}]\n{page_dict[p]}")

    text = "\n\n".join(pages)
    if len(text) > max_chars:
        text = text[:max_chars] + "\n...[truncated]"
    return text


def image_to_data_url(path: str, max_side: int = 2200, jpeg_quality: int = 85) -> str:
    with Image.open(path) as im:
        im = im.convert("RGB")
        w, h = im.size

        scale = max(w, h) / max_side if max(w, h) > max_side else 1.0
        if scale > 1.0:
            im = im.resize((int(w / scale), int(h / scale)), Image.LANCZOS)

        buf = io.BytesIO()
        im.save(buf, format="JPEG", quality=jpeg_quality, optimize=True)
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

    return f"data:image/jpeg;base64,{b64}"


def get_material_relevant_text_snippets(
    full_text: str,
    material_name: str,
    composition: str = None,
    media: str = None,
    extra_pages=None,
    max_snippets: int = 12,
    max_chars: int = 12000
):
    """
    Collect paragraphs/sentences from full_text that are highly relevant to the target material and physicochemical properties.
    - Include not only material-specific sentences,
    - but also shared characterization, method, and manufacturer sentences.
    """
    if extra_pages is None:
        extra_pages = []

    page_dict = split_full_text_by_page(full_text)
    material_terms, composition_terms, media_terms = build_material_query_terms(
        material_name=material_name,
        composition=composition,
        media=media
    )

    pchem_keywords = [
        "tem", "sem", "afm", "xrd",
        "dls", "nta", "fcs", "hydrodynamic",
        "zeta", "surface charge", "electrophoretic", "els",
        "bet", "surface area",
        "characterization", "characterisation",
        "particle size", "primary size", "core size",
        "diameter", "colloidal", "dispersion"
    ]

    shared_method_keywords = [
        "measured by", "determined by", "characterized by", "characterised by",
        "measured using", "analyzed by", "analysed by",
        "manufacturer", "supplier", "vendor",
        "zetasizer", "malvern", "nitrogen adsorption",
        "electron microscopy", "dynamic light scattering"
    ]

    section_keywords = [
        "materials and methods", "methods", "characterization",
        "physicochemical", "particle characterization",
        "characterisation", "nanoparticle characterization"
    ]

    scored_snippets = []
    seen_norm = set()

    for page_num, page_text in page_dict.items():
        paragraphs = split_page_into_paragraphs(page_text)

        for para in paragraphs:
            norm = normalize_match_text(para)
            if not norm:
                continue

            score = 0

            mat_phrase = normalize_match_text(material_name)
            if mat_phrase and mat_phrase in norm:
                score += 8

            mat_hits = sum(1 for t in material_terms if t in norm)
            comp_hits = sum(1 for t in composition_terms if t in norm)
            media_hits = sum(1 for t in media_terms if t and t in norm)
            kw_hits = sum(1 for kw in pchem_keywords if kw in norm)
            shared_hits = sum(1 for kw in shared_method_keywords if kw in norm)
            sec_hits = sum(1 for kw in section_keywords if kw in norm)

            score += min(mat_hits, 4) * 2
            score += min(comp_hits, 2) * 2
            score += min(media_hits, 3) * 2
            score += min(kw_hits, 4) * 2
            score += min(shared_hits, 2) * 2
            score += min(sec_hits, 2) * 2

            if page_num in extra_pages:
                score += 3

            has_target_anchor = (mat_hits + comp_hits + media_hits) > 0
            has_shared_pchem_signal = (
                kw_hits >= 2
                or (kw_hits >= 1 and shared_hits >= 1)
                or (kw_hits >= 1 and sec_hits >= 1)
                or shared_hits >= 2
            )

            if score >= 4 and (
                has_target_anchor
                or page_num in extra_pages
                or has_shared_pchem_signal
            ):
                key = normalize_match_text(para)
                if key not in seen_norm:
                    seen_norm.add(key)
                    scored_snippets.append({
                        "page": page_num,
                        "score": score,
                        "text": para
                    })

    if len(scored_snippets) < 3 and extra_pages:
        for p in extra_pages:
            if p in page_dict:
                txt = page_dict[p].strip()
                key = normalize_match_text(txt)
                if key and key not in seen_norm:
                    seen_norm.add(key)
                    scored_snippets.append({
                        "page": p,
                        "score": 1,
                        "text": txt
                    })

    scored_snippets = sorted(scored_snippets, key=lambda x: (-x["score"], x["page"]))

    selected = []
    used_chars = 0
    selected_pages = []

    for item in scored_snippets:
        snippet = f"[Page {item['page']} | score={item['score']}]\n{item['text']}"
        if used_chars + len(snippet) > max_chars:
            continue
        selected.append(snippet)
        used_chars += len(snippet)
        selected_pages.append(item["page"])
        if len(selected) >= max_snippets:
            break

    return "\n\n".join(selected), sorted(set(selected_pages))

In [ ]:
# Step 6.2: PChem extraction schema and constants
PCHEM_CLAIM_SCHEMA = {
    "name": "physchem_claim_harvest",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "claims": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "property_name": {
                            "type": "string",
                            "enum": [
                                "core_size",
                                "hydrodynamic_size",
                                "surface_charge",
                                "surface_area"
                            ]
                        },
                        "value": {"type": "string"},
                        "unit": {"type": "string"},
                        "measurement_method": {"type": "string"},
                        "source_type": {"type": "string"},
                        "source_origin": {"type": "string"},
                        "page": {"type": "integer"},
                        "candidate_row_id": {"type": "integer"},
                        "evidence_text": {"type": "string"},
                        "applies_to_target_material": {"type": "string"},
                        "applies_to_target_media": {"type": "string"},

                        "raw_value_text": {"type": "string"},
                        "raw_unit_text": {"type": "string"},
                        "raw_method_text": {"type": "string"},
                        "source_material_label": {"type": "string"},
                        "source_medium_label": {"type": "string"},
                        "row_label": {"type": "string"},
                        "column_header": {"type": "string"},
                        "panel_label": {"type": "string"}
                    },
                    "required": [
                        "property_name",
                        "value",
                        "unit",
                        "measurement_method",
                        "source_type",
                        "source_origin",
                        "page",
                        "candidate_row_id",
                        "evidence_text",
                        "applies_to_target_material",
                        "applies_to_target_media",

                        "raw_value_text",
                        "raw_unit_text",
                        "raw_method_text",
                        "source_material_label",
                        "source_medium_label",
                        "row_label",
                        "column_header",
                        "panel_label"
                    ]
                }
            }
        },
        "required": ["claims"]
    }
}

PCHEM_METHOD_ALLOWED = {
    "core_size": {"TEM", "SEM", "AFM", "XRD", "None"},
    "hydrodynamic_size": {"DLS", "NTA", "FCS", "None"},
    "surface_charge": {"ELS", "Zeta potential", "None"},
    "surface_area": {"BET", "None"},
}

PCHEM_PROPERTIES = [
    "core_size",
    "hydrodynamic_size",
    "surface_charge",
    "surface_area",
]

PCHEM_CANONICAL_UNITS = {
    "core_size": "nm",
    "hydrodynamic_size": "nm",
    "surface_charge": "mV",
    "surface_area": "m2/g",
}

In [ ]:
# Step 6.3: PChem normalization helpers
def _norm_str(x):
    if x is None:
        return ""
    return str(x).replace("\x00", " ").strip()


def _is_missing(x):
    s = _norm_str(x).lower()
    return s in {"", "none", "nan", "null", "n/a", "na", "not available", "missing"}


def _norm_unicode_text(x):
    s = _norm_str(x)
    s = (
        s.replace("−", "-")
         .replace("–", "-")
         .replace("—", "-")
         .replace("‒", "-")
         .replace("μ", "u")
         .replace("µ", "u")
         .replace("㎛", "um")
         .replace("Å", "angstrom")
         .replace("Å", "angstrom")
         .replace("²", "2")
    )
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _format_float_str(v):
    try:
        x = float(v)
    except Exception:
        return "None"

    if abs(x - round(x)) < 1e-9:
        return str(int(round(x)))
    return f"{x:.6f}".rstrip("0").rstrip(".")


def _extract_numeric_value_or_midpoint(text):
    if _is_missing(text):
        return None

    s = _norm_unicode_text(text).replace(",", "")
    s = s.replace("~", "-")
    s = s.replace(" to ", " - ")

    # Handle ranges first: 50-70, 50 - 70, 50~70, 50 to 70
    range_match = re.search(
        r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)\s*-\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)",
        s
    )
    if range_match:
        try:
            a = float(range_match.group(1))
            b = float(range_match.group(2))
            return (a + b) / 2.0
        except Exception:
            pass

    # single value fallback
    m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
    if not m:
        return None

    try:
        return float(m.group(0))
    except Exception:
        return None


def _canonical_claim_source_kind(source_type):
    if _is_missing(source_type):
        return "text"

    s = _norm_str(source_type).lower()

    has_text = "text" in s
    has_caption = "caption" in s
    has_figure = "figure" in s
    has_table = "table" in s

    kinds = sum([has_text, has_caption, has_figure, has_table])

    if has_table and kinds == 1:
        return "table"
    if has_figure and kinds == 1:
        return "figure"
    if has_text and kinds == 1:
        return "text"
    if has_caption and kinds == 1:
        return "caption"
    if kinds >= 2:
        return "mixed"

    return "text"


def _canonical_source_origin(x, evidence_text=None, column_header=None):
    joined = " ".join([
        _norm_unicode_text(x),
        _norm_unicode_text(evidence_text),
        _norm_unicode_text(column_header),
    ]).lower()

    if not joined.strip():
        return "Not specified"

    if any(k in joined for k in ["manufacturer", "vendor", "supplier", "provided by", "from sigma", "from aldrich"]):
        return "Manufacturer"

    if any(k in joined for k in ["measured", "determined", "experiment", "measur", "characterized", "characterised"]):
        return "Experiment"

    return "Not specified"


def _canonical_method(prop, method=None, raw_method_text=None, column_header=None, evidence_text=None):
    joined = " ".join([
        _norm_unicode_text(method),
        _norm_unicode_text(raw_method_text),
        _norm_unicode_text(column_header),
        _norm_unicode_text(evidence_text),
    ]).lower()

    if not joined.strip():
        return "None"

    if prop == "core_size":
        if "tem" in joined or "transmission electron microscopy" in joined:
            return "TEM"
        if "sem" in joined or "scanning electron microscopy" in joined:
            return "SEM"
        if "afm" in joined or "atomic force microscopy" in joined:
            return "AFM"
        if "xrd" in joined or "x-ray diffraction" in joined:
            return "XRD"
        return "None"

    if prop == "hydrodynamic_size":
        if re.search(r"\bdls\b", joined) or "dynamic light scattering" in joined:
            return "DLS"
        if re.search(r"\bnta\b", joined) or "nanoparticle tracking analysis" in joined:
            return "NTA"
        if re.search(r"\bfcs\b", joined) or "fluorescence correlation spectroscopy" in joined:
            return "FCS"
        return "None"

    if prop == "surface_charge":
        if "electrophoretic light scattering" in joined or re.search(r"\bels\b", joined):
            return "ELS"
        if "zeta" in joined or "zeta potential" in joined or "zetasizer" in joined:
            return "Zeta potential"
        return "None"

    if prop == "surface_area":
        if "bet" in joined or "brunauer" in joined or "nitrogen adsorption" in joined:
            return "BET"
        return "None"

    return "None"


def _infer_unit_from_claim(prop, claim):
    text = " ".join([
        _norm_unicode_text(claim.get("raw_unit_text")),
        _norm_unicode_text(claim.get("unit")),
        _norm_unicode_text(claim.get("raw_value_text")),
        _norm_unicode_text(claim.get("column_header")),
        _norm_unicode_text(claim.get("evidence_text")),
    ]).lower()

    if prop in {"core_size", "hydrodynamic_size"}:
        if any(k in text for k in ["angstrom", "ångstrom", "å"]):
            return "angstrom"
        if re.search(r"(?<![a-z])um(?![a-z])", text) or "microm" in text:
            return "um"
        if "nm" in text or "nanometer" in text or "nanometre" in text:
            return "nm"
        return None

    if prop == "surface_charge":
        if "mv" in text or "millivolt" in text:
            return "mV"
        if re.search(r"(?<![a-z])v(?![a-z])", text):
            return "V"
        return None

    if prop == "surface_area":
        if "m2/g" in text or "m^2/g" in text or "m 2/g" in text:
            return "m2/g"
        if "cm2/g" in text or "cm^2/g" in text or "cm 2/g" in text:
            return "cm2/g"
        return None

    return None


def _convert_numeric_to_canonical(prop, num, unit):
    if num is None or unit is None:
        return "None"

    if prop in {"core_size", "hydrodynamic_size"}:
        if unit == "nm":
            return _format_float_str(num)
        if unit == "um":
            return _format_float_str(num * 1000.0)
        if unit == "angstrom":
            return _format_float_str(num * 0.1)
        return "None"

    if prop == "surface_charge":
        if unit == "mV":
            return _format_float_str(num)
        if unit == "V":
            return _format_float_str(num * 1000.0)
        return "None"

    if prop == "surface_area":
        if unit == "m2/g":
            return _format_float_str(num)
        if unit == "cm2/g":
            return _format_float_str(num / 10000.0)
        return "None"

    return "None"


def _normalized_claim_value(prop, claim):
    raw_candidates = [
        claim.get("raw_value_text"),
        claim.get("value"),
    ]

    num = None
    for x in raw_candidates:
        num = _extract_numeric_value_or_midpoint(x)
        if num is not None:
            break

    unit = _infer_unit_from_claim(prop, claim)
    return _convert_numeric_to_canonical(prop, num, unit)


def _pchem_output_unit(prop, value):
    if _norm_str(value) in {"", "None"}:
        return "None"
    return PCHEM_CANONICAL_UNITS.get(prop, "None")


def _clean_numeric_value(x):
    if _is_missing(x):
        return "None"

    s = _norm_unicode_text(x).replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
    if not m:
        return "None"
    return m.group(0)


def _classify_medium_label(label):
    text = _norm_unicode_text(label).lower()
    out = {
        "raw": text,
        "basal": None,
        "fbs_pct": None,
        "serum_free": False,
        "is_pbs": False,
        "is_water": False,
    }

    if not text or text in {"none", "not specified"}:
        return out

    if re.search(r"\bserum[- ]free\b", text) or re.search(r"\bsf\b", text):
        out["serum_free"] = True

    m = re.search(r"(\d+(?:\.\d+)?)\s*%\s*fbs", text)
    if m:
        out["fbs_pct"] = float(m.group(1))

    if "rpmi 1640" in text or re.search(r"\brpmi\b", text):
        out["basal"] = "RPMI 1640"
    elif "dmem/f12" in text or "dmem f12" in text:
        out["basal"] = "DMEM/F12"
    elif "dmem" in text:
        out["basal"] = "DMEM"
    elif "emem" in text:
        out["basal"] = "EMEM"
    elif re.search(r"(?<!e)\bmem\b", text):
        out["basal"] = "MEM"
    elif "f-12" in text or "f12" in text:
        out["basal"] = "F12"
    elif "pbs" in text or "phosphate buffered saline" in text:
        out["basal"] = "PBS"
        out["is_pbs"] = True
    elif any(k in text for k in ["water", "ultrapure", "distilled", "milli-q", "mq water"]):
        out["basal"] = "Water"
        out["is_water"] = True

    return out


def _infer_source_medium_label(claim):
    combined = " ".join([
        _norm_unicode_text(claim.get("source_medium_label")),
        _norm_unicode_text(claim.get("column_header")),
        _norm_unicode_text(claim.get("evidence_text")),
    ])

    if not combined.strip():
        return "Not specified"

    c = _classify_medium_label(combined)
    if c["is_water"]:
        return "Water"
    if c["is_pbs"]:
        return "PBS"
    if c["basal"] and c["fbs_pct"] is not None:
        pct = _format_float_str(c["fbs_pct"])
        return f"{c['basal']} ({pct}% FBS)"
    if c["basal"] and c["serum_free"]:
        return f"{c['basal']} (Serum-free)"
    if c["basal"]:
        return c["basal"]
    if c["serum_free"]:
        return "Serum-free"
    return _norm_str(claim.get("source_medium_label")) or "Not specified"

def _column_header_keyword_bonus_for_core_size(claim, prop):
    """
    Column-header keyword bonus for scoring core_size value claims.

    Rules:
    - Contains 'size'    -> +5
    - Contains 'primary' -> +8
    - Contains 'nominal' -> +10

    If multiple keywords are present, add the bonuses cumulatively.
    Example:
    - 'Nominal size (nm)' -> +15
    - 'Primary particle size' -> +13
    """
    if prop != "core_size":
        return 0

    header = _norm_unicode_text(claim.get("column_header")).lower()
    if not header:
        return 0

    bonus = 0

    if "size" in header:
        bonus += 5
    if "primary" in header:
        bonus += 8
    if "nominal" in header:
        bonus += 10

    return bonus

def _media_match_tier(target_media, source_medium_label, prop):
    if prop not in {"hydrodynamic_size", "surface_charge"}:
        return 4

    t = _classify_medium_label(target_media)
    s = _classify_medium_label(source_medium_label)

    if not s["raw"]:
        return 1

    if _norm_unicode_text(target_media).lower() == _norm_unicode_text(source_medium_label).lower():
        return 5

    if t["basal"] and s["basal"] and t["basal"] == s["basal"]:
        if t["fbs_pct"] is not None and s["fbs_pct"] is not None:
            if abs(t["fbs_pct"] - s["fbs_pct"]) < 1e-9:
                return 5
            return 4
        if t["serum_free"] and s["serum_free"]:
            return 5
        return 4

    if s["serum_free"] and t["serum_free"]:
        return 3

    if s["is_pbs"]:
        return 2
    if s["is_water"]:
        return 2

    if not t["raw"]:
        return 2

    return 1


def _media_score_for_claim(claim, target_media, prop):
    tier = _media_match_tier(
        target_media=target_media,
        source_medium_label=_infer_source_medium_label(claim),
        prop=prop
    )

    base = {
        5: 40,
        4: 30,
        3: 18,
        2: 8,
        1: 0,
    }.get(tier, 0)

    status = _norm_str(claim.get("applies_to_target_media", "Unclear")).lower()

    if status == "yes":
        base += 10
    elif status == "unclear":
        base += 2
    elif status == "not relevant":
        base += 4 if prop not in {"hydrodynamic_size", "surface_charge"} else 0
    elif status == "no":
        base -= 3

    if not _is_missing(claim.get("source_medium_label")):
        base += 4

    return base


def _material_ok(claim):
    status = _norm_str(claim.get("applies_to_target_material", "Unclear")).lower()
    return status != "no"


def _claim_score_for_value(claim, prop, target_media):
    if not _material_ok(claim):
        return -10**9

    score = 0

    value = _normalized_claim_value(prop, claim)
    if value != "None":
        score += 100
    else:
        score -= 1000

    method = _canonical_method(
        prop,
        claim.get("measurement_method"),
        claim.get("raw_method_text"),
        claim.get("column_header"),
        claim.get("evidence_text")
    )
    if method != "None":
        score += 8

    origin = _canonical_source_origin(
        claim.get("source_origin"),
        claim.get("evidence_text"),
        claim.get("column_header")
    )
    if origin == "Experiment":
        score += 5
    elif origin == "Manufacturer":
        score += 3

    kind = _canonical_claim_source_kind(claim.get("source_type"))
    kind_score = {
        "table": 20,
        "mixed": 16,
        "figure": 14,
        "text": 12,
        "caption": 5,
    }.get(kind, 0)
    score += kind_score

    score += _media_score_for_claim(claim, target_media, prop)

    if not _is_missing(claim.get("row_label")):
        score += 2
    if not _is_missing(claim.get("column_header")):
        score += 3

    score += _column_header_keyword_bonus_for_core_size(claim, prop)

    if len(_norm_str(claim.get("evidence_text"))) >= 15:
        score += 3

    return score


def _claim_score_for_method(claim, prop, target_media):
    if not _material_ok(claim):
        return -10**9

    method = _canonical_method(
        prop,
        claim.get("measurement_method"),
        claim.get("raw_method_text"),
        claim.get("column_header"),
        claim.get("evidence_text")
    )
    if method == "None":
        return -10**9

    score = 50
    score += _media_score_for_claim(claim, target_media, prop)

    kind = _canonical_claim_source_kind(claim.get("source_type"))
    kind_score = {
        "table": 14,
        "mixed": 12,
        "text": 12,
        "figure": 8,
        "caption": 6,
    }.get(kind, 0)
    score += kind_score

    if _normalized_claim_value(prop, claim) != "None":
        score += 4
    if not _is_missing(claim.get("raw_method_text")):
        score += 4

    return score


def _claim_score_for_origin(claim, prop, target_media):
    if not _material_ok(claim):
        return -10**9

    origin = _canonical_source_origin(
        claim.get("source_origin"),
        claim.get("evidence_text"),
        claim.get("column_header")
    )
    if origin == "Not specified":
        return -10**9

    score = 30
    score += _media_score_for_claim(claim, target_media, prop)

    kind = _canonical_claim_source_kind(claim.get("source_type"))
    kind_score = {
        "table": 10,
        "mixed": 9,
        "text": 8,
        "figure": 6,
        "caption": 4,
    }.get(kind, 0)
    score += kind_score

    return score


def _dedup_claims(claims):
    seen = set()
    out = []

    for c in claims:
        key = (
            _norm_str(c.get("property_name")),
            _norm_str(c.get("raw_value_text")),
            _norm_str(c.get("raw_unit_text")),
            _norm_str(c.get("raw_method_text")),
            _norm_str(c.get("source_material_label")),
            _norm_str(c.get("source_medium_label")),
            _norm_str(c.get("row_label")),
            _norm_str(c.get("column_header")),
            _norm_str(c.get("panel_label")),
            _norm_str(c.get("source_type")),
            int(c.get("page", -1)) if str(c.get("page", "")).strip().lstrip("-").isdigit() else -1,
            int(c.get("candidate_row_id", -1)) if str(c.get("candidate_row_id", "")).strip().lstrip("-").isdigit() else -1,
            _norm_str(c.get("applies_to_target_material")),
            _norm_str(c.get("applies_to_target_media")),
            _norm_str(c.get("evidence_text"))[:200],
        )
        if key not in seen:
            seen.add(key)
            out.append(c)

    return out


def _filter_claims_for_property(claims, prop, target_media):
    xs = [c for c in claims if _norm_str(c.get("property_name")) == prop and _material_ok(c)]
    return _dedup_claims(xs)


def _pick_best_value_claim(prop, claims, target_media):
    candidates = _filter_claims_for_property(claims, prop, target_media)
    candidates = [c for c in candidates if _normalized_claim_value(prop, c) != "None"]
    if not candidates:
        return None
    return max(candidates, key=lambda c: _claim_score_for_value(c, prop, target_media))


def _pick_best_method_claim(prop, claims, target_media):
    candidates = _filter_claims_for_property(claims, prop, target_media)
    candidates = [
        c for c in candidates
        if _canonical_method(
            prop,
            c.get("measurement_method"),
            c.get("raw_method_text"),
            c.get("column_header"),
            c.get("evidence_text")
        ) != "None"
    ]
    if not candidates:
        return None
    return max(candidates, key=lambda c: _claim_score_for_method(c, prop, target_media))


def _pick_best_origin_claim(prop, claims, target_media):
    candidates = _filter_claims_for_property(claims, prop, target_media)
    candidates = [
        c for c in candidates
        if _canonical_source_origin(
            c.get("source_origin"),
            c.get("evidence_text"),
            c.get("column_header")
        ) != "Not specified"
    ]
    if not candidates:
        return None
    return max(candidates, key=lambda c: _claim_score_for_origin(c, prop, target_media))


def _claim_to_key(c):
    return (
        _norm_str(c.get("property_name")),
        _norm_str(c.get("raw_value_text")),
        _norm_str(c.get("raw_unit_text")),
        _norm_str(c.get("raw_method_text")),
        _norm_str(c.get("source_material_label")),
        _norm_str(c.get("source_medium_label")),
        _norm_str(c.get("row_label")),
        _norm_str(c.get("column_header")),
        _norm_str(c.get("panel_label")),
        _norm_str(c.get("source_type")),
        int(c.get("page", -1)) if str(c.get("page", "")).strip().lstrip("-").isdigit() else -1,
        int(c.get("candidate_row_id", -1)) if str(c.get("candidate_row_id", "")).strip().lstrip("-").isdigit() else -1,
    )


def _unique_claim_list(claim_list):
    seen = set()
    out = []
    for c in claim_list:
        if c is None:
            continue
        k = _claim_to_key(c)
        if k not in seen:
            seen.add(k)
            out.append(c)
    return out


def _build_property_result(prop, claims, target_media):
    value_claim = _pick_best_value_claim(prop, claims, target_media)

    value = _normalized_claim_value(prop, value_claim) if value_claim else "None"

    if value == "None":
        return {
            prop: "None",
            f"{prop}_unit": "None",
            f"{prop}_measurement_method": "None",
            f"{prop}_source": "Not specified",
            "_used_claims": [],
        }

    method_claim = _pick_best_method_claim(prop, claims, target_media)
    origin_claim = _pick_best_origin_claim(prop, claims, target_media)

    used_claims = _unique_claim_list([value_claim, method_claim, origin_claim])

    unit = _pchem_output_unit(prop, value)

    method = (
        _canonical_method(
            prop,
            method_claim.get("measurement_method"),
            method_claim.get("raw_method_text"),
            method_claim.get("column_header"),
            method_claim.get("evidence_text")
        ) if method_claim else "None"
    )
    origin = (
        _canonical_source_origin(
            origin_claim.get("source_origin"),
            origin_claim.get("evidence_text"),
            origin_claim.get("column_header")
        ) if origin_claim else "Not specified"
    )

    return {
        prop: value,
        f"{prop}_unit": unit,
        f"{prop}_measurement_method": method,
        f"{prop}_source": origin,
        "_used_claims": used_claims,
    }


def _collect_pages_from_claims(claims):
    pages = []
    for c in claims:
        p = c.get("page", None)
        try:
            p = int(p)
            if p >= 1:
                pages.append(p)
        except Exception:
            pass
    pages = sorted(set(pages))
    if not pages:
        return "None"
    return ", ".join(map(str, pages))


def _derive_primary_source_type(used_claims):
    if not used_claims:
        return "none"

    kinds = set(_canonical_claim_source_kind(c.get("source_type")) for c in used_claims)

    kinds.discard("")
    if not kinds:
        return "none"
    if len(kinds) == 1:
        one = list(kinds)[0]
        if one in {"text", "figure", "table"}:
            return one
        return "mixed"

    return "mixed"


def _build_evidence_summary(prop_results):
    parts = []
    for prop in ["core_size", "hydrodynamic_size", "surface_charge", "surface_area"]:
        val = prop_results.get(prop, "None")
        method = prop_results.get(f"{prop}_measurement_method", "None")

        if val != "None" and method != "None":
            parts.append(f"{prop}={val} ({method})")
        elif val != "None":
            parts.append(f"{prop}={val}")
        elif method != "None":
            parts.append(f"{prop}_method={method}")

    if not parts:
        return "No usable physicochemical evidence consolidated"
    return "; ".join(parts)


def _has_any_usable_pchem_value(record):
    keys = [
        "core_size",
        "hydrodynamic_size",
        "surface_charge",
        "surface_area",
        "core_size_measurement_method",
        "hydrodynamic_size_measurement_method",
        "surface_charge_measurement_method",
        "surface_area_measurement_method",
    ]
    return any(_norm_str(record.get(k, "None")) != "None" for k in keys)


def _pchem_property_needs_backfill(record, prop):
    val = _norm_str(record.get(prop, "None"))
    method = _norm_str(record.get(f"{prop}_measurement_method", "None"))
    source = _norm_str(record.get(f"{prop}_source", "Not specified"))

    return (
        val in {"", "None"}
        or method in {"", "None"}
        or source in {"", "Not specified"}
    )


def _find_missing_pchem_properties(record_result):
    records = record_result.get("records", [])
    if not records:
        return PCHEM_PROPERTIES[:]

    rec = records[0]
    missing = []
    for prop in PCHEM_PROPERTIES:
        if _pchem_property_needs_backfill(rec, prop):
            missing.append(prop)
    return missing

In [ ]:
# Step 6.4: PChem multimodal prompt builders and claim extraction
def _build_pchem_multimodal_user_content(
    material_row: pd.Series,
    candidate_rows: pd.DataFrame,
    full_text: str,
    top_n_candidates: int = 6,
    max_snippets: int = 12,
    page_window: int = 1
):
    candidate_rows = candidate_rows.copy()

    if "page" not in candidate_rows.columns:
        candidate_rows["page"] = pd.Series(dtype="float")
    if "image_path" not in candidate_rows.columns:
        candidate_rows["image_path"] = pd.Series(dtype="object")
    if "caption" not in candidate_rows.columns:
        candidate_rows["caption"] = "None"
    if "image_property" not in candidate_rows.columns:
        candidate_rows["image_property"] = "unknown"
    if "row_id" not in candidate_rows.columns:
        candidate_rows = candidate_rows.reset_index(drop=True)
        candidate_rows["row_id"] = candidate_rows.index.astype(int)

    candidate_rows["page"] = pd.to_numeric(candidate_rows["page"], errors="coerce")
    candidate_rows = candidate_rows[candidate_rows["page"].notna()].copy()
    candidate_rows["page"] = candidate_rows["page"].astype(int)

    candidate_rows = (
        candidate_rows
        .sort_values(["page", "row_id"])
        .drop_duplicates(subset=["image_path"])
        .head(top_n_candidates)
        .copy()
    )

    expanded_pages = set()
    for p in candidate_rows["page"].tolist():
        for q in range(max(1, p - page_window), p + page_window + 1):
            expanded_pages.add(q)

    global_context, context_pages = get_material_relevant_text_snippets(
        full_text=full_text,
        material_name=material_row["material_name"],
        composition=material_row["composition"],
        media=material_row["media"],
        extra_pages=sorted(expanded_pages),
        max_snippets=max_snippets,
        max_chars=12000
    )

    user_intro = f"""
Target material metadata:
- material: {material_row['material']}
- material_name: {material_row['material_name']}
- composition: {material_row['composition']}
- media: {material_row['media']}
"""

    content = [{"type": "text", "text": user_intro}]

    if global_context.strip():
        content.append({
            "type": "text",
            "text": (
                "Global text evidence snippets "
                "(shared method/source statements may appear here even if the material name is not repeated):\n"
                f"{global_context}"
            )
        })

    for row in candidate_rows.itertuples(index=False):
        page_context = get_page_context(
            full_text=full_text,
            center_page=int(row.page),
            window=page_window,
            max_chars=3500
        )

        content.append({
            "type": "text",
            "text": (
                f"Candidate source block\n"
                f"- row_id: {int(row.row_id)}\n"
                f"- page: {int(row.page)}\n"
                f"- source_type: {row.image_property}\n"
                f"- caption: {row.caption}\n\n"
                f"Nearby page text:\n{page_context}"
            )
        })

        if pd.notna(row.image_path) and os.path.exists(row.image_path):
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": image_to_data_url(row.image_path),
                    "detail": "high"
                }
            })

    return content


def extract_pchem_claims_for_one_material(
    material_row: pd.Series,
    candidate_rows: pd.DataFrame,
    full_text: str,
    client,
    model: str,
    temperature: float = 0,
    top_n_candidates: int = 6,
    max_snippets: int = 12,
    page_window: int = 1
) -> dict:
    content = _build_pchem_multimodal_user_content(
        material_row=material_row,
        candidate_rows=candidate_rows,
        full_text=full_text,
        top_n_candidates=top_n_candidates,
        max_snippets=max_snippets,
        page_window=page_window
    )

    system_prompt = """
You are an expert curator extracting physicochemical evidence for nanomaterials from scientific papers.

This is NOT the final consolidation step.
Your job is to harvest candidate claims related to these properties:
- core_size
- hydrodynamic_size
- surface_charge
- surface_area

You must use ALL supplied evidence jointly:
- global text snippets
- nearby page text
- figure/table caption
- figure/table image

Important:
- Do NOT choose a single best source.
- Do NOT produce one final record yet.
- Emit one claim per detectable evidence fragment.
- Exhaustive extraction is more important than selectivity.

A claim may be partial.
Examples:
- text says "measured by DLS" but no number -> emit claim with value="None", raw_method_text="DLS"
- table cell says "110.8 nm" under "water" -> emit claim with raw_value_text="110.8", raw_unit_text="nm", source_medium_label="water"
- column says "SF medium" and row says "150SiO2-NP" -> preserve those labels
- text says "sizes were provided by the manufacturer" -> source_origin="Manufacturer"

Rules:
1. Extract only claims that may apply to the exact target material, but if uncertain do NOT silently drop; use applies_to_target_material="Unclear".
2. Keep claims even if incomplete.
3. Use candidate_row_id = -1 for text-only claims not tied to a specific figure/table row.
4. source_type examples:
   - text
   - caption
   - figure
   - table
   - text+caption
   - text+figure
   - figure+caption
   - text+figure+caption
   - text+table
   - table+caption
   - text+table+caption
5. source_origin must be one of:
   - Experiment
   - Manufacturer
   - Not specified
6. applies_to_target_material must be:
   - Yes
   - No
   - Unclear
7. applies_to_target_media must be:
   - Yes
   - No
   - Not relevant
   - Unclear
8. raw_value_text must preserve the source expression as closely as possible.
9. raw_unit_text must preserve the source unit as closely as possible.
10. raw_method_text must preserve the source method wording as closely as possible.
11. source_material_label should preserve row label / legend label / text label when available.
12. source_medium_label should preserve the source condition label such as:
   - water
   - ultrapure water
   - PBS
   - SF medium
   - serum-free DMEM
   - RPMI 1640 + 10% FBS
13. row_label / column_header / panel_label should be filled whenever visible or inferable from the supplied evidence.
14. If nothing is found, return {"claims": []}.

Unit normalization rules:
- For the structured fields `value` and `unit`, convert values into the designated canonical unit whenever the conversion is direct and unambiguous.
- Preserve the original source expression in `raw_value_text` and `raw_unit_text`.
- Canonical units:
  - core_size: nm
  - hydrodynamic_size: nm
  - surface_charge: mV
  - surface_area: m2/g
- Conversion examples:
  - 1 µm -> 1000 nm
  - 0.5 µm -> 500 nm
  - 10 Å -> 1 nm
  - 0.03 V -> 30 mV
  - 15000 cm2/g -> 1.5 m2/g
- If the original value is already in the canonical unit, keep it unchanged.
- If conversion is not direct or would require unsupported assumptions, set `value`="None" and `unit`="None", and preserve the source expression in `raw_value_text` and `raw_unit_text`.
- `value` must contain only the numeric value.
- `unit` must contain only the canonical unit label.

Output rules:
- Return strings, not nulls.
- Use "None" when a field is missing.
- value/unit/measurement_method can be partial, but raw_* fields should preserve evidence whenever possible.
"""

    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        response_format={
            "type": "json_schema",
            "json_schema": PCHEM_CLAIM_SCHEMA
        },
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": content},
        ],
    )

    return json.loads(resp.choices[0].message.content)


def extract_pchem_claims_for_missing_properties_for_one_material(
    material_row: pd.Series,
    candidate_rows: pd.DataFrame,
    full_text: str,
    missing_properties: list,
    client,
    model: str,
    temperature: float = 0,
    top_n_candidates: int = 6,
    max_snippets: int = 12,
    page_window: int = 1
) -> dict:
    content = _build_pchem_multimodal_user_content(
        material_row=material_row,
        candidate_rows=candidate_rows,
        full_text=full_text,
        top_n_candidates=top_n_candidates,
        max_snippets=max_snippets,
        page_window=page_window
    )

    missing_text = ", ".join(missing_properties) if missing_properties else "None"

    system_prompt = f"""
You are doing a SECOND PASS backfill for missing physicochemical data.

Target properties to backfill ONLY:
{missing_text}

Use ALL supplied evidence jointly:
- global text snippets
- nearby page text
- figure/table caption
- figure/table image

Goal:
- recover claims that may have been missed in the first pass
- especially method/source statements in text
- especially condition-specific values in table/figure
- especially water / PBS / SF medium / cell medium distinctions

Rules:
1. Emit claims ONLY for the target missing properties listed above.
2. Do NOT silently skip a plausible claim just because it is incomplete.
3. Preserve raw_value_text / raw_unit_text / raw_method_text / source_medium_label / row_label / column_header / panel_label whenever possible.
4. Use candidate_row_id = -1 for text-only claims.
5. source_origin must be one of:
   - Experiment
   - Manufacturer
   - Not specified
6. applies_to_target_material must be:
   - Yes
   - No
   - Unclear
7. applies_to_target_media must be:
   - Yes
   - No
   - Not relevant
   - Unclear
8. If nothing new is found, return {{"claims": []}}.

Unit normalization rules:
- For the structured fields `value` and `unit`, convert values into the designated canonical unit whenever the conversion is direct and unambiguous.
- Preserve the original source expression in `raw_value_text` and `raw_unit_text`.
- Canonical units:
  - core_size: nm
  - hydrodynamic_size: nm
  - surface_charge: mV
  - surface_area: m2/g
- Conversion examples:
  - 1 µm -> 1000 nm
  - 0.5 µm -> 500 nm
  - 10 Å -> 1 nm
  - 0.03 V -> 30 mV
  - 15000 cm2/g -> 1.5 m2/g
- If conversion is not direct or would require unsupported assumptions, set `value`="None" and `unit`="None", and preserve the source expression in `raw_value_text` and `raw_unit_text`.
- `value` must contain only the numeric value.
- `unit` must contain only the canonical unit label.

Output rules:
- Return strings, not nulls.
- Use "None" when a field is missing.
"""

    resp = client.chat.completions.create(
        model=model,
        temperature=temperature,
        response_format={
            "type": "json_schema",
            "json_schema": PCHEM_CLAIM_SCHEMA
        },
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": content},
        ],
    )

    return json.loads(resp.choices[0].message.content)

In [ ]:
# Step 6.5: PChem claim consolidation
def consolidate_pchem_claims_for_one_material(
    material_row: pd.Series,
    claims: list,
    client=None,
    model: str = None,
    temperature: float = 0
) -> dict:
    claims = claims or []
    claims = _dedup_claims(claims)

    target_media = material_row.get("media", "None")

    core_res = _build_property_result("core_size", claims, target_media)
    hydro_res = _build_property_result("hydrodynamic_size", claims, target_media)
    charge_res = _build_property_result("surface_charge", claims, target_media)
    area_res = _build_property_result("surface_area", claims, target_media)

    used_claims = _unique_claim_list(
        core_res.pop("_used_claims", []) +
        hydro_res.pop("_used_claims", []) +
        charge_res.pop("_used_claims", []) +
        area_res.pop("_used_claims", [])
    )

    record = {
        "material": material_row["material"],
        "material_name": material_row["material_name"],
        "composition": material_row["composition"],
        "media": material_row["media"],

        **core_res,
        **hydro_res,
        **charge_res,
        **area_res,

        "evidence_pages": _collect_pages_from_claims(used_claims),
        "primary_source_type": _derive_primary_source_type(used_claims),
        "evidence_summary": _build_evidence_summary({
            **core_res, **hydro_res, **charge_res, **area_res
        }),
    }

    if not _has_any_usable_pchem_value(record):
        return {"records": []}

    return {"records": [record]}

def annotate_pchem_claim_scores_minimal(df_pchem_claims: pd.DataFrame) -> pd.DataFrame:
    """
    Minimal version that attaches consolidation scores to the PChem claim export.
    In the existing PChem logic, there are three scores rather than one:
    - value_score
    - method_score
    - origin_score
    """
    if df_pchem_claims is None or df_pchem_claims.empty:
        return df_pchem_claims

    df = df_pchem_claims.copy()

    df["value_score"] = df.apply(
        lambda r: _claim_score_for_value(
            r.to_dict(),
            r["property_name"],
            r.get("media", "None")
        ),
        axis=1
    )

    df["method_score"] = df.apply(
        lambda r: _claim_score_for_method(
            r.to_dict(),
            r["property_name"],
            r.get("media", "None")
        ),
        axis=1
    )

    df["origin_score"] = df.apply(
        lambda r: _claim_score_for_origin(
            r.to_dict(),
            r["property_name"],
            r.get("media", "None")
        ),
        axis=1
    )

    return df

In [ ]:
# Step 6.6: PChem extraction wrappers
def extract_pchem_data_for_one_material(
    material_row: pd.Series,
    candidate_rows: pd.DataFrame,
    full_text: str,
    client,
    model: str,
    temperature: float = 0,
    top_n_candidates: int = 6,
    max_snippets: int = 12,
    page_window: int = 1,
    return_claims: bool = False
):
    claims_result = extract_pchem_claims_for_one_material(
        material_row=material_row,
        candidate_rows=candidate_rows,
        full_text=full_text,
        client=client,
        model=model,
        temperature=temperature,
        top_n_candidates=top_n_candidates,
        max_snippets=max_snippets,
        page_window=page_window
    )

    combined_claims = claims_result.get("claims", [])

    record_result = consolidate_pchem_claims_for_one_material(
        material_row=material_row,
        claims=combined_claims,
        client=client,
        model=model,
        temperature=temperature
    )

    missing_properties = _find_missing_pchem_properties(record_result)

    if missing_properties:
        backfill_result = extract_pchem_claims_for_missing_properties_for_one_material(
            material_row=material_row,
            candidate_rows=candidate_rows,
            full_text=full_text,
            missing_properties=missing_properties,
            client=client,
            model=model,
            temperature=temperature,
            top_n_candidates=top_n_candidates,
            max_snippets=max_snippets,
            page_window=page_window
        )

        combined_claims = _dedup_claims(
            combined_claims + backfill_result.get("claims", [])
        )

        record_result = consolidate_pchem_claims_for_one_material(
            material_row=material_row,
            claims=combined_claims,
            client=client,
            model=model,
            temperature=temperature
        )

    if return_claims:
        return record_result, {"claims": combined_claims}

    return record_result


def extract_pchem_data_for_materials_in_one_paper(
    df_materials_one: pd.DataFrame,
    df_pchem_candidates_one: pd.DataFrame,
    full_text_one: str,
    client,
    model: str,
    temperature: float = 0,
    top_n_candidates: int = 6,
    max_snippets: int = 12,
    page_window: int = 1,
    return_claims: bool = False
):
    record_rows = []
    claim_rows = []

    material_cols = ["pdf_key", "material", "material_name", "composition", "media"]
    df_materials_run = df_materials_one[material_cols].drop_duplicates().copy()

    for mat_row in df_materials_run.itertuples(index=False):
        if df_pchem_candidates_one is not None and not df_pchem_candidates_one.empty:
            candidate_rows_one = df_pchem_candidates_one[
                df_pchem_candidates_one["material"] == mat_row.material
            ].copy()
        else:
            candidate_rows_one = pd.DataFrame(
                columns=["material", "row_id", "page", "image_property", "caption", "image_path"]
            )

        try:
            record_result, claims_result = extract_pchem_data_for_one_material(
                material_row=pd.Series({
                    "material": mat_row.material,
                    "material_name": mat_row.material_name,
                    "composition": mat_row.composition,
                    "media": mat_row.media,
                }),
                candidate_rows=candidate_rows_one,
                full_text=full_text_one,
                client=client,
                model=model,
                temperature=temperature,
                top_n_candidates=top_n_candidates,
                max_snippets=max_snippets,
                page_window=page_window,
                return_claims=True
            )

            for rec in record_result.get("records", []):
                rec["pdf_key"] = mat_row.pdf_key
                record_rows.append(rec)

            for claim in claims_result.get("claims", []):
                claim["pdf_key"] = mat_row.pdf_key
                claim["material"] = mat_row.material
                claim["material_name"] = mat_row.material_name
                claim["composition"] = mat_row.composition
                claim["media"] = mat_row.media
                claim_rows.append(claim)

        except Exception as e:
            print(f"[ERROR] {mat_row.material}: {e}")

    df_pchem_one = pd.DataFrame(record_rows)
    df_pchem_claims_one = pd.DataFrame(claim_rows)

    if not df_pchem_one.empty:
        preferred_order = [
            "pdf_key",
            "material",
            "material_name",
            "composition",
            "media",

            "core_size",
            "core_size_unit",
            "core_size_measurement_method",
            "core_size_source",

            "hydrodynamic_size",
            "hydrodynamic_size_unit",
            "hydrodynamic_size_measurement_method",
            "hydrodynamic_size_source",

            "surface_charge",
            "surface_charge_unit",
            "surface_charge_measurement_method",
            "surface_charge_source",

            "surface_area",
            "surface_area_unit",
            "surface_area_measurement_method",
            "surface_area_source",

            "evidence_pages",
            "primary_source_type",
            "evidence_summary",
        ]
        df_pchem_one = df_pchem_one[[c for c in preferred_order if c in df_pchem_one.columns]]
        df_pchem_one = df_pchem_one.drop_duplicates().reset_index(drop=True)
        
        df_pchem_claims_one = annotate_pchem_claim_scores_minimal(df_pchem_claims_one)

    if return_claims:
        return df_pchem_one, df_pchem_claims_one

    return df_pchem_one


def clean_pchem_output(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    numeric_cols = [
        "core_size",
        "hydrodynamic_size",
        "surface_charge",
        "surface_area"
    ]

    for col in numeric_cols:
        if col in df.columns:
            df[col + "_num"] = pd.to_numeric(df[col], errors="coerce")

    return df


def add_pchem_unit_columns_for_export(df: pd.DataFrame) -> pd.DataFrame:
    """
    For exporting all_pchem_claims_df:
    Keep long-format claim rows while
    creating sparse wide columns so each *_unit column appears immediately to the right of its descriptor.
    """
    df = df.copy()

    props = ["core_size", "hydrodynamic_size", "surface_charge", "surface_area"]

    if df.empty:
        for prop in props:
            df[prop] = pd.Series(dtype="object")
            df[f"{prop}_unit"] = pd.Series(dtype="object")
        return df

    for prop in props:
        mask = df["property_name"].eq(prop)
        df[prop] = df["value"].where(mask, "None").fillna("None")
        df[f"{prop}_unit"] = df["unit"].where(mask, "None").fillna("None")

    preferred = [
        "pdf_key",
        "title",
        "material",
        "material_name",
        "composition",
        "media",
        "property_name",

        "core_size",
        "core_size_unit",

        "hydrodynamic_size",
        "hydrodynamic_size_unit",

        "surface_charge",
        "surface_charge_unit",

        "surface_area",
        "surface_area_unit",

        "value",
        "unit",
        "measurement_method",
        "source_type",
        "source_origin",
        "page",
        "candidate_row_id",
        "evidence_text",
        "applies_to_target_material",
        "applies_to_target_media",
        "raw_value_text",
        "raw_unit_text",
        "raw_method_text",
        "source_material_label",
        "source_medium_label",
        "row_label",
        "column_header",
        "panel_label",
    ]

    return df[[c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]]


def extract_pchem_data_for_collection(
    paper_texts: list,
    df_all_materials: pd.DataFrame,
    client,
    chat_model_name: str,
    layout_model,
    img_dir: str,
    dpi: int = 200,
    temperature: float = 0,
    top_n_candidates: int = 6,
    max_snippets: int = 12,
    page_window: int = 1,
    debug: bool = False
):
    pchem_df_parts = []
    pchem_claims_df_parts = []

    for paper in paper_texts:
        pdf_key = paper["pdf_key"]
        pdf_path = paper["pdf_path"]
        full_text_one = paper["full_text"]
        title = paper["title"]

        print(f"\n===== Processing PChem: {pdf_key} =====")
        print("Title:", title)

        df_materials_one = df_all_materials[
            df_all_materials["pdf_key"] == pdf_key
        ].copy()

        if df_materials_one.empty:
            print(f"[SKIP] No material rows for pdf_key={pdf_key}")
            continue

        df_images_one = extract_figures_tables_with_caption_merging(
            pdf_file_path=pdf_path,
            key=pdf_key,
            img_dir=img_dir,
            model=layout_model,
            dpi=dpi,
            debug=debug
        )

        if df_images_one.empty:
            print(f"[INFO] No image/table rows for pdf_key={pdf_key}")
            df_images_one = pd.DataFrame(columns=["image_path", "caption", "image_property", "page"])

        df_images_one["pdf_key"] = pdf_key
        df_images_one["title"] = title

        df_pchem_candidates_one = build_pchem_candidates_df(
            df_materials=df_materials_one,
            df_images=df_images_one,
            client=client,
            model=chat_model_name
        )

        if df_pchem_candidates_one.empty:
            df_pchem_candidates_one = pd.DataFrame(
                columns=["material", "row_id", "page", "image_property", "caption", "image_path"]
            )

        df_pchem_one, df_pchem_claims_one = extract_pchem_data_for_materials_in_one_paper(
            df_materials_one=df_materials_one,
            df_pchem_candidates_one=df_pchem_candidates_one,
            full_text_one=full_text_one,
            client=client,
            model=chat_model_name,
            temperature=temperature,
            top_n_candidates=top_n_candidates,
            max_snippets=max_snippets,
            page_window=page_window,
            return_claims=True
        )

        df_pchem_one = clean_pchem_output(df_pchem_one)

        if not df_pchem_one.empty:
            if "title" not in df_pchem_one.columns:
                df_pchem_one.insert(1, "title", title)
            pchem_df_parts.append(df_pchem_one)

        if not df_pchem_claims_one.empty:
            if "title" not in df_pchem_claims_one.columns:
                df_pchem_claims_one["title"] = title
            pchem_claims_df_parts.append(df_pchem_claims_one)

    all_pchem_df = (
        pd.concat(pchem_df_parts, ignore_index=True)
        if pchem_df_parts else pd.DataFrame()
    )

    all_pchem_claims_df = (
        pd.concat(pchem_claims_df_parts, ignore_index=True)
        if pchem_claims_df_parts else pd.DataFrame()
    )

    if not all_pchem_claims_df.empty:
        all_pchem_claims_df = add_pchem_unit_columns_for_export(all_pchem_claims_df)

    return all_pchem_df, all_pchem_claims_df

# Step 7: Select toxicity figure/table candidates

Convert PChem/material outputs into toxicity targets and identify visual evidence likely to contain assay results.

In [ ]:
# Step 7.1: Define toxicity target output columns
TOX_RECORD_COLUMNS = [
    "pdf_key",
    "title",
    "material",
    "material_name",
    "composition",
    "media",

    "core_size",
    "core_size_unit",
    "core_size_measurement_method",
    "core_size_source",

    "hydrodynamic_size",
    "hydrodynamic_size_unit",
    "hydrodynamic_size_measurement_method",
    "hydrodynamic_size_source",

    "surface_charge",
    "surface_charge_unit",
    "surface_charge_measurement_method",
    "surface_charge_source",

    "surface_area",
    "surface_area_unit",
    "surface_area_measurement_method",
    "surface_area_source",

    "cell_assay",
    "cell_name",
    "cell_species",
    "cell_origin",
    "cell_type",

    "exposure_time",
    "exposure_time_unit",
    "exposure_dose",
    "exposure_dose_unit",
    "assay_results",
    "is_synthetic_control",

    "x_title",
    "y_title",

    "endpoint_type",
    "value_origin",
    "evidence_text",
    "cell_assay_raw",

    "source_row_id",
    "source_page",
    "source_type",
    "source_caption",
    "panel_label",
]


TOX_CLAIM_COLUMNS = [
    "claim_id",

    "pdf_key",
    "title",
    "material",
    "material_name",
    "composition",
    "media",

    "core_size",
    "core_size_unit",
    "core_size_measurement_method",
    "core_size_source",

    "hydrodynamic_size",
    "hydrodynamic_size_unit",
    "hydrodynamic_size_measurement_method",
    "hydrodynamic_size_source",

    "surface_charge",
    "surface_charge_unit",
    "surface_charge_measurement_method",
    "surface_charge_source",

    "surface_area",
    "surface_area_unit",
    "surface_area_measurement_method",
    "surface_area_source",

    "cell_assay",
    "cell_name",
    "cell_species",
    "cell_origin",
    "cell_type",

    "exposure_time",
    "exposure_time_unit",
    "exposure_dose",
    "exposure_dose_unit",
    "assay_results",

    "x_title",
    "y_title",

    "endpoint_type",
    "value_origin",
    "panel_label",
    "series_label",

    "source_type",
    "source_caption",
    "page",
    "candidate_row_id",
    "evidence_text",
    "applies_to_target_material",
    "applies_to_target_media",
    "is_direct_datapoint",

    "cell_assay_raw",
    "assay_results_num",

    "material_media_pass",
    "minimum_payload_pass",

    "cell_assay_normalization_confidence",
    "cell_assay_normalization_reason",

    "tox_readout_relation",
    "tox_readout_confidence",
    "tox_readout_reason",

    "tox_claim_score",
]

In [ ]:
# Step 7.2: Build toxicity targets from PChem results
def first_non_none(series):
    for x in series:
        s = str(x).strip() if pd.notna(x) else ""
        if s not in ["", "None", "nan", "NaN", "null"]:
            return x
    return "None"


def build_tox_targets_from_pchem_for_one_paper(df_pchem_one: pd.DataFrame) -> pd.DataFrame:
    """
    Create target rows for toxicity extraction from the current paper's PChem results.
    Unit of analysis: material + media.
    """
    if df_pchem_one is None or df_pchem_one.empty:
        return pd.DataFrame(columns=[
            "pdf_key", "material", "material_name", "composition", "media",
            "core_size", "core_size_measurement_method", "core_size_source",
            "hydrodynamic_size", "hydrodynamic_size_measurement_method", "hydrodynamic_size_source",
            "surface_charge", "surface_charge_measurement_method", "surface_charge_source",
            "surface_area", "surface_area_measurement_method", "surface_area_source",
        ])

    df = df_pchem_one.copy()

    required_cols = [
        "pdf_key", "material", "material_name", "composition", "media",
        "core_size", "core_size_measurement_method", "core_size_source",
        "hydrodynamic_size", "hydrodynamic_size_measurement_method", "hydrodynamic_size_source",
        "surface_charge", "surface_charge_measurement_method", "surface_charge_source",
        "surface_area", "surface_area_measurement_method", "surface_area_source",
    ]
    for col in required_cols:
        if col not in df.columns:
            df[col] = "None"

    agg_cols = {
        "pdf_key": first_non_none,
        "material_name": first_non_none,
        "composition": first_non_none,

        "core_size": first_non_none,
        "core_size_measurement_method": first_non_none,
        "core_size_source": first_non_none,

        "hydrodynamic_size": first_non_none,
        "hydrodynamic_size_measurement_method": first_non_none,
        "hydrodynamic_size_source": first_non_none,

        "surface_charge": first_non_none,
        "surface_charge_measurement_method": first_non_none,
        "surface_charge_source": first_non_none,

        "surface_area": first_non_none,
        "surface_area_measurement_method": first_non_none,
        "surface_area_source": first_non_none,
    }

    df_tox_targets_one = (
        df.groupby(["material", "media"], dropna=False, as_index=False)
          .agg(agg_cols)
          .copy()
    )

    col_order = [
        "pdf_key", "material", "material_name", "composition", "media",
        "core_size", "core_size_measurement_method", "core_size_source",
        "hydrodynamic_size", "hydrodynamic_size_measurement_method", "hydrodynamic_size_source",
        "surface_charge", "surface_charge_measurement_method", "surface_charge_source",
        "surface_area", "surface_area_measurement_method", "surface_area_source",
    ]
    df_tox_targets_one = df_tox_targets_one[[c for c in col_order if c in df_tox_targets_one.columns]]

    return df_tox_targets_one

In [ ]:
# Step 7.3: Select toxicity figure/table candidates
TOX_SELECTOR_SCHEMA = {
    "name": "toxicity_assay_figure_table_selector",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "materials": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "material": {"type": "string"},
                        "selected_items": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "additionalProperties": False,
                                "properties": {
                                    "row_id": {"type": "integer"},
                                    "page": {"type": "integer"},
                                    "image_property": {"type": "string"},
                                    "caption": {"type": "string"}
                                },
                                "required": ["row_id", "page", "image_property", "caption"]
                            }
                        }
                    },
                    "required": ["material", "selected_items"]
                }
            }
        },
        "required": ["materials"]
    }
}


def prepare_tox_caption_candidates(df_images_one: pd.DataFrame) -> pd.DataFrame:
    df = df_images_one.copy().reset_index(drop=True)
    df["row_id"] = df.index.astype(int)

    keep_cols = ["row_id", "page", "image_property", "caption"]
    for col in keep_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column in df_images_one: {col}")

    return df[keep_cols]


def select_tox_figures_tables_from_captions(
    df_tox_targets_one: pd.DataFrame,
    df_images_one: pd.DataFrame,
    client,
    model,
    temperature: float = 0,
    event_rows: list = None,
    pdf_key: str = "None",
    title: str = "None",
    material: str = "MULTI_TARGETS",
) -> dict:
    if event_rows is None:
        event_rows = []

    target_records = (
        df_tox_targets_one[["material", "material_name", "composition", "media"]]
        .drop_duplicates()
        .fillna("None")
        .to_dict(orient="records")
    )

    candidates_df = prepare_tox_caption_candidates(df_images_one)
    candidates = candidates_df.to_dict(orient="records")

    system_prompt = """
You are an expert in nanotoxicology data curation.

Your task is NOT to extract numeric assay values yet.
Your task is to decide which figure(s) or table(s) are the best sources for toxicity assay data extraction
for each target nanomaterial condition.

Input:
- A list of target nanomaterial rows, each defined by material / material_name / composition / media
- A list of figure/table captions from the same paper

Selection goal:
For each target, choose the figure/table items that are most likely to contain toxicity-assay datapoints
for the exact material condition.

Prefer captions that mention:
- cell viability
- cytotoxicity
- toxicity
- dose-dependent or concentration-dependent effects
- time-dependent effects
- specific assay names (MTT, XTT, WST-1, CCK-8, LDH, NRU, alamarBlue, ATP)
- apoptosis / necrosis
- cell line / monocyte / macrophage / epithelial cell / fibroblast / etc.
- exposure concentration and/or time

Do NOT prefer captions mainly about:
- synthesis
- characterization only
- TEM/SEM morphology only
- hydrodynamic size / zeta potential / BET / XRD only
- dispersion stability only
- uptake only, unless toxicity results are explicitly included
- oxidative stress
- DCF / DCFH-DA / carboxy-H2DCFDA
- TBARS
- MDA
- lipid peroxidation
- RT-qPCR
- qPCR
- mRNA expression
- gene expression
- cytokine release
- ELISA
- inflammation markers
- HLA-DR / CD11b / CD14 / immune activation markers

Decision rule:
- If a caption falls under both Prefer and Do NOT prefer, Prefer takes priority.

Rules:
- Work only from the captions and metadata provided.
- Return up to 5 selected items per target.
- If no good candidate exists, return an empty selected_items list for that target.
"""

    user_prompt = f"""
Target rows:
{json.dumps(target_records, ensure_ascii=False, indent=2)}

Figure/Table candidates:
{json.dumps(candidates, ensure_ascii=False, indent=2)}

Return JSON only.
"""

    resp = _tox_logged_chat_completion_create(
        client=client,
        model=model,
        temperature=temperature,
        response_format={
            "type": "json_schema",
            "json_schema": TOX_SELECTOR_SCHEMA
        },
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],

        event_rows=event_rows,
        stage="tox_selector_api",
        pdf_key=pdf_key,
        title=title,
        material=material,

        api_call_name="select_tox_figures_tables_from_captions",
        candidate_row_id=-1,
        source_page=-1,
        caption_len=-1,
        n_images=len(candidates),
    )

    return json.loads(resp.choices[0].message.content)


def tox_selector_result_to_df(selector_result: dict) -> pd.DataFrame:
    rows = []
    for mat_obj in selector_result.get("materials", []):
        material = mat_obj["material"]
        for item in mat_obj.get("selected_items", []):
            rows.append({
                "material": material,
                "row_id": item["row_id"],
                "page": item["page"],
                "image_property": item["image_property"],
                "caption": item["caption"],
            })
    return pd.DataFrame(rows)


def build_tox_candidates_df(df_tox_selector_one: pd.DataFrame, df_images_one: pd.DataFrame) -> pd.DataFrame:
    if df_tox_selector_one is None or df_tox_selector_one.empty:
        return pd.DataFrame(columns=["material", "row_id", "page", "image_property", "caption", "image_path"])

    df_images_idx = df_images_one.reset_index().rename(columns={"index": "row_id"})

    df_out = df_tox_selector_one.merge(
        df_images_idx,
        on="row_id",
        how="left",
        suffixes=("_sel", "")
    )

    if "page_sel" in df_out.columns:
        if "page" in df_out.columns:
            df_out["page"] = df_out["page"].fillna(df_out["page_sel"])
        else:
            df_out["page"] = df_out["page_sel"]
        df_out.drop(columns=["page_sel"], inplace=True, errors="ignore")

    if "image_property_sel" in df_out.columns:
        if "image_property" in df_out.columns:
            df_out["image_property"] = df_out["image_property"].fillna(df_out["image_property_sel"])
        else:
            df_out["image_property"] = df_out["image_property_sel"]
        df_out.drop(columns=["image_property_sel"], inplace=True, errors="ignore")

    if "caption_sel" in df_out.columns:
        if "caption" in df_out.columns:
            df_out["caption"] = df_out["caption"].fillna(df_out["caption_sel"])
        else:
            df_out["caption"] = df_out["caption_sel"]
        df_out.drop(columns=["caption_sel"], inplace=True, errors="ignore")

    preferred = ["material", "row_id", "page", "image_property", "caption", "image_path"]
    for col in preferred:
        if col not in df_out.columns:
            df_out[col] = "None"

    return df_out[preferred + [c for c in df_out.columns if c not in preferred]]

In [ ]:
# Step 7.4: Ensure selected toxicity candidates retain material metadata
def ensure_material_in_tox_candidates(
    df_tox_candidates_one: pd.DataFrame,
    df_tox_targets_one: pd.DataFrame
) -> pd.DataFrame:
    df_out = df_tox_candidates_one.copy()

    if df_out.empty:
        empty_cols = ["material", "row_id", "page", "image_property", "caption", "image_path"]
        return pd.DataFrame(columns=empty_cols + [c for c in df_tox_targets_one.columns if c != "pdf_key"])

    if "material" not in df_out.columns:
        raise ValueError("df_tox_candidates_one must contain 'material' column.")

    meta_cols = [
        "material", "material_name", "composition", "media",
        "core_size", "core_size_measurement_method", "core_size_source",
        "hydrodynamic_size", "hydrodynamic_size_measurement_method", "hydrodynamic_size_source",
        "surface_charge", "surface_charge_measurement_method", "surface_charge_source",
        "surface_area", "surface_area_measurement_method", "surface_area_source",
    ]
    meta_df = df_tox_targets_one[meta_cols].drop_duplicates()

    df_out = df_out.merge(
        meta_df,
        on="material",
        how="left",
        suffixes=("", "_target")
    )

    for col in [
        "material_name", "composition", "media",
        "core_size", "core_size_measurement_method", "core_size_source",
        "hydrodynamic_size", "hydrodynamic_size_measurement_method", "hydrodynamic_size_source",
        "surface_charge", "surface_charge_measurement_method", "surface_charge_source",
        "surface_area", "surface_area_measurement_method", "surface_area_source",
    ]:
        alt = f"{col}_target"
        if alt in df_out.columns:
            if col in df_out.columns:
                df_out[col] = df_out[col].fillna(df_out[alt])
            else:
                df_out[col] = df_out[alt]
            df_out.drop(columns=[alt], inplace=True, errors="ignore")

    return df_out

# Step 8: Extract toxicity data

Extract toxicity assay records from candidate figures/tables and supporting text, validate material-media-cell combinations, and postprocess records.

In [ ]:
# Step 8.1: Define toxicity export columns
TOX_RECORD_COLUMNS = [
    "pdf_key",
    "title",
    "material",
    "material_name",
    "composition",
    "media",

    "core_size",
    "core_size_unit",
    "core_size_measurement_method",
    "core_size_source",

    "hydrodynamic_size",
    "hydrodynamic_size_unit",
    "hydrodynamic_size_measurement_method",
    "hydrodynamic_size_source",

    "surface_charge",
    "surface_charge_unit",
    "surface_charge_measurement_method",
    "surface_charge_source",

    "surface_area",
    "surface_area_unit",
    "surface_area_measurement_method",
    "surface_area_source",

    "cell_assay",
    "cell_name",
    "cell_species",
    "cell_origin",
    "cell_type",

    "exposure_time",
    "exposure_time_unit",
    "exposure_dose",
    "exposure_dose_unit",
    "assay_results",

    "x_title",
    "y_title",

    "endpoint_type",
    "value_origin",
    "evidence_text",
    "cell_assay_raw",

    "source_row_id",
    "source_page",
    "source_type",
    "source_caption",
    "panel_label",
]

TOX_EVENT_LOG_COLUMNS = [
    "timestamp",
    "level",
    "status",
    "stage",
    "pdf_key",
    "title",
    "material",
    "extracted",
    "n_records",
    "n_claims",
    "error_type",
    "message",

    "api_call_name",
    "api_call_seq",
    "payload_bytes",
    "payload_chars",
    "payload_serialize_ok",
    "payload_serialize_error",
    "candidate_row_id",
    "source_page",
    "caption_len",
    "n_images",
]

TOX_EXPORT_SORT_PRIORITY = [
    "title",
    "material",
    "cell_assay",
    "cell_name",
    "exposure_time",
    "exposure_dose",
]

In [ ]:
# Step 8.2: Define toxicity extraction logging helpers
def _tox_now_str():
    return datetime.now(timezone("Asia/Seoul")).strftime("%Y-%m-%d %H:%M:%S %Z")

def _append_tox_event(
    event_rows: list,
    level: str,
    status: str,
    stage: str,
    pdf_key: str = "None",
    title: str = "None",
    material: str = "None",
    extracted: str = "No",
    n_records: int = 0,
    n_claims: int = 0,
    error_type: str = "None",
    message: str = "None",

    api_call_name: str = "None",
    api_call_seq: int = 0,
    payload_bytes: int = -1,
    payload_chars: int = -1,
    payload_serialize_ok: str = "Unknown",
    payload_serialize_error: str = "None",
    candidate_row_id: int = -1,
    source_page: int = -1,
    caption_len: int = -1,
    n_images: int = 0,
):
    event_rows.append({
        "timestamp": _tox_now_str(),
        "level": str(level),
        "status": str(status),
        "stage": str(stage),
        "pdf_key": str(pdf_key),
        "title": str(title),
        "material": str(material),
        "extracted": str(extracted),
        "n_records": int(n_records),
        "n_claims": int(n_claims),
        "error_type": str(error_type),
        "message": str(message),

        "api_call_name": str(api_call_name),
        "api_call_seq": int(api_call_seq),
        "payload_bytes": int(payload_bytes),
        "payload_chars": int(payload_chars),
        "payload_serialize_ok": str(payload_serialize_ok),
        "payload_serialize_error": str(payload_serialize_error),
        "candidate_row_id": int(candidate_row_id),
        "source_page": int(source_page),
        "caption_len": int(caption_len),
        "n_images": int(n_images),
    })

def _build_tox_event_log_df(event_rows: list) -> pd.DataFrame:
    df = pd.DataFrame(event_rows)
    if df.empty:
        return pd.DataFrame(columns=TOX_EVENT_LOG_COLUMNS)

    for col in TOX_EVENT_LOG_COLUMNS:
        if col not in df.columns:
            df[col] = "None" if col not in {"n_records", "n_claims"} else 0

    return df[TOX_EVENT_LOG_COLUMNS].copy()

In [ ]:
# Step 8.3: Define unit-aware toxicity target helpers
def first_non_none(series):
    for x in series:
        s = str(x).strip() if pd.notna(x) else ""
        if s not in ["", "None", "nan", "NaN", "null"]:
            return x
    return "None"


def build_tox_targets_from_pchem_for_one_paper(df_pchem_one: pd.DataFrame) -> pd.DataFrame:
    """
    Create target rows for toxicity extraction from the current paper's PChem results.
    Unit of analysis: material + media.
    Include unit columns in the transferred context.
    """
    if df_pchem_one is None or df_pchem_one.empty:
        return pd.DataFrame(columns=[
            "pdf_key", "material", "material_name", "composition", "media",

            "core_size", "core_size_unit", "core_size_measurement_method", "core_size_source",
            "hydrodynamic_size", "hydrodynamic_size_unit", "hydrodynamic_size_measurement_method", "hydrodynamic_size_source",
            "surface_charge", "surface_charge_unit", "surface_charge_measurement_method", "surface_charge_source",
            "surface_area", "surface_area_unit", "surface_area_measurement_method", "surface_area_source",
        ])

    df = df_pchem_one.copy()

    required_cols = [
        "pdf_key", "material", "material_name", "composition", "media",

        "core_size", "core_size_unit", "core_size_measurement_method", "core_size_source",
        "hydrodynamic_size", "hydrodynamic_size_unit", "hydrodynamic_size_measurement_method", "hydrodynamic_size_source",
        "surface_charge", "surface_charge_unit", "surface_charge_measurement_method", "surface_charge_source",
        "surface_area", "surface_area_unit", "surface_area_measurement_method", "surface_area_source",
    ]
    for col in required_cols:
        if col not in df.columns:
            df[col] = "None"

    agg_cols = {
        "pdf_key": first_non_none,
        "material_name": first_non_none,
        "composition": first_non_none,

        "core_size": first_non_none,
        "core_size_unit": first_non_none,
        "core_size_measurement_method": first_non_none,
        "core_size_source": first_non_none,

        "hydrodynamic_size": first_non_none,
        "hydrodynamic_size_unit": first_non_none,
        "hydrodynamic_size_measurement_method": first_non_none,
        "hydrodynamic_size_source": first_non_none,

        "surface_charge": first_non_none,
        "surface_charge_unit": first_non_none,
        "surface_charge_measurement_method": first_non_none,
        "surface_charge_source": first_non_none,

        "surface_area": first_non_none,
        "surface_area_unit": first_non_none,
        "surface_area_measurement_method": first_non_none,
        "surface_area_source": first_non_none,
    }

    df_tox_targets_one = (
        df.groupby(["material", "media"], dropna=False, as_index=False)
          .agg(agg_cols)
          .copy()
    )

    col_order = [
        "pdf_key", "material", "material_name", "composition", "media",

        "core_size", "core_size_unit", "core_size_measurement_method", "core_size_source",
        "hydrodynamic_size", "hydrodynamic_size_unit", "hydrodynamic_size_measurement_method", "hydrodynamic_size_source",
        "surface_charge", "surface_charge_unit", "surface_charge_measurement_method", "surface_charge_source",
        "surface_area", "surface_area_unit", "surface_area_measurement_method", "surface_area_source",
    ]
    df_tox_targets_one = df_tox_targets_one[[c for c in col_order if c in df_tox_targets_one.columns]]

    return df_tox_targets_one


def ensure_material_in_tox_candidates(
    df_tox_candidates_one: pd.DataFrame,
    df_tox_targets_one: pd.DataFrame
) -> pd.DataFrame:
    df_out = df_tox_candidates_one.copy()

    if df_out.empty:
        empty_cols = ["material", "row_id", "page", "image_property", "caption", "image_path"]
        target_cols = [c for c in df_tox_targets_one.columns if c != "pdf_key"]
        merged_cols = list(dict.fromkeys(empty_cols + target_cols))  # Remove duplicates
        return pd.DataFrame(columns=merged_cols)

    if "material" not in df_out.columns:
        raise ValueError("df_tox_candidates_one must contain 'material' column.")

    meta_cols = [
        "material", "material_name", "composition", "media",

        "core_size", "core_size_unit", "core_size_measurement_method", "core_size_source",
        "hydrodynamic_size", "hydrodynamic_size_unit", "hydrodynamic_size_measurement_method", "hydrodynamic_size_source",
        "surface_charge", "surface_charge_unit", "surface_charge_measurement_method", "surface_charge_source",
        "surface_area", "surface_area_unit", "surface_area_measurement_method", "surface_area_source",
    ]
    meta_df = df_tox_targets_one[meta_cols].drop_duplicates()

    df_out = df_out.merge(
        meta_df,
        on="material",
        how="left",
        suffixes=("", "_target")
    )

    for col in [
        "material_name", "composition", "media",

        "core_size", "core_size_unit", "core_size_measurement_method", "core_size_source",
        "hydrodynamic_size", "hydrodynamic_size_unit", "hydrodynamic_size_measurement_method", "hydrodynamic_size_source",
        "surface_charge", "surface_charge_unit", "surface_charge_measurement_method", "surface_charge_source",
        "surface_area", "surface_area_unit", "surface_area_measurement_method", "surface_area_source",
    ]:
        alt = f"{col}_target"
        if alt in df_out.columns:
            if col in df_out.columns:
                df_out[col] = df_out[col].fillna(df_out[alt])
            else:
                df_out[col] = df_out[alt]
            df_out.drop(columns=[alt], inplace=True, errors="ignore")

    return df_out

In [ ]:
# Step 8.4: Define toxicity extraction schemas

TOX_CELL_ASSAY_ENUM = [
    "MTT",
    "MTS",
    "XTT",
    "WST",
    "CCK-8",
    "LDH",
    "NRU",
    "SRB",
    "Alamar blue",
    "Resazurin",
    "Trypan blue",
    "Calcein AM",
    "Live/Dead viability assay",
    "Annexin V/PI",
    "PI staining",
    "CellTiter-Glo",
    "CellTiter-Blue",
    "CytoTox-Glo",
    "ATP",
    "Tox tracker",
    "Luminometric assay",
    "PrestoBlue",
    "Nitroblue Tetrazolium Assay",
    "PicoGreen",
    "TOTO-3",
    "Resazurin assay",
    "CyQuant Assay",
    "CFE",
    "RCC",
    "BrdU",
    "AK",
    "Fluorescein diacetate (FDA) uptake",
    "COMET",
    "Other",
    "None",
]


TOX_HARVEST_CLAIM_SCHEMA = {
    "name": "toxicity_claim_harvest_v6_raw_assay",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "claims": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "claim_id": {"type": "integer"},

                        "material": {"type": "string"},
                        "material_name": {"type": "string"},
                        "composition": {"type": "string"},
                        "media": {"type": "string"},

                        "cell_assay_raw": {"type": "string"},
                        "cell_assay": {"type": "string"},

                        "cell_name": {"type": "string"},
                        "cell_species": {"type": "string"},
                        "cell_origin": {"type": "string"},
                        "cell_type": {"type": "string"},

                        "exposure_time": {"type": "string"},
                        "exposure_time_unit": {"type": "string"},
                        "exposure_dose": {"type": "string"},
                        "exposure_dose_unit": {"type": "string"},
                        "assay_results": {"type": "string"},

                        "x_title": {"type": "string"},
                        "y_title": {"type": "string"},

                        "endpoint_type": {"type": "string"},
                        "value_origin": {"type": "string"},
                        "panel_label": {"type": "string"},
                        "series_label": {"type": "string"},

                        "source_type": {"type": "string"},
                        "source_caption": {"type": "string"},
                        "page": {"type": "integer"},
                        "candidate_row_id": {"type": "integer"},
                        "evidence_text": {"type": "string"},

                        "applies_to_target_material": {"type": "string"},
                        "applies_to_target_media": {"type": "string"},
                        "is_direct_datapoint": {"type": "string"},
                    },
                    "required": [
                        "claim_id",
                        "material",
                        "material_name",
                        "composition",
                        "media",

                        "cell_assay_raw",
                        "cell_assay",

                        "cell_name",
                        "cell_species",
                        "cell_origin",
                        "cell_type",

                        "exposure_time",
                        "exposure_time_unit",
                        "exposure_dose",
                        "exposure_dose_unit",
                        "assay_results",

                        "x_title",
                        "y_title",

                        "endpoint_type",
                        "value_origin",
                        "panel_label",
                        "series_label",

                        "source_type",
                        "source_caption",
                        "page",
                        "candidate_row_id",
                        "evidence_text",

                        "applies_to_target_material",
                        "applies_to_target_media",
                        "is_direct_datapoint",
                    ],
                },
            }
        },
        "required": ["claims"],
    },
}


TOX_REFINEMENT_CLAIM_SCHEMA = {
    "name": "toxicity_claim_refinement_v5_raw_assay",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "claims": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "claim_id": {"type": "integer"},

                        "material": {"type": "string"},
                        "material_name": {"type": "string"},
                        "composition": {"type": "string"},
                        "media": {"type": "string"},

                        "cell_assay_raw": {"type": "string"},
                        "cell_assay": {"type": "string"},

                        "cell_name": {"type": "string"},
                        "cell_species": {"type": "string"},
                        "cell_origin": {"type": "string"},
                        "cell_type": {"type": "string"},

                        "exposure_time": {"type": "string"},
                        "exposure_time_unit": {"type": "string"},
                        "exposure_dose": {"type": "string"},
                        "exposure_dose_unit": {"type": "string"},
                        "assay_results": {"type": "string"},

                        "x_title": {"type": "string"},
                        "y_title": {"type": "string"},

                        "endpoint_type": {"type": "string"},
                        "value_origin": {"type": "string"},
                        "panel_label": {"type": "string"},
                        "series_label": {"type": "string"},

                        "source_type": {"type": "string"},
                        "source_caption": {"type": "string"},
                        "page": {"type": "integer"},
                        "candidate_row_id": {"type": "integer"},
                        "evidence_text": {"type": "string"},

                        "applies_to_target_material": {"type": "string"},
                        "applies_to_target_media": {"type": "string"},
                        "is_direct_datapoint": {"type": "string"},
                    },
                    "required": [
                        "claim_id",
                        "material",
                        "material_name",
                        "composition",
                        "media",

                        "cell_assay_raw",
                        "cell_assay",

                        "cell_name",
                        "cell_species",
                        "cell_origin",
                        "cell_type",

                        "exposure_time",
                        "exposure_time_unit",
                        "exposure_dose",
                        "exposure_dose_unit",
                        "assay_results",

                        "x_title",
                        "y_title",

                        "endpoint_type",
                        "value_origin",
                        "panel_label",
                        "series_label",

                        "source_type",
                        "source_caption",
                        "page",
                        "candidate_row_id",
                        "evidence_text",

                        "applies_to_target_material",
                        "applies_to_target_media",
                        "is_direct_datapoint",
                    ],
                },
            }
        },
        "required": ["claims"],
    },
}

In [ ]:
# Step 8.5: Define Methods-guided toxicity context schema
TOX_METHODS_CONTEXT_SCHEMA = {
    "name": "toxicity_methods_context_inventory",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "assay_contexts": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "context_id": {"type": "string"},

                        "cell_assay": {"type": "string"},
                        "cell_name": {"type": "string"},
                        "cell_species": {"type": "string"},
                        "cell_origin": {"type": "string"},
                        "cell_type": {"type": "string"},

                        "candidate_exposure_times": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "candidate_exposure_time_unit": {"type": "string"},

                        "candidate_doses": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "candidate_dose_unit": {"type": "string"},

                        "candidate_media": {"type": "string"},

                        "materials_mentioned": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "endpoint_hints": {
                            "type": "array",
                            "items": {"type": "string"}
                        },

                        "evidence_text": {"type": "string"}
                    },
                    "required": [
                        "context_id",
                        "cell_assay",
                        "cell_name",
                        "cell_species",
                        "cell_origin",
                        "cell_type",
                        "candidate_exposure_times",
                        "candidate_exposure_time_unit",
                        "candidate_doses",
                        "candidate_dose_unit",
                        "candidate_media",
                        "materials_mentioned",
                        "endpoint_hints",
                        "evidence_text"
                    ]
                }
            }
        },
        "required": ["assay_contexts"]
    }
}


TOX_METHODS_COMBO_VALIDATION_SCHEMA = {
    "name": "tox_methods_combo_validation",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "material_name": {"type": "string"},
                        "media": {"type": "string"},
                        "cell_name": {"type": "string"},
                        "judgment": {
                            "type": "string",
                            "enum": ["Yes", "No", "Unclear"]
                        }
                    },
                    "required": [
                        "material_name",
                        "media",
                        "cell_name",
                        "judgment"
                    ]
                }
            }
        },
        "required": ["items"]
    }
}

In [ ]:
# Step 8.6: Define toxicity normalization and evidence helpers
def _tox_norm_str(x):
    if x is None:
        return ""
    return str(x).replace("\x00", " ").strip()


def _tox_is_missing(x):
    s = _tox_norm_str(x).lower()
    return s in {"", "none", "nan", "null", "n/a", "na", "not available", "missing"}


def _tox_clean_numeric_value(x):
    if _tox_is_missing(x):
        return "None"

    s = _tox_norm_str(x).replace(",", "")
    s = s.replace("−", "-").replace("–", "-").replace("—", "-")
    m = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
    if not m:
        return "None"
    return m.group(0)


def _tox_clean_time_value(x):
    if _tox_is_missing(x):
        return "None"
    return _tox_clean_numeric_value(x)


def _tox_clean_dose_value(x):
    if _tox_is_missing(x):
        return "None"
    return _tox_clean_numeric_value(x)


def _tox_clean_time_unit(x):
    if _tox_is_missing(x):
        return "None"

    s = _tox_norm_str(x).lower()

    if s in {"h", "hr", "hrs", "hour", "hours"}:
        return "h"
    if s in {"min", "mins", "minute", "minutes"}:
        return "min"
    if s in {"d", "day", "days"}:
        return "day"

    return _tox_norm_str(x)


def _tox_clean_dose_unit(x):
    if _tox_is_missing(x):
        return "None"

    s = _tox_norm_str(x).replace("μ", "µ").strip()
    s_low = s.lower()

    mapping = {
        "ug/ml": "µg/mL",
        "µg/ml": "µg/mL",
        "mg/ml": "mg/mL",
        "ng/ml": "ng/mL",
        "ug/106 cells": "µg/10^6 cells",
        "µg/106 cells": "µg/10^6 cells",
        "ppm/106 cells": "ppm/10^6 cells",
        "ppm": "ppm",
    }

    return mapping.get(s_low, s)


def _tox_canonical_endpoint_type(x):
    if _tox_is_missing(x):
        return "None"

    s = _tox_norm_str(x).lower()

    if "viability" in s:
        return "viability"
    if "cytotoxic" in s:
        return "cytotoxicity"
    if "ldh" in s:
        return "LDH_release"
    if "apopt" in s:
        return "apoptosis_rate"
    if "necro" in s:
        return "necrosis_rate"
    if "ros" in s:
        return "ROS_related_toxicity"
    if "cell death" in s:
        return "cell_death"
    if "atp" in s:
        return "ATP_related_viability"
    if "metabolic" in s:
        return "metabolic activity"

    return _tox_norm_str(x)


def _tox_normalize_assay_result(value, endpoint_type=None):
    val = _tox_clean_numeric_value(value)
    if val == "None":
        return "None"

    try:
        num = float(val)
    except Exception:
        return "None"

    return _format_float_str(num)

TOX_FRACTION_TO_PERCENT_UPPER = 2.0


def _tox_canonical_claim_source_kind(source_type):
    if _tox_is_missing(source_type):
        return "text"

    s = _tox_norm_str(source_type).lower()

    has_text = "text" in s
    has_caption = "caption" in s
    has_figure = "figure" in s
    has_table = "table" in s

    kinds = sum([has_text, has_caption, has_figure, has_table])

    if has_table and kinds == 1:
        return "table"
    if has_figure and kinds == 1:
        return "figure"
    if has_text and kinds == 1:
        return "text"
    if has_caption and kinds == 1:
        return "caption"
    if kinds >= 2:
        return "mixed"

    return "text"


def _tox_media_ok(status):
    s = _tox_norm_str(status).lower()
    return s in {"yes", "unclear", "not relevant", ""}


def _tox_material_ok(status):
    s = _tox_norm_str(status).lower()
    return s in {"yes", "unclear", ""}


def _tox_dedup_claims(claims: list) -> list:
    seen = set()
    out = []

    for c in claims:
        key = (
            _tox_norm_str(c.get("material")),
            _tox_norm_str(c.get("material_name")),
            _tox_norm_str(c.get("composition")),
            _tox_norm_str(c.get("media")),

            _tox_norm_str(c.get("cell_assay")),
            _tox_norm_str(c.get("cell_name")),
            _tox_norm_str(c.get("cell_species")),
            _tox_norm_str(c.get("cell_origin")),
            _tox_norm_str(c.get("cell_type")),

            _tox_norm_str(c.get("exposure_time")),
            _tox_norm_str(c.get("exposure_time_unit")),
            _tox_norm_str(c.get("exposure_dose")),
            _tox_norm_str(c.get("exposure_dose_unit")),
            _tox_norm_str(c.get("assay_results")),

            _tox_norm_str(c.get("endpoint_type")),
            _tox_norm_str(c.get("value_origin")),
            _tox_norm_str(c.get("panel_label")),
            _tox_norm_str(c.get("series_label")),

            _tox_norm_str(c.get("source_type")),
            _tox_norm_str(c.get("source_caption")),
            int(c.get("page", -1)) if str(c.get("page", "")).strip().lstrip("-").isdigit() else -1,
            int(c.get("candidate_row_id", -1)) if str(c.get("candidate_row_id", "")).strip().lstrip("-").isdigit() else -1,

            _tox_norm_str(c.get("evidence_text"))[:250],
            _tox_norm_str(c.get("applies_to_target_material")),
            _tox_norm_str(c.get("applies_to_target_media")),
            _tox_norm_str(c.get("is_direct_datapoint")),
        )
        if key not in seen:
            seen.add(key)
            out.append(c)

    return out


def _tox_claim_score(claim: dict) -> int:
    if not _tox_material_ok(claim.get("applies_to_target_material")):
        return -10**9
    if not _tox_media_ok(claim.get("applies_to_target_media")):
        return -10**9

    score = 0

    result_val = _tox_normalize_assay_result(
        claim.get("assay_results"),
        claim.get("endpoint_type")
    )
    if result_val != "None":
        score += 100
    else:
        score -= 1000

    dose_val = _tox_clean_dose_value(claim.get("exposure_dose"))
    time_val = _tox_clean_time_value(claim.get("exposure_time"))
    if dose_val != "None":
        score += 15
    if time_val != "None":
        score += 10

    if not _tox_is_missing(claim.get("cell_assay")):
        score += 15
    if not _tox_is_missing(claim.get("cell_name")):
        score += 12
    if not _tox_is_missing(claim.get("cell_type")):
        score += 6

    if not _tox_is_missing(claim.get("exposure_time_unit")):
        score += 3
    if not _tox_is_missing(claim.get("exposure_dose_unit")):
        score += 3

    source_kind = _tox_canonical_claim_source_kind(claim.get("source_type"))
    score += {
        "table": 25,
        "mixed": 22,
        "figure": 18,
        "text": 14,
        "caption": 5,
    }.get(source_kind, 0)

    vo = _tox_norm_str(claim.get("value_origin")).lower()
    if vo in {"text_reported", "direct", "direct_datapoint"}:
        score += 20
    elif vo in {"derived"}:
        score += 14
    elif vo in {"estimated_from_plot", "plot_estimated"}:
        score += 10

    if _tox_norm_str(claim.get("is_direct_datapoint")).lower() == "yes":
        score += 10

    mat_status = _tox_norm_str(claim.get("applies_to_target_material")).lower()
    media_status = _tox_norm_str(claim.get("applies_to_target_media")).lower()

    if mat_status == "yes":
        score += 30
    elif mat_status == "unclear":
        score += 10

    if media_status == "yes":
        score += 15
    elif media_status == "unclear":
        score += 8
    elif media_status == "not relevant":
        score += 10

    if len(_tox_norm_str(claim.get("evidence_text"))) >= 15:
        score += 4

    return score


def _tox_claim_group_key(claim: dict):
    return (
        _tox_norm_str(claim.get("material")),
        _tox_norm_str(claim.get("material_name")),
        _tox_norm_str(claim.get("composition")),
        _tox_norm_str(claim.get("media")),

        _tox_norm_str(claim.get("cell_assay")),
        _tox_norm_str(claim.get("cell_name")),
        _tox_norm_str(claim.get("cell_species")),
        _tox_norm_str(claim.get("cell_origin")),
        _tox_norm_str(claim.get("cell_type")),

        _tox_clean_time_value(claim.get("exposure_time")),
        _tox_clean_time_unit(claim.get("exposure_time_unit")),
        _tox_clean_dose_value(claim.get("exposure_dose")),
        _tox_clean_dose_unit(claim.get("exposure_dose_unit")),
    )


def _tox_has_minimum_payload(claim: dict) -> bool:
    result_val = _tox_normalize_assay_result(claim.get("assay_results"), claim.get("endpoint_type"))
    if result_val == "None":
        return False

    dose_val = _tox_clean_dose_value(claim.get("exposure_dose"))
    time_val = _tox_clean_time_value(claim.get("exposure_time"))

    if dose_val == "None" and time_val == "None" and _tox_is_missing(claim.get("cell_assay")):
        return False

    return True


def _tox_sort_records(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df

    out = df.copy()

    required_cols = [
        "title",
        "material",
        "cell_assay",
        "cell_name",
        "exposure_time",
        "exposure_dose",
    ]
    for col in required_cols:
        if col not in out.columns:
            out[col] = "None"

    # text sort keys
    text_cols = [
        "title",
        "material",
        "cell_assay",
        "cell_name",
    ]
    for col in text_cols:
        out[f"__{col}_key__"] = out[col].map(lambda x: _tox_norm_str(x).lower())

    # numeric sort keys
    out["__exposure_time_num__"] = pd.to_numeric(out["exposure_time"], errors="coerce")
    out["__exposure_dose_num__"] = pd.to_numeric(out["exposure_dose"], errors="coerce")

    out = out.sort_values(
        by=[
            "__title_key__",
            "__material_key__",
            "__cell_assay_key__",
            "__cell_name_key__",
            "__exposure_time_num__",
            "__exposure_dose_num__",
        ],
        kind="stable",
        na_position="last",
    ).reset_index(drop=True)

    out.drop(
        columns=[
            "__title_key__",
            "__material_key__",
            "__cell_assay_key__",
            "__cell_name_key__",
            "__exposure_time_num__",
            "__exposure_dose_num__",
        ],
        inplace=True,
        errors="ignore",
    )

    return out


def _tox_sort_claims(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df

    out = df.copy()

    required_cols = [
        "title",
        "material",
        "cell_assay",
        "cell_name",
        "page",
        "source_caption",
        "panel_label",
        "exposure_time",
        "exposure_dose",
    ]
    for col in required_cols:
        if col not in out.columns:
            out[col] = "None"

    text_cols = [
        "title",
        "material",
        "cell_assay",
        "cell_name",
        "source_caption",
        "panel_label",
    ]
    for col in text_cols:
        out[f"__{col}_key__"] = out[col].map(lambda x: _tox_norm_str(x).lower())

    out["__page_num__"] = pd.to_numeric(out["page"], errors="coerce")
    out["__exposure_time_num__"] = pd.to_numeric(out["exposure_time"], errors="coerce")
    out["__exposure_dose_num__"] = pd.to_numeric(out["exposure_dose"], errors="coerce")

    out = out.sort_values(
        by=[
            "__title_key__",
            "__material_key__",
            "__cell_assay_key__",
            "__cell_name_key__",
            "__page_num__",
            "__source_caption_key__",
            "__panel_label_key__",
            "__exposure_time_num__",
            "__exposure_dose_num__",
        ],
        kind="stable",
        na_position="last",
    ).reset_index(drop=True)

    out.drop(
        columns=[
            "__title_key__",
            "__material_key__",
            "__cell_assay_key__",
            "__cell_name_key__",
            "__source_caption_key__",
            "__panel_label_key__",
            "__page_num__",
            "__exposure_time_num__",
            "__exposure_dose_num__",
        ],
        inplace=True,
        errors="ignore",
    )

    return out


def _tox_prepare_images_df(df_images_one: pd.DataFrame) -> pd.DataFrame:
    df = df_images_one.copy()

    if "caption" not in df.columns:
        df["caption"] = ""

    if "caption_clean" in df.columns:
        df["caption"] = df["caption_clean"].fillna("").astype(str)
    else:
        df["caption"] = df["caption"].fillna("").map(normalize_text_deterministic)

    df["caption"] = df["caption"].map(_tox_sanitize_llm_text)

    return df


def _tox_prepare_candidate_rows(candidate_rows: pd.DataFrame, top_n_candidates: int = 4) -> pd.DataFrame:
    df = candidate_rows.copy()

    if "page" not in df.columns:
        df["page"] = pd.Series(dtype="float")
    if "image_path" not in df.columns:
        df["image_path"] = pd.Series(dtype="object")
    if "caption" not in df.columns:
        df["caption"] = "None"
    if "image_property" not in df.columns:
        df["image_property"] = "unknown"
    if "row_id" not in df.columns:
        df = df.reset_index(drop=True)
        df["row_id"] = df.index.astype(int)

    df["caption"] = df["caption"].fillna("").map(normalize_text_deterministic)
    df["page"] = pd.to_numeric(df["page"], errors="coerce")
    df = df[df["page"].notna()].copy()
    df["page"] = df["page"].astype(int)

    df = (
        df.sort_values(["page", "row_id"])
          .drop_duplicates(subset=["image_path"])
          .head(top_n_candidates)
          .copy()
    )

    return df


def _tox_collect_candidate_pages(candidate_rows: pd.DataFrame, page_window: int = 1) -> list:
    pages = set()
    for p in candidate_rows["page"].tolist():
        for q in range(max(1, int(p) - page_window), int(p) + page_window + 1):
            pages.add(q)
    return sorted(pages)


def _tox_best_non_missing(values):
    cleaned = [_tox_norm_str(v) for v in values if not _tox_is_missing(v)]
    if not cleaned:
        return None

    counts = {}
    order = []
    for v in cleaned:
        if v not in counts:
            counts[v] = 0
            order.append(v)
        counts[v] += 1

    order = sorted(order, key=lambda x: (-counts[x], len(x)))
    return order[0]


def _tox_get_methods_results_text_bundle(
    full_text: str,
    material_name: str,
    composition: str = None,
    media: str = None,
    candidate_pages=None,
    max_snippets: int = 28,
    max_chars: int = 18000
):
    if candidate_pages is None:
        candidate_pages = []

    page_dict = split_full_text_by_page(full_text)
    material_terms, composition_terms, media_terms = build_material_query_terms(
        material_name=material_name,
        composition=composition,
        media=media
    )

    assay_keywords = [
        "mtt", "wst", "wst-1", "wst-8", "cck-8", "ldh", "xtt", "nru",
        "neutral red", "alamarblue", "ros", "apoptosis", "necrosis", "atp",
        "cell viability", "viability", "cytotoxicity", "toxicity",
        "dose dependent", "dose-dependent", "concentration dependent", "concentration-dependent",
        "time dependent", "time-dependent"
    ]

    method_keywords = [
        "materials and methods", "methods", "cells were", "cell culture",
        "incubated", "exposed", "treated", "treatment", "concentration",
        "dose", "doses", "for 24 h", "for 48 h", "for 72 h",
        "after 24 h", "after 48 h", "after 72 h",
        "ug/ml", "µg/ml", "μg/ml", "mg/ml", "ng/ml", "ppm",
        "assay", "measured by", "measured using"
    ]

    cell_keywords = [
        "cell", "cells", "calu-3", "thp-1", "a549", "macrophage", "monocyte",
        "epithelial", "fibroblast", "human", "mouse", "rat"
    ]

    scored = []
    seen = set()

    for page_num, page_text in page_dict.items():
        page_norm = normalize_match_text(page_text)

        page_heading_bonus = 0
        if "materials and methods" in page_norm or re.search(r"\bmethods\b", page_norm):
            page_heading_bonus += 4
        if re.search(r"\bresults\b", page_norm):
            page_heading_bonus += 3

        paragraphs = split_page_into_paragraphs(page_text)
        for para in paragraphs:
            norm = normalize_match_text(para)
            if not norm:
                continue

            score = 0

            mat_hits = sum(1 for t in material_terms if t in norm)
            comp_hits = sum(1 for t in composition_terms if t in norm)
            media_hits = sum(1 for t in media_terms if t and t in norm)
            assay_hits = sum(1 for kw in assay_keywords if kw in norm)
            method_hits = sum(1 for kw in method_keywords if kw in norm)
            cell_hits = sum(1 for kw in cell_keywords if kw in norm)

            score += min(mat_hits, 4) * 2
            score += min(comp_hits, 2) * 2
            score += min(media_hits, 3) * 2
            score += min(assay_hits, 5) * 3
            score += min(method_hits, 5) * 2
            score += min(cell_hits, 4) * 2
            score += page_heading_bonus

            if page_num in candidate_pages:
                score += 3

            strong_context = (
                assay_hits >= 1 and method_hits >= 1
            ) or (
                assay_hits >= 2
            ) or (
                method_hits >= 2 and cell_hits >= 1
            )

            if score >= 6 and ((mat_hits + comp_hits + media_hits) > 0 or strong_context or page_num in candidate_pages):
                key = normalize_match_text(para)
                if key not in seen:
                    seen.add(key)
                    scored.append({
                        "page": page_num,
                        "score": score,
                        "text": para
                    })

    scored = sorted(scored, key=lambda x: (-x["score"], x["page"]))

    selected = []
    used_chars = 0
    used_pages = []

    for item in scored:
        block = f"[Page {item['page']} | score={item['score']}]\n{item['text']}"
        if used_chars + len(block) > max_chars:
            continue
        selected.append(block)
        used_chars += len(block)
        used_pages.append(item["page"])
        if len(selected) >= max_snippets:
            break

    return "\n\n".join(selected), sorted(set(used_pages))


def _tox_overlay_claim(base_claim: dict, revised_claim: dict) -> dict:
    out = dict(base_claim)

    for k, v in revised_claim.items():
        if k == "claim_id":
            out[k] = int(v)
            continue

        if k in {"page", "candidate_row_id"}:
            try:
                iv = int(v)
            except Exception:
                iv = None

            if iv is None:
                continue

            if k == "page":
                if iv >= 1:
                    out[k] = iv
            else:
                if iv >= 0 or _tox_norm_str(out.get(k, "")).strip() == "":
                    out[k] = iv
            continue

        if not _tox_is_missing(v):
            out[k] = v

    return out


def _tox_merge_claim_versions(original_claims: list, revised_claims: list) -> list:
    orig_map = {}
    for c in original_claims:
        cid = int(c.get("claim_id", -1))
        orig_map[cid] = dict(c)

    for rc in revised_claims:
        cid = int(rc.get("claim_id", -1))
        if cid in orig_map:
            orig_map[cid] = _tox_overlay_claim(orig_map[cid], rc)
        else:
            orig_map[cid] = dict(rc)

    return [orig_map[k] for k in sorted(orig_map.keys())]


def _tox_propagate_shared_fields(claims: list) -> list:
    if not claims:
        return claims

    claims = [dict(c) for c in claims]

    group_map = {}
    for i, c in enumerate(claims):
        key = (
            _tox_norm_str(c.get("material")),
            int(c.get("candidate_row_id", -1)) if str(c.get("candidate_row_id", "")).strip().lstrip("-").isdigit() else -1,
            int(c.get("page", -1)) if str(c.get("page", "")).strip().lstrip("-").isdigit() else -1,
            _tox_norm_str(c.get("source_caption")),
            _tox_norm_str(c.get("panel_label")),
        )
        group_map.setdefault(key, []).append(i)

    fields_to_propagate = [
        "cell_assay_raw",
        "cell_assay",
        "cell_name",
        "cell_species",
        "cell_origin",
        "cell_type",
        "exposure_time",
        "exposure_time_unit",
        "endpoint_type",
        "value_origin",
        "applies_to_target_material",
        "applies_to_target_media",
        "x_title",
        "y_title",
    ]

    for _, idxs in group_map.items():
        group_claims = [claims[i] for i in idxs]

        consensus = {}
        for field in fields_to_propagate:
            best = _tox_best_non_missing([c.get(field) for c in group_claims])
            if best is not None:
                consensus[field] = best

        for i in idxs:
            for field, value in consensus.items():
                if _tox_is_missing(claims[i].get(field)):
                    claims[i][field] = value

    return claims

def _tox_prepare_methods_text(methods_text_one: str) -> str:
    if not isinstance(methods_text_one, str):
        return "None"

    methods_text_one = methods_text_one.strip()
    if methods_text_one in {"", "None"}:
        return "None"

    return normalize_text_deterministic(methods_text_one)


def _tox_next_claim_id(claims: list) -> int:
    ids = []
    for c in claims or []:
        try:
            ids.append(int(c.get("claim_id", -1)))
        except Exception:
            pass

    ids = [x for x in ids if x >= 1]
    return (max(ids) + 1) if ids else 1


def _tox_minify_claims_for_expansion(claims: list) -> list:
    keep_cols = [
        "claim_id",
        "cell_assay",
        "cell_name",
        "exposure_time",
        "exposure_time_unit",
        "exposure_dose",
        "exposure_dose_unit",
        "assay_results",
        "endpoint_type",
        "panel_label",
        "series_label",
        "page",
        "candidate_row_id",
        "source_caption",
        "applies_to_target_material",
        "applies_to_target_media",
    ]

    out = []
    for c in claims or []:
        row = {}
        for k in keep_cols:
            v = c.get(k, "None")
            row[k] = v if not _tox_is_missing(v) else "None"
        out.append(row)

    return out


def _tox_build_source_summaries(prepared_rows: pd.DataFrame) -> list:
    rows = []
    for row in prepared_rows.itertuples(index=False):
        rows.append({
            "row_id": int(row.row_id),
            "page": int(row.page),
            "source_type": _tox_norm_str(row.image_property) or "None",
            "caption": _tox_norm_str(row.caption) or "None",
        })
    return rows


def _tox_claim_key_without_id(claim: dict) -> str:
    cc = dict(claim)
    cc.pop("claim_id", None)
    return json.dumps(cc, ensure_ascii=False, sort_keys=True)


def _tox_filter_only_new_claims(existing_claims: list, candidate_new_claims: list) -> list:
    existing_keys = {_tox_claim_key_without_id(c) for c in existing_claims or []}
    out = []

    for c in candidate_new_claims or []:
        k = _tox_claim_key_without_id(c)
        if k not in existing_keys:
            out.append(c)
            existing_keys.add(k)

    return out

def _tox_sanitize_llm_text(x, default: str = "None") -> str:
    if x is None:
        return default

    s = str(x)

    # Remove control characters, including NUL
    s = re.sub(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]", " ", s)

    # Remove lone surrogates, a common cause of JSON serialization failures
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Cs")

    # Safely normalize through UTF-8
    s = s.encode("utf-8", "ignore").decode("utf-8", "ignore")

    # Normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()

    return s if s else default


def _tox_sanitize_llm_jsonable(obj):
    if isinstance(obj, dict):
        return {k: _tox_sanitize_llm_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_tox_sanitize_llm_jsonable(v) for v in obj]
    if isinstance(obj, str):
        return _tox_sanitize_llm_text(obj)
    return obj


def _tox_sanitize_chat_message_content(content):
    """
    Make OpenAI message content JSON-safe.
    - Sanitize text.
    - Process multimodal lists/dicts recursively.
    - Preserve image_url.url and ordinary URL strings as much as possible.
    """
    if isinstance(content, str):
        return _tox_sanitize_llm_text(content)

    if isinstance(content, list):
        return [_tox_sanitize_chat_message_content(x) for x in content]

    if isinstance(content, dict):
        out = {}
        for k, v in content.items():
            # Preserve data URLs and ordinary URLs as much as possible
            if k == "url" and isinstance(v, str):
                out[k] = v.strip()
            else:
                out[k] = _tox_sanitize_chat_message_content(v)
        return out

    return content


def _tox_sanitize_messages_for_openai(messages):
    """
    Make all OpenAI chat messages JSON-safe.
    """
    safe_messages = []

    for msg in messages or []:
        if not isinstance(msg, dict):
            continue

        safe_msg = {}
        for k, v in msg.items():
            if k == "content":
                safe_msg[k] = _tox_sanitize_chat_message_content(v)
            else:
                safe_msg[k] = _tox_sanitize_llm_jsonable(v)

        safe_messages.append(safe_msg)

    return safe_messages


def _tox_make_json_safe(obj):
    """
    Python object -> sanitize -> strict JSON roundtrip
    Purpose:
    - Remove or detect issues such as surrogates, control characters, and NaN values
      locally before making the API call.
    """
    sanitized = _tox_sanitize_llm_jsonable(obj)
    dumped = json.dumps(
        sanitized,
        ensure_ascii=True,
        allow_nan=False
    )
    return json.loads(dumped)


def _tox_material_media_cell_combo_key(material_name, media, cell_name):
    return (
        _tox_norm_str(material_name) or "None",
        _tox_norm_str(media) or "None",
        _tox_norm_str(cell_name) or "None",
    )


def build_unique_material_media_cell_combos_from_claims(claims: list) -> list:
    """
    From currently extracted claims,
    extract unique (material_name, media, cell_name) combinations.
    Exclude a combination from validation if any of the three values is missing.
    """
    seen = set()
    combos = []

    for c in claims or []:
        material_name = _tox_norm_str(c.get("material_name")) or "None"
        media = _tox_norm_str(c.get("media")) or "None"
        cell_name = _tox_norm_str(c.get("cell_name")) or "None"

        if material_name == "None" or media == "None" or cell_name == "None":
            continue

        key = _tox_material_media_cell_combo_key(material_name, media, cell_name)
        if key not in seen:
            seen.add(key)
            combos.append({
                "material_name": material_name,
                "media": media,
                "cell_name": cell_name,
            })

    return combos


def validate_material_media_cell_combos_with_methods(
    methods_text_one: str,
    combos: list,
    client,
    model: str,
    temperature: float = 0,
    event_rows: list = None,
    pdf_key: str = "None",
    title: str = "None",
    material: str = "None",
) -> dict:
    """
    Using only the Methods section,
    judge whether the (material_name, media, cell_name) combination is correct as Yes / No / Unclear.
    """
    if event_rows is None:
        event_rows = []

    if not combos:
        return {"items": []}

    methods_text_clean = _tox_prepare_methods_text(methods_text_one)
    if methods_text_clean == "None":
        return {"items": []}

    methods_text_clean = _tox_sanitize_llm_text(methods_text_clean)

    system_prompt = """
You are validating extracted toxicity combinations using ONLY the paper's Methods section.

Input:
- Methods section text
- A list of extracted unique combinations:
  material_name, media, cell_name

Task:
For each combination, output exactly one judgment:
- Yes
- No
- Unclear

Rules:
1. Use the Methods section only.
2. Check whether the given cell_name is matched with the given media for the given material_name in the assay setup.
3. Distinguish clearly between:
   - cell maintenance/culture medium
   - nanoparticle stock/dispersion medium
   - actual exposure/treatment medium
4. Use actual exposure/treatment medium as the main basis.
5. Do NOT treat water, PBS, ultrapure water, or generic dispersion vehicle as the target media unless Methods explicitly indicate cell exposure in that vehicle.
6. If the Methods support that the combination is correct, output Yes.
7. If the Methods clearly contradict the combination, output No.
8. If the Methods do not allow a confident decision, output Unclear.
9. Return JSON only.
"""

    user_prompt = _tox_sanitize_llm_text(
        f"""Methods section:
{methods_text_clean}

Combinations to validate:
{json.dumps(combos, ensure_ascii=False, indent=2)}

Return JSON only."""
    )

    resp = _tox_logged_chat_completion_create(
        client=client,
        model=model,
        temperature=temperature,
        response_format={
            "type": "json_schema",
            "json_schema": TOX_METHODS_COMBO_VALIDATION_SCHEMA
        },
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],

        event_rows=event_rows,
        stage="methods_combo_validation_api",
        pdf_key=pdf_key,
        title=title,
        material=material,

        api_call_name="validate_material_media_cell_combos_with_methods",
        candidate_row_id=-1,
        source_page=-1,
        caption_len=-1,
        n_images=0,
    )

    return json.loads(resp.choices[0].message.content)


def apply_methods_combo_judgment_to_claims(
    claims: list,
    combo_validation_result: dict
) -> list:
    """
    Apply Methods combo validator results to claims.

    Application rules:
    - judgment == "Yes" -> applies_to_target_media = "Yes"
    - judgment == "No" -> applies_to_target_media = "No"
    - judgment == "Unclear" -> keep the existing value
    """
    items = combo_validation_result.get("items", [])

    judgment_map = {}
    for row in items:
        key = _tox_material_media_cell_combo_key(
            row.get("material_name"),
            row.get("media"),
            row.get("cell_name"),
        )
        judgment_map[key] = _tox_norm_str(row.get("judgment")) or "Unclear"

    out = []
    for c in claims or []:
        cc = dict(c)

        key = _tox_material_media_cell_combo_key(
            cc.get("material_name"),
            cc.get("media"),
            cc.get("cell_name"),
        )

        judgment = judgment_map.get(key, "Unclear")

        if judgment == "Yes":
            cc["applies_to_target_media"] = "Yes"
        elif judgment == "No":
            cc["applies_to_target_media"] = "No"
        # Unclear -> keep the existing value

        out.append(cc)

    return out


def _tox_force_cell_assay_enum(label: str) -> str:
    """
    Force cell_assay to avoid remaining as free-text under all circumstances.
    """
    normalized = _normalize_cell_assay_label(label)
    if normalized in TOX_CELL_ASSAY_ENUM:
        return normalized
    return "Other"


def _tox_finalize_claim_cell_assay_fields(claim: dict) -> dict:
    """
    Do not normalize assays during claim generation/refinement.
    Copy the current raw assay expression into both cell_assay_raw and cell_assay.
    Normalize for the first time during the claim-to-record conversion stage.
    """
    out = dict(claim)

    raw = _tox_norm_str(out.get("cell_assay_raw"))
    cur = _tox_norm_str(out.get("cell_assay"))

    if _tox_is_missing(raw) and not _tox_is_missing(cur):
        raw = cur

    if _tox_is_missing(raw) and _tox_is_missing(cur):
        raw = "None"

    out["cell_assay_raw"] = raw
    out["cell_assay"] = raw

    return out

def _tox_next_api_call_seq(event_rows: list, pdf_key: str, material: str) -> int:
    """
    Increase the API call sequence number within the same pdf_key/material.
    """
    count = 0
    for row in event_rows:
        if (
            str(row.get("pdf_key", "None")) == str(pdf_key)
            and str(row.get("material", "None")) == str(material)
            and str(row.get("level", "")) == "api"
        ):
            count += 1
    return count + 1


def _tox_estimate_json_payload_size(payload_obj):
    """
    Serialize the payload using strict JSON immediately before sending it to the OpenAI SDK
    and return byte size, character size, and serialization success/failure.
    """
    try:
        payload_text = json.dumps(
            payload_obj,
            ensure_ascii=False,
            allow_nan=False
        )
        payload_bytes = len(payload_text.encode("utf-8"))
        payload_chars = len(payload_text)
        return {
            "payload_bytes": payload_bytes,
            "payload_chars": payload_chars,
            "payload_serialize_ok": "Yes",
            "payload_serialize_error": "None",
        }
    except Exception as e:
        return {
            "payload_bytes": -1,
            "payload_chars": -1,
            "payload_serialize_ok": "No",
            "payload_serialize_error": f"{type(e).__name__}: {e}",
        }

def _tox_logged_chat_completion_create(
    *,
    client,
    model: str,
    temperature: float,
    response_format: dict,
    messages: list,

    event_rows: list,
    stage: str,
    pdf_key: str,
    title: str,
    material: str,

    api_call_name: str,
    candidate_row_id: int = -1,
    source_page: int = -1,
    caption_len: int = -1,
    n_images: int = 0,
):
    """
    Before the OpenAI call:
    - Sanitize messages and response_format to be JSON-safe.
    - Pre-check whether strict JSON serialization is possible.
    - Write event logs for both success and failure.
    """

    # 1) Make messages / response_format safe
    safe_messages = _tox_make_json_safe(
        _tox_sanitize_messages_for_openai(messages)
    )
    safe_response_format = _tox_make_json_safe(response_format)

    payload_obj = {
        "model": model,
        "temperature": temperature,
        "response_format": safe_response_format,
        "messages": safe_messages,
    }

    size_info = _tox_estimate_json_payload_size(payload_obj)
    api_call_seq = _tox_next_api_call_seq(
        event_rows=event_rows,
        pdf_key=pdf_key,
        material=material
    )

    # 2) Block the API call if local strict JSON serialization fails
    if size_info["payload_serialize_ok"] == "No":
        err_msg = f"Payload serialization failed before API call: {size_info['payload_serialize_error']}"
        _append_tox_event(
            event_rows=event_rows,
            level="api",
            status="ERROR",
            stage=stage,
            pdf_key=pdf_key,
            title=title,
            material=material,
            extracted="No",
            n_records=0,
            n_claims=0,
            error_type="PayloadSerializationError",
            message=err_msg,

            api_call_name=api_call_name,
            api_call_seq=api_call_seq,
            payload_bytes=size_info["payload_bytes"],
            payload_chars=size_info["payload_chars"],
            payload_serialize_ok=size_info["payload_serialize_ok"],
            payload_serialize_error=size_info["payload_serialize_error"],
            candidate_row_id=candidate_row_id,
            source_page=source_page,
            caption_len=caption_len,
            n_images=n_images,
        )
        raise ValueError(err_msg)

    # 3) Make the actual API call
    try:
        resp = client.chat.completions.create(
            model=model,
            temperature=temperature,
            response_format=safe_response_format,
            messages=safe_messages,
        )

        _append_tox_event(
            event_rows=event_rows,
            level="api",
            status="SUCCESS",
            stage=stage,
            pdf_key=pdf_key,
            title=title,
            material=material,
            extracted="No",
            n_records=0,
            n_claims=0,
            error_type="None",
            message="OpenAI call succeeded",

            api_call_name=api_call_name,
            api_call_seq=api_call_seq,
            payload_bytes=size_info["payload_bytes"],
            payload_chars=size_info["payload_chars"],
            payload_serialize_ok=size_info["payload_serialize_ok"],
            payload_serialize_error=size_info["payload_serialize_error"],
            candidate_row_id=candidate_row_id,
            source_page=source_page,
            caption_len=caption_len,
            n_images=n_images,
        )
        return resp

    except Exception as e:
        _append_tox_event(
            event_rows=event_rows,
            level="api",
            status="ERROR",
            stage=stage,
            pdf_key=pdf_key,
            title=title,
            material=material,
            extracted="No",
            n_records=0,
            n_claims=0,
            error_type=type(e).__name__,
            message=str(e),

            api_call_name=api_call_name,
            api_call_seq=api_call_seq,
            payload_bytes=size_info["payload_bytes"],
            payload_chars=size_info["payload_chars"],
            payload_serialize_ok=size_info["payload_serialize_ok"],
            payload_serialize_error=size_info["payload_serialize_error"],
            candidate_row_id=candidate_row_id,
            source_page=source_page,
            caption_len=caption_len,
            n_images=n_images,
        )
        raise

In [ ]:
# Step 8.7: Stage A toxicity claim harvesting from selected candidates
def _extract_tox_claims_from_single_candidate(
    material_row: pd.Series,
    candidate_row: pd.Series,
    full_text_one: str,
    client,
    model: str,
    claim_id_start: int = 1,
    temperature: float = 0,
    page_window: int = 1,
    event_rows: list = None,
    pdf_key: str = "None",
    title: str = "None",
) -> dict:
    if event_rows is None:
        event_rows = []

    page_context = _tox_sanitize_llm_text(
        get_page_context(
            full_text=full_text_one,
            center_page=int(candidate_row["page"]),
            window=page_window,
            max_chars=5000
        )
    )

    caption_text = _tox_sanitize_llm_text(candidate_row.get("caption", "None"))
    source_type_text = _tox_norm_str(candidate_row.get("image_property", "None")) or "None"

    system_prompt = """
You are extracting toxicity-assay datapoint claims from ONE candidate figure or table.

Priority:
- capture the numeric backbone first
- extract as many visible or table-reported datapoints as possible
- later stages will refine missing metadata and normalize assay naming

Use only the supplied evidence for this candidate:
- target material metadata
- candidate caption
- nearby page text
- candidate image/table

Legend/series inspection is mandatory.

For figures:
- You MUST explicitly inspect panel labels, legends, line labels, color labels, marker labels, bar-pattern labels, and group labels before emitting claims.
- Do NOT extract only a subset of visible conditions if multiple legend-defined series are present and readable.
- If a panel contains multiple series/conditions/materials, check the legend first and map each datapoint to the correct series.
- Use series_label to record the exact legend/series identity whenever available.
- Use panel_label to record the exact panel identity whenever available.
- A separate claim should be emitted for each distinct series × dose/time datapoint that is visible and attributable.
- If a legend is present and readable, failing to inspect it is an extraction error.

For tables:
- Apply the same logic to row labels, column headers, subgroup headers, and footnotes.
- Do not extract only part of a condition grid if other readable rows/columns belong to the same target context.

Rules:
1. Emit one claim per visible/table-reported datapoint whenever possible.
2. If assay_results is visible/readable, output it even if time or dose is missing.
3. Extract exposure_dose if it appears on x-axis, row labels, legends, series labels, or caption.
   Put only the numeric part in exposure_dose and put the unit in exposure_dose_unit.
4. Extract exposure_time if it appears in panel title, legend, caption, or nearby text.
   Put only the numeric part in exposure_time and put the unit in exposure_time_unit.
5. Preserve the assay wording as close to source wording as possible.
   - cell_assay_raw = source-like assay wording
   - cell_assay = SAME source-like assay wording
   - Do NOT normalize assay names here
6. source_type examples:
   - figure
   - table
   - figure+caption
   - table+caption
   - text+figure
   - text+table
   - text+figure+caption
   - text+table+caption
7. applies_to_target_material must be one of:
   - Yes
   - No
   - Unclear
8. applies_to_target_media must be one of:
   - Yes
   - No
   - Not relevant
   - Unclear
9. is_direct_datapoint must be:
   - Yes
   - No
10. If exact numeric bar height is not readable, do not invent.
11. Return strings, not nulls. Use "None" when missing.
12. If the source is a figure and the axis titles are visible, extract:
    - x_title = exact x-axis title
    - y_title = exact y-axis title
13. Preserve axis-title wording as closely as possible, including units.
14. If the source is a table/text, or the axis titles are absent/unreadable, set:
    - x_title = "None"
    - y_title = "None"

Important:
- Do not wait for complete metadata.
- Getting assay_results + any visible dose/time/unit is more important than completeness.
- Return JSON only.
"""

    user_intro = _tox_sanitize_llm_text(f"""
Target material-condition metadata:
- material: {material_row['material']}
- material_name: {material_row['material_name']}
- composition: {material_row['composition']}
- media: {material_row['media']}

Candidate source metadata:
- row_id: {int(candidate_row['row_id'])}
- page: {int(candidate_row['page'])}
- source_type: {source_type_text}
- caption: {caption_text}

Nearby page text:
{page_context}
""")

    content = [{"type": "text", "text": user_intro}]

    if pd.notna(candidate_row.get("image_path")) and os.path.exists(candidate_row["image_path"]):
        content.append({
            "type": "image_url",
            "image_url": {
                "url": image_to_data_url(candidate_row["image_path"]),
                "detail": "high"
            }
        })

    resp = _tox_logged_chat_completion_create(
        client=client,
        model=model,
        temperature=temperature,
        response_format={
            "type": "json_schema",
            "json_schema": TOX_HARVEST_CLAIM_SCHEMA
        },        
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": content},
        ],
        
        event_rows=event_rows,
        stage="candidate_harvest_api",
        pdf_key=pdf_key,
        title=title,
        material=material_row["material"],
        
        api_call_name="_extract_tox_claims_from_single_candidate",
        candidate_row_id=int(candidate_row["row_id"]),
        source_page=int(candidate_row["page"]),
        caption_len=len(caption_text),
        n_images=1,
    )

    
    result = json.loads(resp.choices[0].message.content)

    claims = []
    next_id = claim_id_start

    for c in result.get("claims", []):
        cc = _tox_sanitize_llm_jsonable(dict(c))

        cc["claim_id"] = next_id
        next_id += 1

        cc["material"] = _tox_norm_str(cc.get("material")) or _tox_norm_str(material_row["material"])
        cc["material_name"] = _tox_norm_str(cc.get("material_name")) or _tox_norm_str(material_row["material_name"])
        cc["composition"] = _tox_norm_str(cc.get("composition")) or _tox_norm_str(material_row["composition"])
        cc["media"] = _tox_norm_str(cc.get("media")) or _tox_norm_str(material_row["media"])

        cc["source_caption"] = _tox_norm_str(cc.get("source_caption")) or caption_text
        cc["page"] = int(cc.get("page", int(candidate_row["page"])))
        cc["candidate_row_id"] = int(cc.get("candidate_row_id", int(candidate_row["row_id"])))

        cc["cell_origin"] = "None"
        cc["cell_type"] = "None"

        cc["x_title"] = _tox_norm_str(cc.get("x_title")) or "None"
        cc["y_title"] = _tox_norm_str(cc.get("y_title")) or "None"

        src = _tox_norm_str(cc.get("source_type")).lower()
        if "figure" not in src:
            cc["x_title"] = "None"
            cc["y_title"] = "None"

        cc = _tox_finalize_claim_cell_assay_fields(cc)

        claims.append(cc)

    return {"claims": claims}


def extract_tox_claims_from_candidate_sources_for_one_material(
    material_row: pd.Series,
    candidate_rows: pd.DataFrame,
    full_text_one: str,
    client,
    model: str,
    temperature: float = 0,
    top_n_candidates: int = 4,
    page_window: int = 1,
    event_rows: list = None,
    pdf_key: str = "None",
    title: str = "None",
) -> dict:
    if event_rows is None:
        event_rows = []

    prepared_rows = _tox_prepare_candidate_rows(candidate_rows, top_n_candidates=top_n_candidates)

    all_claims = []
    next_id = 1

    for _, row in prepared_rows.iterrows():
        try:
            one = _extract_tox_claims_from_single_candidate(
                material_row=material_row,
                candidate_row=row,
                full_text_one=full_text_one,
                client=client,
                model=model,
                claim_id_start=next_id,
                temperature=temperature,
                page_window=page_window,
                event_rows=event_rows,
                pdf_key=pdf_key,
                title=title,
            )
            claims = one.get("claims", [])
            all_claims.extend(claims)
            if claims:
                next_id = max(int(x["claim_id"]) for x in all_claims) + 1

        except Exception as e:
            _append_tox_event(
                event_rows=event_rows,
                level="material",
                status="WARN",
                stage="candidate_harvest",
                pdf_key=pdf_key,
                title=title,
                material=material_row["material"],
                extracted="No",
                n_records=0,
                n_claims=0,
                error_type=type(e).__name__,
                message=f"row_id={row.get('row_id', 'None')} | {e}"
            )
            print(
                f"[WARN] candidate harvest failed for material={material_row['material']} "
                f"row_id={row.get('row_id')} page={row.get('page')} "
            f"caption_len={len(_tox_norm_str(row.get('caption')))}: {e}"
            )

    return {"claims": all_claims}

In [ ]:
# Step 8.8: Stage A-2 Methods-guided recall expansion
def extract_tox_methods_contexts_for_one_material(
    methods_text_one: str,
    material_row: pd.Series,
    client,
    model: str,
    temperature: float = 0,
    event_rows: list = None,
    pdf_key: str = "None",
    title: str = "None",
) -> dict:
    if event_rows is None:
        event_rows = []
    
    if not isinstance(methods_text_one, str) or methods_text_one.strip() in {"", "None"}:
        return {"assay_contexts": []}

    system_prompt = """
You are extracting toxicity-assay context hints from a paper's Methods section.

This is NOT final data extraction.
This is NOT a hard constraint generator.

Goal:
- identify assay contexts that may help recover missed toxicity datapoints later
- extract likely assay metadata and likely condition axes
- keep the output permissive and recall-oriented

Important:
- Do NOT assume that every candidate dose/time listed in Methods appears in every figure/table
- Do NOT force complete combinations
- Treat extracted doses/times as candidate hints only
- Keep uncertain information if plausibly stated in Methods

Return one object per assay context.
Use "None" when missing.
Return {"assay_contexts": []} if nothing useful is found.
"""

    user_prompt = _tox_sanitize_llm_text(f"""
Target material-condition metadata:
- material: {material_row['material']}
- material_name: {material_row['material_name']}
- composition: {material_row['composition']}
- media: {material_row['media']}

Methods text:
{methods_text_one}

Return JSON only.
""")

    resp = _tox_logged_chat_completion_create(
        client=client,
        model=model,
        temperature=temperature,
        response_format={
            "type": "json_schema",
            "json_schema": TOX_METHODS_CONTEXT_SCHEMA
        },
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],

        event_rows=event_rows,
        stage="methods_context_inventory_api",
        pdf_key=pdf_key,
        title=title,
        material=material_row["material"],
        
        api_call_name="extract_tox_methods_contexts_for_one_material",
        candidate_row_id=-1,
        source_page=-1,
        caption_len=-1,
        n_images=0,    
    )

    return json.loads(resp.choices[0].message.content)


def expand_tox_claims_with_methods_for_one_material(
    material_row: pd.Series,
    harvested_claims: list,
    candidate_rows: pd.DataFrame,
    full_text_one: str,
    methods_text_one: str,
    client,
    model: str,
    temperature: float = 0,
    top_n_candidates: int = 4,
    max_snippets: int = 28,
    page_window: int = 1,
    event_rows: list = None,
    pdf_key: str = "None",
    title: str = "None",
) -> dict:
    if event_rows is None:
        event_rows = []

    def _local_sanitize_text(x, default: str = "None") -> str:
        if "_tox_sanitize_llm_text" in globals():
            return globals()["_tox_sanitize_llm_text"](x, default=default)

        if x is None:
            return default

        s = str(x)
        s = re.sub(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]", " ", s)
        s = "".join(ch for ch in s if unicodedata.category(ch) != "Cs")
        s = s.encode("utf-8", "ignore").decode("utf-8", "ignore")
        s = re.sub(r"\s+", " ", s).strip()
        return s if s else default

    def _local_sanitize_jsonable(obj):
        if "_tox_sanitize_llm_jsonable" in globals():
            return globals()["_tox_sanitize_llm_jsonable"](obj)

        if isinstance(obj, dict):
            return {k: _local_sanitize_jsonable(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [_local_sanitize_jsonable(v) for v in obj]
        if isinstance(obj, str):
            return _local_sanitize_text(obj)
        return obj

    if not harvested_claims:
        return {"claims": []}

    methods_text_one = _tox_prepare_methods_text(methods_text_one)
    if methods_text_one == "None":
        return {"claims": []}

    methods_text_one = _local_sanitize_text(methods_text_one)

    methods_context_result = extract_tox_methods_contexts_for_one_material(
        methods_text_one=methods_text_one,
        material_row=material_row,
        client=client,
        model=model,
        temperature=temperature,
        event_rows=event_rows,
        pdf_key=pdf_key,
        title=title,
    )

    assay_contexts = methods_context_result.get("assay_contexts", [])
    assay_contexts = _local_sanitize_jsonable(assay_contexts)

    if not assay_contexts:
        return {"claims": []}

    prepared_rows = _tox_prepare_candidate_rows(
        candidate_rows,
        top_n_candidates=top_n_candidates
    )
    if prepared_rows.empty:
        return {"claims": []}

    source_summaries = _local_sanitize_jsonable(
        _tox_build_source_summaries(prepared_rows)
    )

    candidate_pages = _tox_collect_candidate_pages(
        prepared_rows,
        page_window=page_window
    )

    methods_results_bundle, bundle_pages = _tox_get_methods_results_text_bundle(
        full_text=full_text_one,
        material_name=material_row["material_name"],
        composition=material_row["composition"],
        media=material_row["media"],
        candidate_pages=candidate_pages,
        max_snippets=max_snippets,
        max_chars=18000
    )
    methods_results_bundle = _local_sanitize_text(methods_results_bundle)

    row_meta = {}
    for row in prepared_rows.itertuples(index=False):
        row_meta[int(row.row_id)] = {
            "page": int(row.page),
            "caption": _local_sanitize_text(_tox_norm_str(row.caption) or "None"),
            "source_type": _local_sanitize_text(_tox_norm_str(row.image_property) or "None"),
        }

    existing_claims_summary = _local_sanitize_jsonable(
        _tox_minify_claims_for_expansion(harvested_claims)
    )

    safe_assay_contexts_str = json.dumps(
        _local_sanitize_jsonable(assay_contexts),
        ensure_ascii=True,
        indent=2
    )
    safe_existing_claims_str = json.dumps(
        _local_sanitize_jsonable(existing_claims_summary),
        ensure_ascii=True,
        indent=2
    )
    safe_source_summaries_str = json.dumps(
        _local_sanitize_jsonable(source_summaries),
        ensure_ascii=True,
        indent=2
    )

    intro_text = _local_sanitize_text(f"""
Target material-condition metadata:
- material: {material_row['material']}
- material_name: {material_row['material_name']}
- composition: {material_row['composition']}
- media: {material_row['media']}

Methods-derived assay contexts (soft hints only):
{safe_assay_contexts_str}

Already harvested claims (do NOT duplicate these; use them only to find gaps):
{safe_existing_claims_str}

Candidate source summaries:
{safe_source_summaries_str}

Methods/results text bundle:
{methods_results_bundle if methods_results_bundle.strip() else 'None'}
""")

    content = [{"type": "text", "text": intro_text}]

    for row in prepared_rows.itertuples(index=False):
        page_context = _local_sanitize_text(
            get_page_context(
                full_text=full_text_one,
                center_page=int(row.page),
                window=page_window,
                max_chars=4500
            )
        )

        content.append({
            "type": "text",
            "text": _local_sanitize_text(
                f"Candidate source block\n"
                f"- row_id: {int(row.row_id)}\n"
                f"- page: {int(row.page)}\n"
                f"- source_type: {_tox_norm_str(row.image_property)}\n"
                f"- caption: {_tox_norm_str(row.caption) or 'None'}\n\n"
                f"Nearby page text:\n{page_context}"
            )
        })

        if pd.notna(row.image_path) and os.path.exists(row.image_path):
            content.append({
                "type": "image_url",
                "image_url": {
                    "url": image_to_data_url(row.image_path),
                    "detail": "high"
                }
            })

    system_prompt = """
You are doing a SECOND PASS recall expansion for toxicity datapoints.

Goal:
- find additional datapoints missed in the first harvest
- use Methods-derived assay contexts only as soft recall hints
- recover missing datapoints only when the candidate image/table/caption/nearby text provides support

Critical rules:
1. Do NOT treat Methods as hard constraints.
2. Do NOT create a new numeric assay_results value from Methods alone.
3. A new claim must be supported at least partially by:
   - candidate image/table, or
   - candidate caption, or
   - nearby page text associated with that candidate.
4. Methods may help you notice:
   - likely assay identity
   - likely cell line
   - likely exposure time
   - likely dose grid / dose unit
   - likely medium/context
5. Return ONLY additional claims that are missing from the harvested set.
6. Do NOT re-emit claims that are already represented in harvested_claims.
7. If no additional claim exists, return {"claims": []}.
8. For every returned claim, set claim_id to -1. The caller will renumber.
9. Return strings, not nulls. Use "None" when missing.
10. applies_to_target_material must be one of:
    - Yes
    - No
    - Unclear
11. applies_to_target_media must be one of:
    - Yes
    - No
    - Not relevant
    - Unclear
12. is_direct_datapoint must be:
    - Yes
    - No
13. Set x_title and y_title to "None" unless directly supported by the candidate figure.
14. Do NOT invent or infer axis titles from Methods alone.
"""

    resp = _tox_logged_chat_completion_create(
        client=client,
        model=model,
        temperature=temperature,
        response_format={
            "type": "json_schema",
            "json_schema": TOX_HARVEST_CLAIM_SCHEMA
        },
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": content},
        ],

        event_rows=event_rows,
        stage="methods_expansion_api",
        pdf_key=pdf_key,
        title=title,
        material=material_row["material"],

        api_call_name="expand_tox_claims_with_methods_for_one_material",
        candidate_row_id=-1,
        source_page=-1,
        caption_len=-1,
        n_images=len(prepared_rows),
    )

    result = json.loads(resp.choices[0].message.content)

    next_id = _tox_next_claim_id(harvested_claims)
    new_claims = []

    for c in result.get("claims", []):
        cc = _local_sanitize_jsonable(dict(c))

        cc["claim_id"] = next_id
        next_id += 1

        cc["material"] = _tox_norm_str(cc.get("material")) or _tox_norm_str(material_row["material"])
        cc["material_name"] = _tox_norm_str(cc.get("material_name")) or _tox_norm_str(material_row["material_name"])
        cc["composition"] = _tox_norm_str(cc.get("composition")) or _tox_norm_str(material_row["composition"])
        cc["media"] = _tox_norm_str(cc.get("media")) or _tox_norm_str(material_row["media"])

        try:
            row_id = int(cc.get("candidate_row_id", -1))
        except Exception:
            row_id = -1

        try:
            page_val = int(cc.get("page", -1))
        except Exception:
            page_val = -1

        if row_id in row_meta:
            if _tox_is_missing(cc.get("source_caption")):
                cc["source_caption"] = row_meta[row_id]["caption"]

            if _tox_is_missing(cc.get("source_type")):
                cc["source_type"] = row_meta[row_id]["source_type"]

            if page_val < 1:
                cc["page"] = row_meta[row_id]["page"]
        else:
            cc["candidate_row_id"] = max(row_id, -1)
            if page_val < 1:
                cc["page"] = -1
            if _tox_is_missing(cc.get("source_caption")):
                cc["source_caption"] = "None"
            if _tox_is_missing(cc.get("source_type")):
                cc["source_type"] = "None"

        cc["x_title"] = _tox_norm_str(cc.get("x_title")) or "None"
        cc["y_title"] = _tox_norm_str(cc.get("y_title")) or "None"

        src = _tox_norm_str(cc.get("source_type")).lower()
        if "figure" not in src:
            cc["x_title"] = "None"
            cc["y_title"] = "None"

        cc = _tox_finalize_claim_cell_assay_fields(cc)

        new_claims.append(cc)

    new_claims = _tox_filter_only_new_claims(
        existing_claims=harvested_claims,
        candidate_new_claims=new_claims
    )

    return {"claims": new_claims}

In [ ]:
# Step 8.9: Stage B toxicity claim refinement
def _tox_build_refinement_source_blocks(
    prepared_rows: pd.DataFrame,
    full_text_one: str,
    page_window: int = 1,
    max_chars_per_block: int = 3500,
):
    """
    Create detailed context blocks per candidate row to strengthen source grounding for each claim
    during the refinement stage.
    """
    blocks = []

    if prepared_rows is None or prepared_rows.empty:
        return blocks

    for row in prepared_rows.itertuples(index=False):
        page_context = get_page_context(
            full_text=full_text_one,
            center_page=int(row.page),
            window=page_window,
            max_chars=max_chars_per_block
        )

        blocks.append({
            "candidate_row_id": int(row.row_id),
            "page": int(row.page),
            "source_type": _tox_norm_str(getattr(row, "image_property", "None")) or "None",
            "caption": _tox_norm_str(getattr(row, "caption", "None")) or "None",
            "nearby_page_text": _tox_sanitize_llm_text(page_context),
        })

    return blocks


def refine_tox_claims_with_caption_and_text_for_one_material(
    material_row: pd.Series,
    harvested_claims: list,
    candidate_rows: pd.DataFrame,
    full_text_one: str,
    client,
    model: str,
    temperature: float = 0,
    top_n_candidates: int = 4,
    page_window: int = 1,
    max_snippets: int = 28,
    event_rows: list = None,
    pdf_key: str = "None",
    title: str = "None",
) -> dict:
    if event_rows is None:
        event_rows = []

    def _local_sanitize_text(x, default: str = "None") -> str:
        if "_tox_sanitize_llm_text" in globals():
            return globals()["_tox_sanitize_llm_text"](x, default=default)

        if x is None:
            return default

        s = str(x)
        s = re.sub(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]", " ", s)
        s = "".join(ch for ch in s if unicodedata.category(ch) != "Cs")
        s = s.encode("utf-8", "ignore").decode("utf-8", "ignore")
        s = re.sub(r"\s+", " ", s).strip()
        return s if s else default

    def _local_sanitize_jsonable(obj):
        if "_tox_sanitize_llm_jsonable" in globals():
            return globals()["_tox_sanitize_llm_jsonable"](obj)

        if isinstance(obj, dict):
            return {k: _local_sanitize_jsonable(v) for k, v in obj.items()}
        if isinstance(obj, list):
            return [_local_sanitize_jsonable(v) for v in obj]
        if isinstance(obj, str):
            return _local_sanitize_text(obj)
        return obj

    if not harvested_claims:
        return {"claims": []}

    prepared_rows = _tox_prepare_candidate_rows(
        candidate_rows,
        top_n_candidates=top_n_candidates
    )
    candidate_pages = _tox_collect_candidate_pages(
        prepared_rows,
        page_window=page_window
    )

    methods_results_bundle, bundle_pages = _tox_get_methods_results_text_bundle(
        full_text=full_text_one,
        material_name=material_row["material_name"],
        composition=material_row["composition"],
        media=material_row["media"],
        candidate_pages=candidate_pages,
        max_snippets=max_snippets,
        max_chars=18000
    )
    methods_results_bundle = _local_sanitize_text(methods_results_bundle)

    source_summaries = []
    for row in prepared_rows.itertuples(index=False):
        source_summaries.append({
            "row_id": int(row.row_id),
            "page": int(row.page),
            "source_type": _tox_norm_str(row.image_property),
            "caption": _tox_norm_str(row.caption)
        })
    source_summaries = _local_sanitize_jsonable(source_summaries)

    source_context_blocks = _local_sanitize_jsonable(
        _tox_build_refinement_source_blocks(
            prepared_rows=prepared_rows,
            full_text_one=full_text_one,
            page_window=page_window,
            max_chars_per_block=3500,
        )
    )

    harvested_claims_clean = _local_sanitize_jsonable(harvested_claims)

    system_prompt = """
You are refining already-harvested toxicity datapoint claims.

Goal:
- keep the same datapoints
- fill missing metadata using captions and text
- correct obvious field mistakes
- do NOT invent numeric assay_results

Critical priority for THIS refinement:
- improve cell_assay_raw and cell_assay whenever the caption, nearby page text, or methods/results text provides assay evidence
- do NOT normalize assay names in this step
- preserve source-like assay wording
- if the source says "MTT assay", keep "MTT assay"
- if the source says "LDH release assay", keep "LDH release assay"
- if the source says "propidium iodide staining", keep "propidium iodide staining"

Evidence priority for assay filling:
1. caption and nearby page text from the SAME candidate_row_id as the claim
2. panel-specific wording linked to the claim's panel_label
3. methods/results text bundle
4. shared source context from the same figure/table candidate if the evidence clearly applies to all datapoints in that source

Important instructions:
1. Return one revised claim for each input claim_id.
2. Keep claim_id unchanged.
3. You may fill or correct:
   - cell_assay_raw
   - cell_assay
   - cell_name
   - cell_species
   - cell_origin
   - cell_type
   - exposure_time
   - exposure_time_unit
   - exposure_dose
   - exposure_dose_unit
   - endpoint_type
   - value_origin
   - panel_label
   - series_label
   - applies_to_target_material
   - applies_to_target_media
   - source_type
   - evidence_text
   - x_title
   - y_title
4. Do NOT invent numeric assay_results.
5. You may keep assay_results unchanged if supported.
6. For assay inference, actively look for keywords such as:
   - MTT
   - MTS
   - XTT
   - WST, WST-1, WST-8
   - CCK-8
   - LDH
   - NRU / neutral red uptake
   - SRB
   - alamar blue / alamarBlue
   - resazurin
   - ATP / CellTiter-Glo / CellTiter-Blue / CytoTox-Glo
   - trypan blue
   - calcein AM
   - live/dead
   - Annexin V/PI
   - propidium iodide / PI staining
7. If the caption or nearby source block clearly indicates the assay, do NOT leave cell_assay_raw / cell_assay as None.
8. If multiple datapoints come from the same candidate source and the assay clearly applies to that entire source, you may assign that same assay wording across those claims.
9. Keep cell_assay_raw and cell_assay identical in this step unless there is a strong reason not to.
10. Prefer explicit assay wording from caption/text over generic labels like "viability assay".
11. If still unknown, use "None".
12. Return strings, not nulls.
"""

    user_payload = _local_sanitize_jsonable({
        "target_material": {
            "material": material_row["material"],
            "material_name": material_row["material_name"],
            "composition": material_row["composition"],
            "media": material_row["media"]
        },
        "source_summaries": source_summaries,
        "source_context_blocks": source_context_blocks,
        "methods_results_text_bundle": methods_results_bundle,
        "harvested_claims": harvested_claims_clean
    })

    safe_user_payload_str = json.dumps(
        user_payload,
        ensure_ascii=True,
        indent=2
    )

    safe_system_prompt = _local_sanitize_text(system_prompt)
    safe_user_message = _local_sanitize_text(
        "Refine the harvested toxicity claims.\n\n" + safe_user_payload_str
    )

    resp = _tox_logged_chat_completion_create(
        client=client,
        model=model,
        temperature=temperature,
        response_format={
            "type": "json_schema",
            "json_schema": TOX_REFINEMENT_CLAIM_SCHEMA
        },
        messages=[
            {"role": "system", "content": safe_system_prompt},
            {"role": "user", "content": safe_user_message},
        ],

        event_rows=event_rows,
        stage="caption_text_refinement_api",
        pdf_key=pdf_key,
        title=title,
        material=material_row["material"],

        api_call_name="refine_tox_claims_with_caption_and_text_for_one_material",
        candidate_row_id=-1,
        source_page=-1,
        caption_len=-1,
        n_images=len(prepared_rows),
    )

    revised = json.loads(resp.choices[0].message.content)
    revised_claims = _local_sanitize_jsonable(revised.get("claims", []))

    revised_claims = [_tox_finalize_claim_cell_assay_fields(c) for c in revised_claims]

    merged = _tox_merge_claim_versions(harvested_claims, revised_claims)
    merged = [_tox_finalize_claim_cell_assay_fields(c) for c in merged]
    merged = _tox_propagate_shared_fields(merged)
    merged = _tox_dedup_claims(merged)

    return {"claims": merged}

In [ ]:
# Step 8.10: Orchestrate toxicity claim extraction
def extract_tox_claims_for_one_material(
    material_row: pd.Series,
    candidate_rows: pd.DataFrame,
    full_text_one: str,
    methods_text_one: str,
    client,
    model: str,
    temperature: float = 0,
    top_n_candidates: int = 4,
    max_snippets: int = 28,
    page_window: int = 1,
    event_rows: list = None,
    pdf_key: str = "None",
    title: str = "None",
) -> dict:
    if event_rows is None:
        event_rows = []

    try:
        harvested = extract_tox_claims_from_candidate_sources_for_one_material(
            material_row=material_row,
            candidate_rows=candidate_rows,
            full_text_one=full_text_one,
            client=client,
            model=model,
            temperature=temperature,
            top_n_candidates=top_n_candidates,
            page_window=page_window,
            event_rows=event_rows,
            pdf_key=pdf_key,
            title=title,
        )
    except Exception as e:
        raise RuntimeError(f"stage=harvest_candidate_sources | {e}") from e

    harvested_claims = harvested.get("claims", [])
    if not harvested_claims:
        return {"claims": []}

    try:
        expanded = expand_tox_claims_with_methods_for_one_material(
            material_row=material_row,
            harvested_claims=harvested_claims,
            candidate_rows=candidate_rows,
            full_text_one=full_text_one,
            methods_text_one=methods_text_one,
            client=client,
            model=model,
            temperature=temperature,
            top_n_candidates=top_n_candidates,
            max_snippets=max_snippets,
            page_window=page_window,
            event_rows=event_rows,
            pdf_key=pdf_key,
            title=title,
        )
    except Exception as e:
        raise RuntimeError(f"stage=methods_expansion | {e}") from e

    combined_claims = _tox_dedup_claims(
        harvested_claims + expanded.get("claims", [])
    )

    try:
        refined = refine_tox_claims_with_caption_and_text_for_one_material(
            material_row=material_row,
            harvested_claims=combined_claims,
            candidate_rows=candidate_rows,
            full_text_one=full_text_one,
            client=client,
            model=model,
            temperature=temperature,
            top_n_candidates=top_n_candidates,
            page_window=page_window,
            max_snippets=max_snippets,
            event_rows=event_rows,
            pdf_key=pdf_key,
            title=title,
        )
    except Exception as e:
        raise RuntimeError(f"stage=caption_text_refinement | {e}") from e

    claims = refined.get("claims", [])
    claims = _tox_dedup_claims(claims)

    # Combo-level media validation based on the Methods section
    # Combination unit:
    #   (material_name, media, cell_name)
    # Judgment:
    #   Yes / No / Unclear
    # Application:
    #   Yes -> applies_to_target_media = Yes
    #   No  -> applies_to_target_media = No
    #   Unclear -> keep the existing value
    try:
        combos = build_unique_material_media_cell_combos_from_claims(claims)

        if combos:
            combo_validation_result = validate_material_media_cell_combos_with_methods(
                methods_text_one=methods_text_one,
                combos=combos,
                client=client,
                model=model,
                temperature=temperature,
                event_rows=event_rows,
                pdf_key=pdf_key,
                title=title,
                material=material_row["material"],
            )

            claims = apply_methods_combo_judgment_to_claims(
                claims=claims,
                combo_validation_result=combo_validation_result
            )

            claims = _tox_dedup_claims(claims)

    except Exception as e:
        raise RuntimeError(f"stage=methods_combo_validation | {e}") from e

    cleaned_claims = []
    for idx, c in enumerate(claims, start=1):
        cc = dict(c)
        try:
            cc["claim_id"] = int(cc.get("claim_id", idx))
        except Exception:
            cc["claim_id"] = idx
        cc = _tox_finalize_claim_cell_assay_fields(cc)
        cleaned_claims.append(cc)

    return {"claims": cleaned_claims}

In [ ]:
# Step 8.11: Consolidate toxicity claims
TOX_ASSAY_NORMALIZATION_SCHEMA = {
    "name": "tox_claim_cell_assay_normalization",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "claim_id": {"type": "integer"},
                        "cell_assay": {"type": "string"},
                        "confidence": {"type": "string"},
                        "reason": {"type": "string"},
                    },
                    "required": [
                        "claim_id",
                        "cell_assay",
                        "confidence",
                        "reason",
                    ],
                },
            }
        },
        "required": ["items"],
    },
}


TOX_GROUP_READOUT_RELATION_SCHEMA = {
    "name": "tox_group_readout_relation_classifier",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "group_id": {"type": "string"},
                        "tox_readout_relation": {
                            "type": "string",
                            "enum": ["Direct", "Indirect", "Unclear"]
                        },
                        "confidence": {"type": "string"},
                        "reason": {"type": "string"},
                    },
                    "required": [
                        "group_id",
                        "tox_readout_relation",
                        "confidence",
                        "reason",
                    ],
                },
            }
        },
        "required": ["items"],
    },
}


def _tox_claim_has_basic_record_eligibility(claim: dict) -> bool:
    return _tox_material_ok(claim.get("applies_to_target_material")) and \
           _tox_media_ok(claim.get("applies_to_target_media")) and \
           _tox_has_minimum_payload(claim)


def _tox_direct_record_ok(x: str) -> bool:
    return _tox_norm_str(x).lower() == "direct"


def _tox_assign_claim_ids(claims: list) -> list:
    out = []
    for idx, c in enumerate(claims or [], start=1):
        cc = dict(c)
        cc["claim_id"] = idx
        out.append(cc)
    return out


def _tox_safe_int(x, default: int = -1) -> int:
    try:
        return int(x)
    except Exception:
        return default


def _tox_group_readout_relation_key(claim: dict):
    """
    Grouping key:
    - pdf_key
    - material
    - cell_name
    - cell_assay   <-- initially normalized value
    - page
    - panel_label
    - candidate_row_id
    """
    return (
        _tox_norm_str(claim.get("pdf_key", "None")),
        _tox_norm_str(claim.get("material", "None")),
        _tox_norm_str(claim.get("cell_name", "None")),
        _tox_norm_str(claim.get("cell_assay", "None")),
        _tox_safe_int(claim.get("page", -1), -1),
        _tox_norm_str(claim.get("panel_label", "None")),
        _tox_safe_int(claim.get("candidate_row_id", -1), -1),
    )


def _tox_build_group_relation_payloads(claims: list):
    """
    eligible claim list -> group payloads + reverse map
    """
    groups = {}
    for c in claims:
        key = _tox_group_readout_relation_key(c)
        groups.setdefault(key, []).append(dict(c))

    payloads = []
    group_to_claim_ids = {}
    claim_id_to_group_id = {}

    group_idx = 1
    for _, group_claims in groups.items():
        group_id = f"GREL{group_idx:05d}"
        group_idx += 1

        for c in group_claims:
            cid = int(c["claim_id"])
            claim_id_to_group_id[cid] = group_id

        group_to_claim_ids[group_id] = [int(c["claim_id"]) for c in group_claims]

        group_claims = sorted(
            group_claims,
            key=lambda c: (
                pd.to_numeric(c.get("exposure_time", None), errors="coerce")
                if str(c.get("exposure_time", "")).strip() not in {"", "None"} else float("inf"),
                pd.to_numeric(c.get("exposure_dose", None), errors="coerce")
                if str(c.get("exposure_dose", "")).strip() not in {"", "None"} else float("inf"),
                int(c.get("claim_id", 10**9)),
            )
        )

        first = group_claims[0]

        cell_assay_raw_values = []
        y_title_values = []
        source_caption_values = []

        for c in group_claims:
            raw_assay = _tox_norm_str(c.get("cell_assay_raw"))
            if not _tox_is_missing(raw_assay):
                cell_assay_raw_values.append(raw_assay)

            y_title = _tox_norm_str(c.get("y_title"))
            if not _tox_is_missing(y_title):
                y_title_values.append(y_title)

            src_cap = _tox_norm_str(c.get("source_caption"))
            if not _tox_is_missing(src_cap):
                source_caption_values.append(src_cap)

        # unique preserve order
        def _uniq(xs):
            out = []
            seen = set()
            for x in xs:
                if x not in seen:
                    seen.add(x)
                    out.append(x)
            return out

        cell_assay_raw_values = _uniq(cell_assay_raw_values)
        y_title_values = _uniq(y_title_values)
        source_caption_values = _uniq(source_caption_values)

        datapoints = []
        for c in group_claims:
            datapoints.append({
                "claim_id": int(c["claim_id"]),
                "exposure_time": _tox_norm_str(c.get("exposure_time")) or "None",
                "exposure_time_unit": _tox_norm_str(c.get("exposure_time_unit")) or "None",
                "exposure_dose": _tox_norm_str(c.get("exposure_dose")) or "None",
                "exposure_dose_unit": _tox_norm_str(c.get("exposure_dose_unit")) or "None",
                "assay_results": _tox_norm_str(c.get("assay_results")) or "None",
                "assay_results_num": _tox_clean_numeric_value(c.get("assay_results")),
            })

        payloads.append({
            "group_id": group_id,
            "group_metadata": {
                "pdf_key": _tox_norm_str(first.get("pdf_key")) or "None",
                "material": _tox_norm_str(first.get("material")) or "None",
                "cell_name": _tox_norm_str(first.get("cell_name")) or "None",
                "cell_assay": _tox_norm_str(first.get("cell_assay")) or "None",
                "page": _tox_safe_int(first.get("page", -1), -1),
                "panel_label": _tox_norm_str(first.get("panel_label")) or "None",
                "candidate_row_id": _tox_safe_int(first.get("candidate_row_id", -1), -1),
            },
            "context": {
                "cell_assay_raw_candidates": cell_assay_raw_values if cell_assay_raw_values else ["None"],
                "y_title_candidates": y_title_values if y_title_values else ["None"],
                "source_caption_candidates": source_caption_values if source_caption_values else ["None"],
            },
            "n_points": len(datapoints),
            "datapoints": datapoints,
        })

    return payloads, group_to_claim_ids, claim_id_to_group_id


def classify_tox_claim_groups_readout_relation_with_llm(
    claims: list,
    client,
    model: str,
    temperature: float = 0,
    batch_size: int = 40,
    event_rows: list = None,
    pdf_key: str = "MULTI",
    title: str = "MULTI",
    material: str = "MULTI",
) -> list:
    if event_rows is None:
        event_rows = []

    if not claims:
        return claims

    out = [dict(c) for c in claims]

    payloads, group_to_claim_ids, claim_id_to_group_id = _tox_build_group_relation_payloads(out)

    if not payloads:
        for c in out:
            c["tox_readout_relation"] = "Unclear"
            c["tox_readout_confidence"] = "None"
            c["tox_readout_reason"] = "No eligible group payload"
        return out

    result_map = {}

    for i in range(0, len(payloads), batch_size):
        batch = payloads[i:i+batch_size]

        system_prompt = """
You are classifying toxicity readout groups from nanotoxicology experiments.

You must classify each group into:
- Direct
- Indirect
- Unclear

A Direct group means the values directly represent:
- cell viability
- cytotoxicity
- cell death
- membrane integrity failure
- apoptotic / necrotic fraction
- PI-positive cell fraction
- Annexin/PI death fraction
- LDH-based cell damage / cytotoxicity

An Indirect group means the values mainly represent:
- fluorescence intensity
- relative fluorescence
- signal ratio
- fold change
- mechanistic / pathway signal
- oxidative stress signal
- mitochondrial membrane potential signal
- calcium signal
- uptake / internalization signal

Decision priority:
1. cell_assay
2. y_title
3. source_caption
4. scale / trend of datapoints

Important rules:
- If the group has only 1 datapoint, or trend is weak/unclear, do NOT force trend-based reasoning.
  In that case prioritize:
  1) cell_assay
  2) y_title
  3) source_caption
  4) assay_results numeric scale
- cell_assay family is the strongest clue.
- MTT / MTS / XTT / WST / CCK-8 / NRU / SRB / Alamar blue / Resazurin / ATP / LDH / Trypan blue / Annexin V/PI / PI staining / PI uptake
  are usually Direct viability / cell-death related readouts.
- MitoSOX / JC-1 / Fluo-4 / DCF / DCFH-DA / uptake / internalization / generic fluorescence intensity
  are usually Indirect mechanistic readouts.
- "Metabolic activity" for MTT / MTS / XTT / WST / CCK-8 style assays should usually be treated as Direct.
- A 0~1 or 0~100 scale can be consistent with Direct readouts, but scale alone is NOT sufficient.
- Very large arbitrary values, raw intensity-like values, ratios, fold changes, or strongly signal-like wording support Indirect.
- If evidence conflicts or is too weak, return Unclear.

Examples:
- cell_assay=MTS, y_title=Metabolic activity, datapoints ~ [1.00, 0.92, 0.81]
  -> Direct
- cell_assay=MTT, y_title=Relative viability, datapoints ~ [1.0, 0.95, 0.72]
  -> Direct
- cell_assay=LDH, y_title=Cytotoxicity (%), datapoints ~ [5, 18, 42]
  -> Direct
- cell_assay=PI staining, y_title=PI-positive cells (%)
  -> Direct
- cell_assay=MitoSOX, y_title=Fluorescence intensity
  -> Indirect
- cell_assay=JC-1, y_title=Red/green fluorescence ratio
  -> Indirect
- cell_assay=Fluo-4, y_title=Relative fluorescence
  -> Indirect

Return one item per group_id.
Keep reason short.
Return JSON only.
"""

        user_prompt = (
            "Classify the following toxicity readout groups.\n\n"
            f"{json.dumps(batch, ensure_ascii=False, indent=2)}"
        )

        resp = _tox_logged_chat_completion_create(
            client=client,
            model=model,
            temperature=temperature,
            response_format={
                "type": "json_schema",
                "json_schema": TOX_GROUP_READOUT_RELATION_SCHEMA
            },
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],

            event_rows=event_rows,
            stage="group_readout_relation_api",
            pdf_key=pdf_key,
            title=title,
            material=material,

            api_call_name="classify_tox_claim_groups_readout_relation_with_llm",
            candidate_row_id=-1,
            source_page=-1,
            caption_len=-1,
            n_images=0,
        )

        parsed = json.loads(resp.choices[0].message.content)
        for row in parsed.get("items", []):
            gid = _tox_norm_str(row.get("group_id"))
            if gid:
                result_map[gid] = {
                    "tox_readout_relation": _tox_norm_str(row.get("tox_readout_relation")) or "Unclear",
                    "tox_readout_confidence": _tox_norm_str(row.get("confidence")) or "None",
                    "tox_readout_reason": _tox_norm_str(row.get("reason")) or "None",
                }

    for c in out:
        cid = int(c["claim_id"])
        gid = claim_id_to_group_id.get(cid, None)

        if gid is not None and gid in result_map:
            c["tox_readout_relation"] = result_map[gid]["tox_readout_relation"]
            c["tox_readout_confidence"] = result_map[gid]["tox_readout_confidence"]
            c["tox_readout_reason"] = result_map[gid]["tox_readout_reason"]
        else:
            c["tox_readout_relation"] = "Unclear"
            c["tox_readout_confidence"] = "None"
            c["tox_readout_reason"] = "No group-level decision"

    return out


def normalize_tox_claim_cell_assays_with_llm(
    claims: list,
    client,
    model: str,
    temperature: float = 0,
    batch_size: int = 80,
    event_rows: list = None,
    pdf_key: str = "None",
    title: str = "None",
    material: str = "None",
) -> list:
    if event_rows is None:
        event_rows = []

    if not claims:
        return claims

    out = [dict(c) for c in claims]

    for c in out:
        c = _tox_finalize_claim_cell_assay_fields(c)

    items = []
    for c in out:
        raw = _tox_norm_str(c.get("cell_assay_raw")) or _tox_norm_str(c.get("cell_assay")) or "None"
        items.append({
            "claim_id": int(c["claim_id"]),
            "cell_assay_raw": raw,
        })

    result_map = {}

    for i in range(0, len(items), batch_size):
        batch = items[i:i+batch_size]

        system_prompt = """
You are normalizing assay names for nanotoxicology toxicity claims.

Input:
- claim_id
- cell_assay_raw

Task:
- write a concise normalized assay name into `cell_assay`
- preserve assay identity
- do NOT decide direct/indirect toxicity relation here
- do NOT use "Other" unless the assay identity is truly not recoverable

Examples:
- "MTS assay" -> "MTS"
- "MTT assay" -> "MTT"
- "propidium iodide staining" -> "PI staining"
- "PI uptake assay" -> "PI uptake"
- "Annexin V-FITC/PI staining" -> "Annexin V/PI"
- "MitoSOX staining" -> "MitoSOX"
- "JC-1 staining" -> "JC-1"
- "Fluo-4 staining" -> "Fluo-4"

Rules:
- Return one item per claim_id
- Return strings, not nulls
- Keep reason short
- Return JSON only
"""

        user_prompt = (
            "Normalize the following cell_assay_raw values.\n\n"
            f"{json.dumps(batch, ensure_ascii=False, indent=2)}"
        )

        resp = _tox_logged_chat_completion_create(
            client=client,
            model=model,
            temperature=temperature,
            response_format={
                "type": "json_schema",
                "json_schema": TOX_ASSAY_NORMALIZATION_SCHEMA
            },
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],

            event_rows=event_rows,
            stage="assay_normalization_api",
            pdf_key=pdf_key,
            title=title,
            material=material,

            api_call_name="normalize_tox_claim_cell_assays_with_llm",
            candidate_row_id=-1,
            source_page=-1,
            caption_len=-1,
            n_images=0,
        )

        parsed = json.loads(resp.choices[0].message.content)
        for row in parsed.get("items", []):
            result_map[int(row["claim_id"])] = row

    for c in out:
        cid = int(c["claim_id"])
        raw = _tox_norm_str(c.get("cell_assay_raw")) or _tox_norm_str(c.get("cell_assay")) or "None"

        if cid in result_map:
            c["cell_assay"] = _tox_norm_str(result_map[cid].get("cell_assay")) or raw
            c["cell_assay_normalization_confidence"] = _tox_norm_str(result_map[cid].get("confidence")) or "None"
            c["cell_assay_normalization_reason"] = _tox_norm_str(result_map[cid].get("reason")) or "None"
        else:
            c["cell_assay"] = raw
            c["cell_assay_normalization_confidence"] = "None"
            c["cell_assay_normalization_reason"] = "None"

        c["cell_assay_raw"] = raw

    return out


def prepare_tox_claims_for_record_conversion(
    material_row: pd.Series,
    claims: list,
    client,
    model: str,
    temperature: float = 0,
    assay_norm_batch_size: int = 80,
    readout_batch_size: int = 40,
    pdf_key: str = "None",
    event_rows: list = None,
    title: str = "None",
) -> dict:
    if event_rows is None:
        event_rows = []

    material_label = (
        material_row.get("material", "None")
        if material_row is not None else "None"
    )

    claims = _tox_dedup_claims(claims or [])
    claims = _tox_assign_claim_ids(claims)

    annotated_claims = []
    stage2_candidates = []

    for c in claims:
        cc = _tox_finalize_claim_cell_assay_fields(dict(c))

        cc["pdf_key"] = _tox_norm_str(cc.get("pdf_key")) or _tox_norm_str(pdf_key) or "None"

        mm_pass = "Yes" if (
            _tox_material_ok(cc.get("applies_to_target_material")) and
            _tox_media_ok(cc.get("applies_to_target_media"))
        ) else "No"

        payload_pass = "Yes" if _tox_has_minimum_payload(cc) else "No"

        cc["material_media_pass"] = mm_pass
        cc["minimum_payload_pass"] = payload_pass
        cc["assay_results_num"] = _tox_clean_numeric_value(cc.get("assay_results"))

        cc["cell_assay_normalization_confidence"] = "None"
        cc["cell_assay_normalization_reason"] = "None"
        cc["tox_readout_relation"] = "None"
        cc["tox_readout_confidence"] = "None"
        cc["tox_readout_reason"] = "None"

        annotated_claims.append(cc)

        if mm_pass == "Yes" and payload_pass == "Yes":
            stage2_candidates.append(dict(cc))

    # Initial assay normalization
    if stage2_candidates:
        stage2_candidates = normalize_tox_claim_cell_assays_with_llm(
            claims=stage2_candidates,
            client=client,
            model=model,
            temperature=temperature,
            batch_size=assay_norm_batch_size,
            event_rows=event_rows,
            pdf_key=pdf_key,
            title=title,
            material=material_label,
        )

    # Group-level Direct / Indirect / Unclear judgment
    if stage2_candidates:
        stage2_candidates = classify_tox_claim_groups_readout_relation_with_llm(
            claims=stage2_candidates,
            client=client,
            model=model,
            temperature=temperature,
            batch_size=readout_batch_size,
            event_rows=event_rows,
            pdf_key=pdf_key,
            title=title,
            material=material_label,
        )

    by_id = {int(c["claim_id"]): c for c in stage2_candidates}

    merged_claims = []
    for c in annotated_claims:
        cid = int(c["claim_id"])
        if cid in by_id:
            merged_claims.append(by_id[cid])
        else:
            merged_claims.append(c)

    # Send only Direct groups as final record candidates
    record_candidates = [
        c for c in merged_claims
        if c.get("material_media_pass") == "Yes"
        and c.get("minimum_payload_pass") == "Yes"
        and _tox_direct_record_ok(c.get("tox_readout_relation"))
    ]

    return {
        "claims": merged_claims,
        "record_candidates": record_candidates,
    }


def consolidate_tox_claims_for_one_material(
    material_row: pd.Series,
    claims: list
) -> dict:
    claims = claims or []
    claims = _tox_dedup_claims(claims)

    claims = [
        c for c in claims
        if _tox_material_ok(c.get("applies_to_target_material"))
        and _tox_media_ok(c.get("applies_to_target_media"))
        and _tox_has_minimum_payload(c)
        and _tox_direct_record_ok(c.get("tox_readout_relation", "Direct"))
    ]

    if not claims:
        return {"records": []}

    groups = {}
    for c in claims:
        k = _tox_claim_group_key(c)
        groups.setdefault(k, []).append(c)

    records = []
    for _, group_claims in groups.items():
        best_claim = max(group_claims, key=_tox_claim_score)

        x_title_val = _tox_norm_str(best_claim.get("x_title")) or "None"
        y_title_val = _tox_norm_str(best_claim.get("y_title")) or "None"

        source_kind = _tox_canonical_claim_source_kind(best_claim.get("source_type"))
        if source_kind not in {"figure", "mixed"}:
            x_title_val = "None"
            y_title_val = "None"

        record = {
            "material": material_row["material"],
            "material_name": material_row["material_name"],
            "composition": material_row["composition"],
            "media": material_row["media"],

            "cell_assay": _tox_norm_str(best_claim.get("cell_assay")) or "None",
            "cell_name": _tox_norm_str(best_claim.get("cell_name")) or "None",
            "cell_species": _tox_norm_str(best_claim.get("cell_species")) or "None",

            "cell_origin": "None",
            "cell_type": "None",

            "exposure_time": _tox_clean_time_value(best_claim.get("exposure_time")),
            "exposure_time_unit": _tox_clean_time_unit(best_claim.get("exposure_time_unit")),

            "exposure_dose": _tox_clean_dose_value(best_claim.get("exposure_dose")),
            "exposure_dose_unit": _tox_clean_dose_unit(best_claim.get("exposure_dose_unit")),

            "assay_results": _tox_normalize_assay_result(
                best_claim.get("assay_results"),
                best_claim.get("endpoint_type")
            ),

            "x_title": x_title_val,
            "y_title": y_title_val,

            "endpoint_type": _tox_canonical_endpoint_type(best_claim.get("endpoint_type")),
            "value_origin": _tox_norm_str(best_claim.get("value_origin")) or "None",
            "evidence_text": _tox_norm_str(best_claim.get("evidence_text")) or "None",
            "cell_assay_raw": _tox_norm_str(best_claim.get("cell_assay_raw")) or "None",

            "source_row_id": int(best_claim.get("candidate_row_id", -1)),
            "source_page": int(best_claim.get("page", -1)),
            "source_type": source_kind,
            "source_caption": _tox_norm_str(best_claim.get("source_caption")) or "None",
            "panel_label": _tox_norm_str(best_claim.get("panel_label")) or "None",
        }

        if record["assay_results"] == "None":
            continue

        records.append(record)

    if not records:
        return {"records": []}

    return {"records": records}

def annotate_tox_claim_scores_minimal(df_tox_claims: pd.DataFrame) -> pd.DataFrame:
    """
    Minimal version that attaches existing consolidation scores to the Tox claim export.
    """
    if df_tox_claims is None or df_tox_claims.empty:
        return df_tox_claims

    df = df_tox_claims.copy()

    df["tox_claim_score"] = df.apply(
        lambda r: _tox_claim_score(r.to_dict()),
        axis=1
    )

    return df

In [ ]:
# Step 8.12: Toxicity extraction wrappers
def extract_tox_data_for_one_material(
    material_row: pd.Series,
    candidate_rows: pd.DataFrame,
    full_text_one: str,
    methods_text_one: str,
    client,
    model: str,
    temperature: float = 0,
    top_n_candidates: int = 4,
    max_snippets: int = 28,
    page_window: int = 1,
    return_claims: bool = False,
    event_rows: list = None,
    pdf_key: str = "None",
    title: str = "None",
):
    if event_rows is None:
        event_rows = []

    try:
        raw_claims_result = extract_tox_claims_for_one_material(
            material_row=material_row,
            candidate_rows=candidate_rows,
            full_text_one=full_text_one,
            methods_text_one=methods_text_one,
            client=client,
            model=model,
            temperature=temperature,
            top_n_candidates=top_n_candidates,
            max_snippets=max_snippets,
            page_window=page_window,
            event_rows=event_rows,
            pdf_key=pdf_key,
            title=title,
        )
    except Exception as e:
        raise RuntimeError(f"stage=extract_tox_claims_for_one_material | {e}") from e

    prepared = prepare_tox_claims_for_record_conversion(
        material_row=material_row,
        claims=raw_claims_result.get("claims", []),
        client=client,
        model=model,
        temperature=temperature,
        assay_norm_batch_size=80,
        readout_batch_size=40,
        pdf_key=pdf_key,
        event_rows=event_rows,
        title=title,
    )

    try:
        record_result = consolidate_tox_claims_for_one_material(
            material_row=material_row,
            claims=prepared.get("record_candidates", [])
        )
    except Exception as e:
        raise RuntimeError(f"stage=consolidate_tox_claims_for_one_material | {e}") from e

    if return_claims:
        return record_result, {"claims": prepared.get("claims", [])}

    return record_result


def extract_tox_data_for_materials_in_one_paper(
    df_tox_targets_one: pd.DataFrame,
    df_tox_candidates_one: pd.DataFrame,
    full_text_one: str,
    methods_text_one: str,
    client,
    model: str,
    temperature: float = 0,
    top_n_candidates: int = 4,
    max_snippets: int = 28,
    page_window: int = 1,
    return_claims: bool = False,
    event_rows: list = None,
    title: str = "None",
):
    if event_rows is None:
        event_rows = []

    df_candidates_one = ensure_material_in_tox_candidates(df_tox_candidates_one, df_tox_targets_one)

    record_rows = []
    claim_rows = []

    target_cols = [
        "pdf_key", "material", "material_name", "composition", "media",

        "core_size", "core_size_unit", "core_size_measurement_method", "core_size_source",
        "hydrodynamic_size", "hydrodynamic_size_unit", "hydrodynamic_size_measurement_method", "hydrodynamic_size_source",
        "surface_charge", "surface_charge_unit", "surface_charge_measurement_method", "surface_charge_source",
        "surface_area", "surface_area_unit", "surface_area_measurement_method", "surface_area_source",
    ]
    df_targets_run = df_tox_targets_one[target_cols].drop_duplicates().copy()

    for tgt_row in df_targets_run.itertuples(index=False):
        candidate_rows_one = df_candidates_one[
            df_candidates_one["material"] == tgt_row.material
        ].copy()

        try:
            record_result, claims_result = extract_tox_data_for_one_material(
                material_row=pd.Series({
                    "material": tgt_row.material,
                    "material_name": tgt_row.material_name,
                    "composition": tgt_row.composition,
                    "media": tgt_row.media,

                    "core_size": getattr(tgt_row, "core_size", "None"),
                    "core_size_unit": getattr(tgt_row, "core_size_unit", "None"),
                    "core_size_measurement_method": getattr(tgt_row, "core_size_measurement_method", "None"),
                    "core_size_source": getattr(tgt_row, "core_size_source", "None"),

                    "hydrodynamic_size": getattr(tgt_row, "hydrodynamic_size", "None"),
                    "hydrodynamic_size_unit": getattr(tgt_row, "hydrodynamic_size_unit", "None"),
                    "hydrodynamic_size_measurement_method": getattr(tgt_row, "hydrodynamic_size_measurement_method", "None"),
                    "hydrodynamic_size_source": getattr(tgt_row, "hydrodynamic_size_source", "None"),

                    "surface_charge": getattr(tgt_row, "surface_charge", "None"),
                    "surface_charge_unit": getattr(tgt_row, "surface_charge_unit", "None"),
                    "surface_charge_measurement_method": getattr(tgt_row, "surface_charge_measurement_method", "None"),
                    "surface_charge_source": getattr(tgt_row, "surface_charge_source", "None"),

                    "surface_area": getattr(tgt_row, "surface_area", "None"),
                    "surface_area_unit": getattr(tgt_row, "surface_area_unit", "None"),
                    "surface_area_measurement_method": getattr(tgt_row, "surface_area_measurement_method", "None"),
                    "surface_area_source": getattr(tgt_row, "surface_area_source", "None"),
                }),
                candidate_rows=candidate_rows_one,
                full_text_one=full_text_one,
                methods_text_one=methods_text_one,
                client=client,
                model=model,
                temperature=temperature,
                top_n_candidates=top_n_candidates,
                max_snippets=max_snippets,
                page_window=page_window,
                return_claims=True,
                event_rows=event_rows,
                pdf_key=tgt_row.pdf_key,
                title=title,
            )

            n_records_one = len(record_result.get("records", []))
            n_claims_one = len(claims_result.get("claims", []))

            if n_records_one > 0:
                _append_tox_event(
                    event_rows=event_rows,
                    level="material",
                    status="SUCCESS",
                    stage="material_extraction",
                    pdf_key=tgt_row.pdf_key,
                    title=title,
                    material=tgt_row.material,
                    extracted="Yes",
                    n_records=n_records_one,
                    n_claims=n_claims_one,
                    message="Material-level tox extraction completed"
                )
            else:
                _append_tox_event(
                    event_rows=event_rows,
                    level="material",
                    status="SKIP",
                    stage="material_extraction",
                    pdf_key=tgt_row.pdf_key,
                    title=title,
                    material=tgt_row.material,
                    extracted="No",
                    n_records=0,
                    n_claims=n_claims_one,
                    message="No usable tox records after extraction/consolidation"
                )

            for rec in record_result.get("records", []):
                rec["pdf_key"] = tgt_row.pdf_key

                rec["core_size"] = getattr(tgt_row, "core_size", "None")
                rec["core_size_unit"] = getattr(tgt_row, "core_size_unit", "None")
                rec["core_size_measurement_method"] = getattr(tgt_row, "core_size_measurement_method", "None")
                rec["core_size_source"] = getattr(tgt_row, "core_size_source", "None")

                rec["hydrodynamic_size"] = getattr(tgt_row, "hydrodynamic_size", "None")
                rec["hydrodynamic_size_unit"] = getattr(tgt_row, "hydrodynamic_size_unit", "None")
                rec["hydrodynamic_size_measurement_method"] = getattr(tgt_row, "hydrodynamic_size_measurement_method", "None")
                rec["hydrodynamic_size_source"] = getattr(tgt_row, "hydrodynamic_size_source", "None")

                rec["surface_charge"] = getattr(tgt_row, "surface_charge", "None")
                rec["surface_charge_unit"] = getattr(tgt_row, "surface_charge_unit", "None")
                rec["surface_charge_measurement_method"] = getattr(tgt_row, "surface_charge_measurement_method", "None")
                rec["surface_charge_source"] = getattr(tgt_row, "surface_charge_source", "None")

                rec["surface_area"] = getattr(tgt_row, "surface_area", "None")
                rec["surface_area_unit"] = getattr(tgt_row, "surface_area_unit", "None")
                rec["surface_area_measurement_method"] = getattr(tgt_row, "surface_area_measurement_method", "None")
                rec["surface_area_source"] = getattr(tgt_row, "surface_area_source", "None")

                record_rows.append(rec)

            for claim in claims_result.get("claims", []):
                claim["pdf_key"] = tgt_row.pdf_key
                claim["material"] = tgt_row.material
                claim["material_name"] = tgt_row.material_name
                claim["composition"] = tgt_row.composition
                claim["media"] = tgt_row.media

                claim["core_size"] = getattr(tgt_row, "core_size", "None")
                claim["core_size_unit"] = getattr(tgt_row, "core_size_unit", "None")
                claim["core_size_measurement_method"] = getattr(tgt_row, "core_size_measurement_method", "None")
                claim["core_size_source"] = getattr(tgt_row, "core_size_source", "None")

                claim["hydrodynamic_size"] = getattr(tgt_row, "hydrodynamic_size", "None")
                claim["hydrodynamic_size_unit"] = getattr(tgt_row, "hydrodynamic_size_unit", "None")
                claim["hydrodynamic_size_measurement_method"] = getattr(tgt_row, "hydrodynamic_size_measurement_method", "None")
                claim["hydrodynamic_size_source"] = getattr(tgt_row, "hydrodynamic_size_source", "None")

                claim["surface_charge"] = getattr(tgt_row, "surface_charge", "None")
                claim["surface_charge_unit"] = getattr(tgt_row, "surface_charge_unit", "None")
                claim["surface_charge_measurement_method"] = getattr(tgt_row, "surface_charge_measurement_method", "None")
                claim["surface_charge_source"] = getattr(tgt_row, "surface_charge_source", "None")

                claim["surface_area"] = getattr(tgt_row, "surface_area", "None")
                claim["surface_area_unit"] = getattr(tgt_row, "surface_area_unit", "None")
                claim["surface_area_measurement_method"] = getattr(tgt_row, "surface_area_measurement_method", "None")
                claim["surface_area_source"] = getattr(tgt_row, "surface_area_source", "None")

                claim_rows.append(claim)

        except Exception as e:
            _append_tox_event(
                event_rows=event_rows,
                level="material",
                status="ERROR",
                stage="material_extraction",
                pdf_key=tgt_row.pdf_key,
                title=title,
                material=tgt_row.material,
                extracted="No",
                n_records=0,
                n_claims=0,
                error_type=type(e).__name__,
                message=str(e)
            )
            print(f"[ERROR] {tgt_row.material}: {e}")

    df_tox_one = pd.DataFrame(record_rows)
    df_tox_claims_one = pd.DataFrame(claim_rows)

    df_tox_claims_one = annotate_tox_claim_scores_minimal(df_tox_claims_one)

    if df_tox_one.empty:
        df_tox_one = pd.DataFrame(columns=TOX_RECORD_COLUMNS)
    else:
        for col in TOX_RECORD_COLUMNS:
            if col not in df_tox_one.columns:
                df_tox_one[col] = "None"
        df_tox_one = df_tox_one[TOX_RECORD_COLUMNS]
        df_tox_one = df_tox_one.drop_duplicates().reset_index(drop=True)
        df_tox_one = _tox_sort_records(df_tox_one)

    if df_tox_claims_one.empty:
        df_tox_claims_one = pd.DataFrame(columns=TOX_CLAIM_COLUMNS)
    else:
        for col in TOX_CLAIM_COLUMNS:
            if col not in df_tox_claims_one.columns:
                df_tox_claims_one[col] = "None"
        df_tox_claims_one = df_tox_claims_one[TOX_CLAIM_COLUMNS]
        df_tox_claims_one = df_tox_claims_one.drop_duplicates().reset_index(drop=True)
        df_tox_claims_one = _tox_sort_claims(df_tox_claims_one)

    if return_claims:
        return df_tox_one, df_tox_claims_one

    return df_tox_one

def clean_tox_output(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "exposure_time" in df.columns:
        df["exposure_time_num"] = pd.to_numeric(df["exposure_time"], errors="coerce")

    if "exposure_dose" in df.columns:
        df["exposure_dose_num"] = pd.to_numeric(df["exposure_dose"], errors="coerce")

    if "assay_results" in df.columns:
        df["assay_results_num"] = pd.to_numeric(df["assay_results"], errors="coerce")

    return df

In [ ]:
# Step 8.13: Run collection-level toxicity extraction
def extract_tox_data_for_collection(
    paper_texts: list,
    all_pchem_df: pd.DataFrame,
    client,
    chat_model_name: str,
    layout_model,
    img_dir: str,
    tmp_dir: str,
    output_dir: str,
    selected_collection: str,
    dpi: int = 200,
    temperature: float = 0,
    top_n_candidates: int = 4,
    max_snippets: int = 28,
    page_window: int = 1,
    debug: bool = False
):
    if all_pchem_df is None or all_pchem_df.empty or "pdf_key" not in all_pchem_df.columns:
        print("[SKIP] all_pchem_df is empty or missing 'pdf_key'. Run PChem extraction first.")
        empty_records = pd.DataFrame(columns=TOX_RECORD_COLUMNS)
        empty_claims = pd.DataFrame(columns=TOX_CLAIM_COLUMNS)
        empty_events = pd.DataFrame(columns=TOX_EVENT_LOG_COLUMNS)
        return empty_records, empty_claims, empty_events

    os.makedirs(tmp_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    tox_df_parts = []
    tox_claims_df_parts = []
    tox_event_rows = []

    for paper in paper_texts:
        pdf_key = paper["pdf_key"]
        pdf_path = paper["pdf_path"]
        full_text_one = paper["full_text"]
        methods_text_one = paper.get("methods", "None")
        title = paper["title"]

        print(f"\n===== Processing Tox: {pdf_key} =====")
        print("Title:", title)

        try:
            # 1) Check PChem results for the current paper
            df_pchem_one = all_pchem_df[
                all_pchem_df["pdf_key"] == pdf_key
            ].copy()

            if df_pchem_one.empty:
                _append_tox_event(
                    event_rows=tox_event_rows,
                    level="paper",
                    status="SKIP",
                    stage="precheck_pchem",
                    pdf_key=pdf_key,
                    title=title,
                    extracted="No",
                    message="No PChem rows for this paper"
                )
                print(f"[SKIP] No PChem rows for pdf_key={pdf_key}")
                continue

            # 2) Create Tox targets
            df_tox_targets_one = build_tox_targets_from_pchem_for_one_paper(df_pchem_one)
            if df_tox_targets_one.empty:
                _append_tox_event(
                    event_rows=tox_event_rows,
                    level="paper",
                    status="SKIP",
                    stage="build_tox_targets",
                    pdf_key=pdf_key,
                    title=title,
                    extracted="No",
                    message="No tox targets generated from PChem rows"
                )
                print(f"[SKIP] No tox targets for pdf_key={pdf_key}")
                continue

            # Tiny test stage 3: Extract images and tables
            df_images_one = extract_figures_tables_with_caption_merging(
                pdf_file_path=pdf_path,
                key=pdf_key,
                img_dir=img_dir,
                model=layout_model,
                dpi=dpi,
                debug=debug
            )

            if df_images_one.empty:
                _append_tox_event(
                    event_rows=tox_event_rows,
                    level="paper",
                    status="INFO",
                    stage="extract_figures_tables",
                    pdf_key=pdf_key,
                    title=title,
                    extracted="No",
                    message="No image/table rows found"
                )
                print(f"[INFO] No image/table rows for pdf_key={pdf_key}")
                df_images_one = pd.DataFrame(
                    columns=["image_path", "caption", "image_property", "page"]
                )

            df_images_one = _tox_prepare_images_df(df_images_one)
            df_images_one["pdf_key"] = pdf_key
            df_images_one["title"] = title

            # 4) Select caption-based Tox candidates
            selector_result_one = select_tox_figures_tables_from_captions(
                df_tox_targets_one=df_tox_targets_one,
                df_images_one=df_images_one,
                client=client,
                model=chat_model_name,
                temperature=0,
                event_rows=tox_event_rows,
                pdf_key=pdf_key,
                title=title,
                material="MULTI_TARGETS",
            )

            df_tox_selector_one = tox_selector_result_to_df(selector_result_one)

            df_tox_candidates_one = build_tox_candidates_df(
                df_tox_selector_one=df_tox_selector_one,
                df_images_one=df_images_one
            )

            if df_tox_candidates_one.empty:
                _append_tox_event(
                    event_rows=tox_event_rows,
                    level="paper",
                    status="SKIP",
                    stage="select_tox_candidates",
                    pdf_key=pdf_key,
                    title=title,
                    extracted="No",
                    message="No tox candidates selected from figure/table captions"
                )
                print(f"[SKIP] No tox candidates selected for pdf_key={pdf_key}")
                continue

            # 5) Extract Tox data
            df_tox_one, df_tox_claims_one = extract_tox_data_for_materials_in_one_paper(
                df_tox_targets_one=df_tox_targets_one,
                df_tox_candidates_one=df_tox_candidates_one,
                full_text_one=full_text_one,
                methods_text_one=methods_text_one,
                client=client,
                model=chat_model_name,
                temperature=temperature,
                top_n_candidates=top_n_candidates,
                max_snippets=max_snippets,
                page_window=page_window,
                return_claims=True,
                event_rows=tox_event_rows,
                title=title,
            )

            df_tox_one = clean_tox_output(df_tox_one)

            if not df_tox_one.empty:
                _append_tox_event(
                    event_rows=tox_event_rows,
                    level="paper",
                    status="SUCCESS",
                    stage="paper_extraction",
                    pdf_key=pdf_key,
                    title=title,
                    extracted="Yes",
                    n_records=len(df_tox_one),
                    n_claims=len(df_tox_claims_one),
                    message="Paper-level tox extraction completed"
                )
            else:
                _append_tox_event(
                    event_rows=tox_event_rows,
                    level="paper",
                    status="SKIP",
                    stage="paper_extraction",
                    pdf_key=pdf_key,
                    title=title,
                    extracted="No",
                    n_records=0,
                    n_claims=len(df_tox_claims_one),
                    message="No final tox records extracted for this paper"
                )

            # 6) Accumulate results
            if not df_tox_one.empty:
                if "title" not in df_tox_one.columns:
                    df_tox_one.insert(1, "title", title)
                else:
                    df_tox_one["title"] = title
                tox_df_parts.append(df_tox_one)

            if not df_tox_claims_one.empty:
                if "title" not in df_tox_claims_one.columns:
                    df_tox_claims_one["title"] = title
                else:
                    df_tox_claims_one["title"] = title
                tox_claims_df_parts.append(df_tox_claims_one)

            # 7) Save temporary outputs per paper
            try:
                tmp_now = datetime.now(timezone("Asia/Seoul")).strftime("%Y%m%d_%H%M%S")
                tmp_filename = f"{selected_collection.replace(' ', '_')}_tox_tmp_{pdf_key}_{tmp_now}.xlsx"
                tmp_path = os.path.join(tmp_dir, tmp_filename)

                df_tox_one_save = df_tox_one.copy()
                df_tox_claims_one_save = df_tox_claims_one.copy()

                if df_tox_one_save.empty:
                    df_tox_one_save = pd.DataFrame(columns=TOX_RECORD_COLUMNS)
                else:
                    for col in TOX_RECORD_COLUMNS:
                        if col not in df_tox_one_save.columns:
                            df_tox_one_save[col] = "None"
                    df_tox_one_save = df_tox_one_save[
                        [c for c in TOX_RECORD_COLUMNS if c in df_tox_one_save.columns] +
                        [c for c in df_tox_one_save.columns if c not in TOX_RECORD_COLUMNS]
                    ]

                if df_tox_claims_one_save.empty:
                    df_tox_claims_one_save = pd.DataFrame(columns=TOX_CLAIM_COLUMNS)
                else:
                    for col in TOX_CLAIM_COLUMNS:
                        if col not in df_tox_claims_one_save.columns:
                            df_tox_claims_one_save[col] = "None"
                    df_tox_claims_one_save = df_tox_claims_one_save[
                        [c for c in TOX_CLAIM_COLUMNS if c in df_tox_claims_one_save.columns] +
                        [c for c in df_tox_claims_one_save.columns if c not in TOX_CLAIM_COLUMNS]
                    ]

                with pd.ExcelWriter(tmp_path, engine="openpyxl") as writer:
                    df_tox_one_save.to_excel(writer, sheet_name="tox_records", index=False)
                    df_tox_claims_one_save.to_excel(writer, sheet_name="tox_claims", index=False)

                print(f"[TMP SAVED] {tmp_path}")

            except Exception as tmp_e:
                _append_tox_event(
                    event_rows=tox_event_rows,
                    level="paper",
                    status="WARN",
                    stage="tmp_save",
                    pdf_key=pdf_key,
                    title=title,
                    extracted="No",
                    error_type=type(tmp_e).__name__,
                    message=str(tmp_e)
                )
                print(f"[WARN] tmp save failed for pdf_key={pdf_key}: {tmp_e}")

        except Exception as e:
            _append_tox_event(
                event_rows=tox_event_rows,
                level="paper",
                status="ERROR",
                stage="extract_tox_data_for_collection",
                pdf_key=pdf_key,
                title=title,
                extracted="No",
                error_type=type(e).__name__,
                message=str(e)
            )
            print(f"[SKIP] Tox extraction failed for pdf_key={pdf_key}: {e}")
            continue

    # Merge all results
    all_tox_df = (
        pd.concat(tox_df_parts, ignore_index=True)
        if tox_df_parts else pd.DataFrame(columns=TOX_RECORD_COLUMNS)
    )

    all_tox_claims_df = (
        pd.concat(tox_claims_df_parts, ignore_index=True)
        if tox_claims_df_parts else pd.DataFrame(columns=TOX_CLAIM_COLUMNS)
    )

    if not all_tox_df.empty:
        for col in TOX_RECORD_COLUMNS:
            if col not in all_tox_df.columns:
                all_tox_df[col] = "None"
        all_tox_df = all_tox_df[
            [c for c in TOX_RECORD_COLUMNS if c in all_tox_df.columns] +
            [c for c in all_tox_df.columns if c not in TOX_RECORD_COLUMNS]
        ]
        all_tox_df = _tox_sort_records(all_tox_df)

    if not all_tox_claims_df.empty:
        for col in TOX_CLAIM_COLUMNS:
            if col not in all_tox_claims_df.columns:
                all_tox_claims_df[col] = "None"
        all_tox_claims_df = all_tox_claims_df[
            [c for c in TOX_CLAIM_COLUMNS if c in all_tox_claims_df.columns] +
            [c for c in all_tox_claims_df.columns if c not in TOX_CLAIM_COLUMNS]
        ]
        all_tox_claims_df = all_tox_claims_df.drop_duplicates().reset_index(drop=True)

    # Create and save the event-log DataFrame
    tox_event_log_df = _build_tox_event_log_df(tox_event_rows)

    now = datetime.now(timezone("Asia/Seoul")).strftime("%Y%m%d_%H%M%S")
    event_log_path = os.path.join(
        output_dir,
        f"{selected_collection.replace(' ', '_')}_tox_event_log_{now}.xlsx"
    )

    with pd.ExcelWriter(event_log_path, engine="openpyxl") as writer:
        tox_event_log_df.to_excel(writer, sheet_name="tox_event_log", index=False)

    print(f"[EVENT LOG SAVED] {event_log_path}")

    return all_tox_df, all_tox_claims_df, tox_event_log_df

In [ ]:
# Step 8.14: Final toxicity postprocessing

TOX_POSTPROCESS_ASSAY_ALLOWED = [
    "MTT",
    "MTS",
    "XTT",
    "WST",
    "CCK-8",
    "LDH",
    "NRU",
    "SRB",
    "Alamar blue",
    "Resazurin",
    "Trypan blue",
    "Calcein AM",
    "Live/Dead viability assay",
    "Annexin V/PI",
    "PI staining",
    "CellTiter-Glo",
    "CellTiter-Blue",
    "CytoTox-Glo",
    "ATP",
    "Tox tracker",
    "Luminometric assay",
    "PrestoBlue",
    "Nitroblue Tetrazolium Assay",
    "PicoGreen",
    "TOTO-3",
    "Resazurin assay",
    "CyQuant Assay",
    "CFE",
    "RCC",
    "BrdU",
    "AK",
    "Fluorescein diacetate (FDA) uptake",
    "COMET",
    "Other",
    "None",
]

CELL_ORIGIN_ALLOWED = [
    "Lung",
    "Nose",
    "Blood",
    "Blood vessel",
    "Skin",
    "Bone",
    "Bone marrow",
    "Muscle",
    "Liver",
    "Colon",
    "Stomach",
    "Small intestine",
    "Esophagus",
    "Kidney",
    "Bladder",
    "Brain",
    "Spinal cord",
    "Eye",
    "Heart",
    "Spleen",
    "Adipose tissue",
    "Ovary",
    "Cervix",
    "Uterus",
    "Breast",
    "Testis",
    "Prostate",
    "Placenta",
    "Thyroid",
    "Adrenal gland",
    "Pancreas",
    "Lymph node",
    "Embryo",
    "Mesothelium",
    "Other",
    "None",
]

CELL_TYPE_ALLOWED = ["Normal", "Cancer", "None"]

TOX_CELL_ASSAY_POSTPROCESS_SCHEMA = {
    "name": "tox_cell_assay_postprocess_classifier",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "cell_name": {"type": "string"},
                        "cell_assay": {"type": "string"},
                        "keep_row": {"type": "string"},
                        "normalized_cell_assay": {"type": "string"},
                        "cell_origin": {"type": "string"},
                        "cell_type": {"type": "string"},
                        "confidence": {"type": "string"},
                        "reason": {"type": "string"},
                    },
                    "required": [
                        "cell_name",
                        "cell_assay",
                        "keep_row",
                        "normalized_cell_assay",
                        "cell_origin",
                        "cell_type",
                        "confidence",
                        "reason",
                    ],
                },
            }
        },
        "required": ["items"],
    },
}

TOX_CELL_NAME_POSTPROCESS_SCHEMA = {
    "name": "tox_cell_name_postprocess_normalizer",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "cell_name": {"type": "string"},
                        "normalized_cell_name": {"type": "string"},
                        "changed": {"type": "string"},
                        "confidence": {"type": "string"},
                        "reason": {"type": "string"},
                    },
                    "required": [
                        "cell_name",
                        "normalized_cell_name",
                        "changed",
                        "confidence",
                        "reason",
                    ],
                },
            }
        },
        "required": ["items"],
    },
}


TOX_XTITLE_EXPOSURE_FILTER_SCHEMA = {
    "name": "tox_xtitle_exposure_axis_filter",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "item_id": {"type": "integer"},
                        "keep_row": {
                            "type": "string",
                            "enum": ["Yes", "No"]
                        },
                        "reason": {"type": "string"},
                    },
                    "required": ["item_id", "keep_row", "reason"],
                },
            }
        },
        "required": ["items"],
    },
}


TOX_XTITLE_BYPASS_ASSAYS = {"MTT", "MTS", "XTT", "WST", "CCK-8"}


def _tox_normalize_cell_assay_for_xtitle_bypass(x: str) -> str:
    s = _tox_norm_str(x)
    if s == "":
        return "None"

    sl = s.lower()
    sl = sl.replace("–", "-").replace("—", "-").replace("−", "-")
    sl = re.sub(r"\s+", " ", sl).strip()

    if sl in {"mtt", "mtt assay"} or "mtt" in sl:
        return "MTT"
    if sl in {"mts", "mts assay"} or re.search(r"\bmts\b", sl):
        return "MTS"
    if sl in {"xtt", "xtt assay"} or re.search(r"\bxtt\b", sl):
        return "XTT"
    if sl in {"wst", "wst assay", "wst-1", "wst 1", "wst-8", "wst 8"} or "wst" in sl:
        return "WST"
    if sl in {"cck-8", "cck8", "cck-8 assay", "cck8 assay"} or "cck-8" in sl or "cck8" in sl:
        return "CCK-8"

    return s


def _tox_xtitle_key(x: str) -> str:
    s = _tox_norm_str(x)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s


def _tox_normalize_group_key_value(x):
    s = _tox_norm_str(x)
    if _tox_is_missing(s):
        return "None"

    num = _tox_clean_numeric_value(s)
    if num != "None":
        try:
            return _format_float_str(float(num))
        except Exception:
            pass

    return s


def _tox_result_filter_key_from_row(row) -> tuple:
    return (
        _tox_norm_str(row.get("pdf_key", "None")) or "None",
        _tox_norm_str(row.get("title", "None")) or "None",
        _tox_norm_str(row.get("material", "None")) or "None",
        _tox_normalize_group_key_value(row.get("core_size", "None")),
        _tox_norm_str(row.get("cell_assay", "None")) or "None",
        _tox_norm_str(row.get("cell_name", "None")) or "None",
        _tox_clean_time_value(row.get("exposure_time", "None")),
        _tox_clean_dose_value(row.get("exposure_dose", "None")),
    )


def _tox_abs_assay_result_ge_threshold(value, threshold: float = 150.0) -> bool:
    val = _tox_clean_numeric_value(value)
    if val == "None":
        return False

    try:
        num = float(val)
    except Exception:
        return False

    return abs(num) >= float(threshold)


def _tox_prepare_rows_for_shared_result_filter(df: pd.DataFrame, source_kind: str) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame(columns=[
            "pdf_key",
            "title",
            "material",
            "core_size",
            "cell_assay",
            "cell_name",
            "exposure_time",
            "exposure_dose",
            "assay_results",
            "x_title",
            "y_title",
            "source_caption",
            "source_type",
            "__filter_key__",
            "__xtitle_key__",
            "__source_kind__",
        ])

    out = df.copy()

    needed_cols = [
        "pdf_key",
        "title",
        "material",
        "core_size",
        "cell_assay",
        "cell_name",
        "exposure_time",
        "exposure_dose",
        "assay_results",
        "x_title",
        "y_title",
        "source_caption",
        "source_type",
    ]
    for col in needed_cols:
        if col not in out.columns:
            out[col] = "None"

    out["__filter_key__"] = out.apply(_tox_result_filter_key_from_row, axis=1)
    out["__xtitle_key__"] = out["x_title"].map(_tox_xtitle_key)
    out["__source_kind__"] = source_kind

    return out


def classify_tox_xtitles_with_llm(
    xtitle_rows: list,
    client,
    model: str,
    temperature: float = 0,
    batch_size: int = 80,
    event_rows: list = None,
    pdf_key: str = "MULTI",
    title: str = "MULTI",
    material: str = "MULTI",
) -> dict:
    """
    Judge whether x_title represents material exposure concentration/dose or exposure time.
    return:
      {
        item_id: {
          "keep_row": "Yes"/"No",
          "reason": "..."
        }
      }
    """
    if event_rows is None:
        event_rows = []

    if not xtitle_rows:
        return {}

    result_map = {}

    for i in range(0, len(xtitle_rows), batch_size):
        batch = xtitle_rows[i:i+batch_size]

        system_prompt = """
You are filtering toxicity datapoints using the x-axis title only.

Keep a row if x_title clearly represents ANY of the following:
1) material exposure concentration / material exposure dose
2) exposure time / treatment time / incubation time
3) a material-identity axis that directly lists the exposed material / nanomaterial name
4) an exposure axis containing concentration or dose units, even if the wording is short or incomplete

Important:
- Be permissive for exposure-related axes.
- Do NOT mark No when x_title contains concentration/dose units.
- Do NOT mark No when x_title contains explicit material names or nanomaterial identity labels.

Return:
- keep_row = "Yes" if x_title is compatible with an exposure concentration/dose axis, exposure time axis, or exposed material identity axis
- keep_row = "No" otherwise

Keep examples:
- Concentration (µg/mL)
- Dose (µg/mL)
- Exposure dose
- Treatment concentration
- Nanoparticle concentration
- Time (h)
- Exposure time (h)
- Treatment time
- Incubation time
- TiO2 (µg/mL)
- ZnO concentration
- AgNP dose
- SiO2 NPs
- Nanoparticles (mg/L)
- TiO2
- ZnO
- AgNP
- SiO2
- Particle concentration
- NP concentration

Concentration / dose unit signals that should generally be kept:
- mg/L
- g/L
- µg/mL
- ug/mL
- ng/mL
- mg/mL
- µg/L
- ug/L
- ppm
- ppb
- μM
- µM
- mM
- nM
- mol/L
- mmol/L

Material-identity signals that should generally be kept:
- explicit nanomaterial names such as TiO2, ZnO, SiO2, CeO2, Ag, Au
- labels such as AgNP, AuNP, NP, NPs, nanoparticle, nanoparticles
- any x_title that is directly naming the exposed material groups

Drop examples:
- Cell viability (%)
- Fold change
- Fluorescence intensity
- Particle size
- Zeta potential
- Surface area
- Wavelength
- Uptake
- ROS
- Absorbance
- Relative fluorescence
- Fold induction
- Gene expression
- Protein expression
- Group/category labels unrelated to exposure material or exposure level

Rules:
- If x_title includes concentration/dose units, prefer Yes.
- If x_title includes a nanomaterial/material name, prefer Yes.
- If x_title is ambiguous but plausibly exposure-related, prefer Yes rather than No.
- Return No only when the axis is clearly not an exposure/material axis.
- Return JSON only.
"""

        user_prompt = (
            "Classify the following x_title items.\n\n"
            f"{json.dumps(batch, ensure_ascii=False, indent=2)}"
        )

        resp = _tox_logged_chat_completion_create(
            client=client,
            model=model,
            temperature=temperature,
            response_format={
                "type": "json_schema",
                "json_schema": TOX_XTITLE_EXPOSURE_FILTER_SCHEMA
            },
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],

            event_rows=event_rows,
            stage="tox_xtitle_filter_api",
            pdf_key=pdf_key,
            title=title,
            material=material,

            api_call_name="classify_tox_xtitles_with_llm",
            candidate_row_id=-1,
            source_page=-1,
            caption_len=-1,
            n_images=0,
        )

        parsed = json.loads(resp.choices[0].message.content)
        for row in parsed.get("items", []):
            item_id = int(row["item_id"])
            result_map[item_id] = {
                "keep_row": "Yes" if _tox_norm_str(row.get("keep_row")).lower() == "yes" else "No",
                "reason": _tox_norm_str(row.get("reason")) or "None",
            }

    return result_map


def _tox_filter_by_xtitle_and_ytitle(
    df: pd.DataFrame,
    xtitle_keep_map: dict
) -> pd.DataFrame:
    if df is None or df.empty:
        return df

    out = df.copy()

    def _keep_by_xtitle(row):
        x_title = _tox_norm_str(row.get("x_title", "None"))

        # NEW RULE:
        # Always exclude rows when x_title contains %
        if "%" in x_title:
            return False

        # Bypass 1: ignore the x_title filter for specific assays
        cell_assay_norm = _tox_normalize_cell_assay_for_xtitle_bypass(
            row.get("cell_assay", "None")
        )
        if cell_assay_norm in TOX_XTITLE_BYPASS_ASSAYS:
            return True

        # Bypass 2: pass rows with an empty x_title
        if _tox_is_missing(x_title):
            return True

        # Use LLM judgment for all remaining rows
        key = _tox_xtitle_key(x_title)
        return xtitle_keep_map.get(key, "No") == "Yes"

    x_keep_mask = out.apply(_keep_by_xtitle, axis=1)

    y_keep_mask = ~out["y_title"].fillna("").astype(str).str.contains(
        "fold", case=False, na=False
    )

    out = out[x_keep_mask & y_keep_mask].copy().reset_index(drop=True)
    return out


def _tox_collect_extreme_result_drop_keys(
    df_tox_records: pd.DataFrame,
    df_tox_claims: pd.DataFrame,
    threshold: float = 150.0
) -> set:
    drop_keys = set()

    for df in [df_tox_records, df_tox_claims]:
        if df is None or df.empty:
            continue

        work = _tox_prepare_rows_for_shared_result_filter(df, source_kind="tmp")
        mask = work["assay_results"].map(lambda x: _tox_abs_assay_result_ge_threshold(x, threshold=threshold))

        if mask.any():
            drop_keys.update(work.loc[mask, "__filter_key__"].tolist())

    return drop_keys


def _tox_drop_rows_by_filter_keys(df: pd.DataFrame, drop_keys: set) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    if not drop_keys:
        return df

    work = _tox_prepare_rows_for_shared_result_filter(df, source_kind="tmp")
    work = work[~work["__filter_key__"].isin(drop_keys)].copy()

    work = work.drop(
        columns=["__filter_key__", "__xtitle_key__", "__source_kind__"],
        errors="ignore"
    ).reset_index(drop=True)

    return work


def run_tox_shared_final_filters(
    df_tox_records: pd.DataFrame,
    df_tox_claims: pd.DataFrame,
    client,
    model: str,
    temperature: float = 0,
    xtitle_batch_size: int = 80,
    extreme_threshold: float = 150.0,
    event_rows: list = None,
    pdf_key: str = "MULTI",
    title: str = "MULTI",
    material: str = "MULTI",
):
    """
    Final shared filters:
    1) LLM: remove rows when x_title is not a material exposure concentration/dose or exposure-time axis.
    2) Rule: remove rows when x_title contains %.
    3) Rule: remove rows when y_title contains 'fold'.
    4) Rule: if any row has |assay_results| >= threshold,
       (pdf_key, title, material, core_size, cell_assay, cell_name, exposure_time, exposure_dose)
       remove the entire combination from both tox_records and tox_claims.
    """
    if event_rows is None:
        event_rows = []

    rec = df_tox_records.copy() if df_tox_records is not None else pd.DataFrame()
    clm = df_tox_claims.copy() if df_tox_claims is not None else pd.DataFrame()

    rec_prep = _tox_prepare_rows_for_shared_result_filter(rec, source_kind="record")
    clm_prep = _tox_prepare_rows_for_shared_result_filter(clm, source_kind="claim")

    xtitle_union = (
        pd.concat([rec_prep, clm_prep], ignore_index=True)
        if (not rec_prep.empty or not clm_prep.empty)
        else pd.DataFrame()
    )

    xtitle_rows = []
    xtitle_keep_map = {}

    if not xtitle_union.empty:
        xtitle_union = xtitle_union[
            ~xtitle_union["x_title"].map(_tox_is_missing)
        ].copy()

        if not xtitle_union.empty:
            xtitle_union = (
                xtitle_union
                .sort_values(["x_title", "y_title", "source_caption"], kind="stable")
                .drop_duplicates(subset=["__xtitle_key__"])
                .reset_index(drop=True)
            )

            for idx, row in xtitle_union.iterrows():
                xtitle_rows.append({
                    "item_id": idx + 1,
                    "x_title": _tox_norm_str(row.get("x_title", "None")) or "None",
                    "sample_y_title": _tox_norm_str(row.get("y_title", "None")) or "None",
                    "sample_source_caption": _tox_norm_str(row.get("source_caption", "None")) or "None",
                    "sample_source_type": _tox_norm_str(row.get("source_type", "None")) or "None",
                    "__xtitle_key__": row["__xtitle_key__"],
                })

            raw_map = classify_tox_xtitles_with_llm(
                xtitle_rows=xtitle_rows,
                client=client,
                model=model,
                temperature=temperature,
                batch_size=xtitle_batch_size,
                event_rows=event_rows,
                pdf_key=pdf_key,
                title=title,
                material=material,
            )

            for item in xtitle_rows:
                keep_val = raw_map.get(item["item_id"], {}).get("keep_row", "No")
                xtitle_keep_map[item["__xtitle_key__"]] = keep_val

    rec = _tox_filter_by_xtitle_and_ytitle(rec, xtitle_keep_map=xtitle_keep_map)
    clm = _tox_filter_by_xtitle_and_ytitle(clm, xtitle_keep_map=xtitle_keep_map)

    drop_keys = _tox_collect_extreme_result_drop_keys(
        df_tox_records=rec,
        df_tox_claims=clm,
        threshold=extreme_threshold
    )

    rec = _tox_drop_rows_by_filter_keys(rec, drop_keys=drop_keys)
    clm = _tox_drop_rows_by_filter_keys(clm, drop_keys=drop_keys)

    if rec is not None and not rec.empty:
        rec = rec.drop_duplicates().reset_index(drop=True)
    if clm is not None and not clm.empty:
        clm = clm.drop_duplicates().reset_index(drop=True)

    return rec, clm


def _cell_name_key(x: str) -> str:
    s = _tox_norm_str(x)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s


def _cell_assay_key(x: str) -> str:
    s = _tox_norm_str(x)
    s = s.replace("–", "-").replace("—", "-").replace("−", "-")
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s


def _normalize_cell_origin_label(x: str) -> str:
    s = _tox_norm_str(x)
    if s == "":
        return "None"

    sl = s.lower()

    direct_map = {v.lower(): v for v in CELL_ORIGIN_ALLOWED}
    if sl in direct_map:
        return direct_map[sl]

    synonym_map = {
        "hematopoietic": "Blood",
        "monocyte": "Blood",
        "macrophage": "Blood",
        "lymphocyte": "Blood",
        "leukocyte": "Blood",

        "alveolar": "Lung",
        "pulmonary": "Lung",
        "bronchial": "Lung",
        "lung": "Lung",

        "olfactory": "Nose",
        "nasal": "Nose",
        "nose": "Nose",

        "hepatic": "Liver",
        "hepatocyte": "Liver",

        "renal": "Kidney",

        "neuronal": "Brain",
        "glial": "Brain",
        "astrocyte": "Brain",
        "brain": "Brain",

        "spinal": "Spinal cord",

        "dermal": "Skin",
        "keratinocyte": "Skin",
        "epidermal": "Skin",

        "intestinal": "Small intestine",
        "enterocyte": "Small intestine",

        "colonic": "Colon",
        "gastric": "Stomach",
        "esophageal": "Esophagus",

        "cervical": "Cervix",
        "uterine": "Uterus",
        "endometrial": "Uterus",

        "cardiac": "Heart",
        "cardiomyocyte": "Heart",

        "vascular": "Blood vessel",
        "endothelial": "Blood vessel",
        "microvascular": "Blood vessel",

        "osteoblast": "Bone",
        "osteocyte": "Bone",

        "marrow": "Bone marrow",

        "myoblast": "Muscle",
        "myocyte": "Muscle",

        "retinal": "Eye",
        "ocular": "Eye",

        "adipocyte": "Adipose tissue",
        "adipose": "Adipose tissue",

        "prostate": "Prostate",
        "placental": "Placenta",
        "thyroid": "Thyroid",
        "adrenal": "Adrenal gland",
        "pancreatic": "Pancreas",
        "splenic": "Spleen",
        "lymphatic": "Lymph node",
        "lymph node": "Lymph node",

        "mesothelial": "Mesothelium",
        "mesothelium": "Mesothelium",

        "embryonic": "Embryo",
        "embryo": "Embryo",
    }

    for k, v in synonym_map.items():
        if sl == k or k in sl:
            return v

    if sl in {"unknown", "unclear", "ambiguous"}:
        return "None"

    return "Other"


def _normalize_cell_type_label(x: str) -> str:
    s = _tox_norm_str(x).lower()
    if s in {"normal", "non-cancer", "noncancer", "non tumor", "non-tumor"}:
        return "Normal"
    if s in {"cancer", "tumor", "tumour", "carcinoma"}:
        return "Cancer"
    return "None"


def _normalize_keep_row_label(x: str) -> str:
    s = _tox_norm_str(x).lower()
    return "Yes" if s in {"yes", "y", "keep", "true", "1"} else "No"


def _normalize_cell_assay_label(x: str) -> str:
    s = _tox_norm_str(x)
    if s == "":
        return "None"

    sl = s.lower()
    sl = sl.replace("–", "-").replace("—", "-").replace("−", "-")
    sl = sl.replace("μ", "µ")
    sl = re.sub(r"\s+", " ", sl).strip()

    direct_map = {
        "mtt": "MTT",
        "mts": "MTS",
        "xtt": "XTT",

        "wst": "WST",
        "wst-1": "WST",
        "wst 1": "WST",
        "wst-8": "WST",
        "wst 8": "WST",

        "cck-8": "CCK-8",
        "cck8": "CCK-8",

        "ldh": "LDH",
        "lactate dehydrogenase": "LDH",

        "nru": "NRU",
        "neutral red": "NRU",
        "neutral red uptake": "NRU",

        "srb": "SRB",

        "alamar blue": "Alamar blue",
        "alamarblue": "Alamar blue",

        "resazurin": "Resazurin",
        "resazurin assay": "Resazurin assay",

        "trypan blue": "Trypan blue",

        "calcein am": "Calcein AM",
        "calcein-am": "Calcein AM",

        "live/dead": "Live/Dead viability assay",
        "live-dead": "Live/Dead viability assay",
        "live/dead viability assay": "Live/Dead viability assay",

        "annexin v/pi": "Annexin V/PI",
        "annexin-v/pi": "Annexin V/PI",
        "annexin v-fitc/pi": "Annexin V/PI",

        "pi staining": "PI staining",
        "propidium iodide staining": "PI staining",
        "propidium iodide": "PI staining",

        "celltiter-glo": "CellTiter-Glo",
        "celltiter glo": "CellTiter-Glo",

        "celltiter-blue": "CellTiter-Blue",
        "celltiter blue": "CellTiter-Blue",

        "cytotox-glo": "CytoTox-Glo",
        "cytotox glo": "CytoTox-Glo",

        "atp": "ATP",
        "atp assay": "ATP",
        "atp viability assay": "ATP",

        "tox tracker": "Tox tracker",
        "toxtracker": "Tox tracker",

        "luminometric assay": "Luminometric assay",
        "luminescence assay": "Luminometric assay",
        "luminescent assay": "Luminometric assay",

        "prestoblue": "PrestoBlue",
        "presto blue": "PrestoBlue",

        "nitroblue tetrazolium assay": "Nitroblue Tetrazolium Assay",
        "nitro blue tetrazolium assay": "Nitroblue Tetrazolium Assay",
        "nbt assay": "Nitroblue Tetrazolium Assay",

        "picogreen": "PicoGreen",
        "pico green": "PicoGreen",

        "toto-3": "TOTO-3",
        "toto 3": "TOTO-3",

        "cyquant assay": "CyQuant Assay",
        "cyquant": "CyQuant Assay",
        "cy-quant assay": "CyQuant Assay",

        "cfe": "CFE",
        "colony forming efficiency": "CFE",
        "colony-forming efficiency": "CFE",

        "rcc": "RCC",
        "relative cloning capacity": "RCC",

        "brdu": "BrdU",
        "brdu assay": "BrdU",

        "ak": "AK",

        "fluorescein diacetate (fda) uptake": "Fluorescein diacetate (FDA) uptake",
        "fda uptake": "Fluorescein diacetate (FDA) uptake",
        "fluorescein diacetate uptake": "Fluorescein diacetate (FDA) uptake",

        "comet": "COMET",
        "comet assay": "COMET",

        "other": "Other",
        "none": "None",
    }

    if sl in direct_map:
        return direct_map[sl]

    contains_rules = [
        (["annexin v/pi", "annexin-v/pi", "annexin v-fitc/pi"], "Annexin V/PI"),

        # PI / propidium iodide family
        (
            [
                "propidium iodide",
                "pi staining",
                "pi-positive",
                "pi positive",
                "flow cytometry cell death assay with propidium iodide staining",
                "flow cytometric propidium iodide staining",
                "propidium iodide uptake assay",
            ],
            "PI staining"
        ),

        (["live/dead", "live-dead", "dead/live"], "Live/Dead viability assay"),

        (["cck-8", "cck8"], "CCK-8"),
        (["wst-1", "wst 1", "wst-8", "wst 8", "water-soluble tetrazolium"], "WST"),
        (["mtt"], "MTT"),
        (["mts"], "MTS"),
        (["xtt"], "XTT"),
        (["ldh", "lactate dehydrogenase"], "LDH"),
        (["neutral red uptake", "neutral red"], "NRU"),
        (["srb"], "SRB"),
        (["alamarblue", "alamar blue"], "Alamar blue"),
        (["resazurin assay"], "Resazurin assay"),
        (["resazurin"], "Resazurin"),
        (["trypan blue"], "Trypan blue"),
        (["calcein am", "calcein-am"], "Calcein AM"),
        (["celltiter-glo", "celltiter glo"], "CellTiter-Glo"),
        (["celltiter-blue", "celltiter blue"], "CellTiter-Blue"),
        (["cytotox-glo", "cytotox glo"], "CytoTox-Glo"),
        (["prestoblue", "presto blue"], "PrestoBlue"),
        (["atp"], "ATP"),
        (["luminometric", "luminescence", "luminescent"], "Luminometric assay"),
        (["tox tracker", "toxtracker"], "Tox tracker"),
        (["nitroblue tetrazolium", "nitro blue tetrazolium", "nbt assay"], "Nitroblue Tetrazolium Assay"),
        (["picogreen", "pico green"], "PicoGreen"),
        (["toto-3", "toto 3"], "TOTO-3"),
        (["cyquant", "cy-quant"], "CyQuant Assay"),
        (["colony forming efficiency", "colony-forming efficiency"], "CFE"),
        (["relative cloning capacity"], "RCC"),
        (["brdu"], "BrdU"),
        (["fda uptake", "fluorescein diacetate"], "Fluorescein diacetate (FDA) uptake"),
        (["comet"], "COMET"),
    ]

    for keys, canon in contains_rules:
        if any(k in sl for k in keys):
            return canon

    generic_keep_signals = [
        "cell viability assay",
        "viability assay",
        "cytotoxicity assay",
        "cell death assay",
        "apoptosis assay",
        "necrosis assay",
        "flow cytometry viability",
        "flow cytometry cell death",
        "membrane integrity assay",
    ]
    if any(k in sl for k in generic_keep_signals):
        return "Other"

    return "None"


def _normalize_cell_name_final_label(x: str) -> str:
    s = _tox_norm_str(x)
    if s == "":
        return "None"

    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\(\s+", "(", s)
    s = re.sub(r"\s+\)", ")", s)
    s = re.sub(r"\s+-\s+", "-", s)

    # 1) Remove wild-type
    s = re.sub(r"\bwild[\s\-]?type\b", "", s, flags=re.I)

    # 2) Remove trailing 'cell' / 'cells'
    #    Example: "hONS cells" -> "hONS", "THP-1 cell" -> "THP-1"
    s = re.sub(r"\bcells?\b$", "", s, flags=re.I).strip()

    # 3) Clean up text around parentheses and commas
    s = re.sub(r"\s+,", ",", s)
    s = re.sub(r",\s*$", "", s)
    s = re.sub(r"\(\s*\)", "", s)
    s = re.sub(r"\s{2,}", " ", s).strip()

    # 4) Apply one additional abbreviation pass for very simple patterns
    #    Safely handles cases such as "A549 cells"
    m = re.fullmatch(r"([A-Za-z0-9][A-Za-z0-9\-\./+ ]*)", s, flags=re.I)
    if m:
        s = m.group(1).strip()

    return s if s else "None"


def classify_cell_and_assay_pairs_with_llm(
    pair_rows: list,
    client,
    model: str,
    temperature: float = 0,
    batch_size: int = 80,
    event_rows: list = None,
    pdf_key: str = "MULTI",
    title: str = "MULTI",
    material: str = "MULTI",
) -> pd.DataFrame:
    if event_rows is None:
        event_rows = []

    unique_pairs = []
    seen = set()

    for row in pair_rows:
        cell_name = _tox_norm_str(row.get("cell_name"))
        cell_assay = _tox_norm_str(row.get("cell_assay"))

        if _tox_is_missing(cell_name) and _tox_is_missing(cell_assay):
            continue

        key = (_cell_name_key(cell_name), _cell_assay_key(cell_assay))
        if key not in seen:
            seen.add(key)
            unique_pairs.append({
                "cell_name": cell_name if cell_name else "None",
                "cell_assay": cell_assay if cell_assay else "None",
            })

    if not unique_pairs:
        return pd.DataFrame(columns=[
            "cell_name",
            "cell_assay",
            "keep_row",
            "normalized_cell_assay",
            "cell_origin",
            "cell_type",
            "confidence",
            "reason",
            "__cell_name_key__",
            "__cell_assay_key__",
        ])

    all_rows = []

    for i in range(0, len(unique_pairs), batch_size):
        batch = unique_pairs[i:i+batch_size]

        system_prompt = f"""
You are post-processing nanotoxicology extraction outputs.

For each input row, do THREE things:
1. classify cell_name -> cell_origin
2. classify cell_name -> cell_type
3. classify cell_assay -> keep/drop + normalized_cell_assay

Important:
- cell_origin must be inferred from cell_name only.
- cell_type must be inferred from cell_name only.
- keep/drop and normalized_cell_assay must be inferred from cell_assay only.

Definitions:
- cell_origin = tissue / organ / lineage origin, NOT biological species.
- Do NOT output Human / Mouse / Rat in cell_origin.
- Allowed cell_origin labels:
  {", ".join(CELL_ORIGIN_ALLOWED)}
- Allowed cell_type labels:
  {", ".join(CELL_TYPE_ALLOWED)}

Assay keep/drop rule:
Keep only assays that are directly related to:
- cell viability
- cytotoxicity
- membrane integrity / cell damage
- direct live/dead status
- direct apoptosis / necrosis status

Keep examples:
- MTT
- MTS
- XTT
- WST-1 / WST-8 / WST
- CCK-8
- LDH
- Neutral Red / NRU
- SRB
- Alamar blue
- Resazurin
- Trypan blue
- Calcein AM
- Live/Dead viability assay
- Annexin V/PI
- CellTiter-Glo
- CellTiter-Blue
- CytoTox-Glo
- ATP viability assay
- apoptosis assays directly used as cell death readouts
- necrosis assays directly used as cell death readouts

Drop examples:
- ROS
- oxidative stress
- DCF / DCFH-DA
- TBARS / MDA / lipid peroxidation
- cytokine release
- IL-6 / IL-8 / TNF-alpha
- ELISA
- inflammation markers
- uptake / internalization
- gene expression / mRNA / qPCR / RT-qPCR
- immune activation markers
- other mechanistic assays not directly representing viability/cytotoxicity/cell death

Rules:
- keep_row must be exactly "Yes" or "No".
- If keep_row = "No", normalized_cell_assay must be "None".
- If keep_row = "Yes", normalized_cell_assay must be one of:
  {", ".join(TOX_POSTPROCESS_ASSAY_ALLOWED)}
- Use the simplest canonical assay name.
- Keep reason short.
- Return JSON only.
"""

        user_prompt = (
            "Post-process the following cell_name / cell_assay rows.\n\n"
            f"{json.dumps(batch, ensure_ascii=False, indent=2)}"
        )

        resp = _tox_logged_chat_completion_create(
            client=client,
            model=model,
            temperature=temperature,
            response_format={
                "type": "json_schema",
                "json_schema": TOX_CELL_ASSAY_POSTPROCESS_SCHEMA
            },
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],

            event_rows=event_rows,
            stage="tox_postprocess_cell_assay_api",
            pdf_key=pdf_key,
            title=title,
            material=material,

            api_call_name="classify_cell_and_assay_pairs_with_llm",
            candidate_row_id=-1,
            source_page=-1,
            caption_len=-1,
            n_images=0,
        )

        result = json.loads(resp.choices[0].message.content)
        all_rows.extend(result.get("items", []))

    df_map = pd.DataFrame(all_rows)

    if df_map.empty:
        return pd.DataFrame(columns=[
            "cell_name",
            "cell_assay",
            "keep_row",
            "normalized_cell_assay",
            "cell_origin",
            "cell_type",
            "confidence",
            "reason",
            "__cell_name_key__",
            "__cell_assay_key__",
        ])

    df_map["cell_name"] = df_map["cell_name"].map(_tox_norm_str)
    df_map["cell_assay"] = df_map["cell_assay"].map(_tox_norm_str)

    df_map["keep_row"] = df_map["keep_row"].map(_normalize_keep_row_label)
    df_map["normalized_cell_assay"] = df_map["normalized_cell_assay"].map(_normalize_cell_assay_label)
    df_map["cell_origin"] = df_map["cell_origin"].map(_normalize_cell_origin_label)
    df_map["cell_type"] = df_map["cell_type"].map(_normalize_cell_type_label)
    df_map["confidence"] = df_map["confidence"].map(_tox_norm_str)
    df_map["reason"] = df_map["reason"].map(_tox_norm_str)

    df_map.loc[df_map["keep_row"] == "No", "normalized_cell_assay"] = "None"

    df_map["__cell_name_key__"] = df_map["cell_name"].map(_cell_name_key)
    df_map["__cell_assay_key__"] = df_map["cell_assay"].map(_cell_assay_key)

    df_map = (
        df_map
        .drop_duplicates(subset=["__cell_name_key__", "__cell_assay_key__"])
        .sort_values(["cell_name", "cell_assay"])
        .reset_index(drop=True)
    )

    return df_map


def build_tox_cell_assay_postprocess_map_from_outputs(
    df_tox_records: pd.DataFrame,
    df_tox_claims: pd.DataFrame,
    client,
    model: str,
    temperature: float = 0,
    batch_size: int = 80,
    event_rows: list = None,
    pdf_key: str = "MULTI",
    title: str = "MULTI",
    material: str = "MULTI",
) -> pd.DataFrame:
    if event_rows is None:
        event_rows = []

    pair_rows = []

    if df_tox_records is not None and not df_tox_records.empty:
        if "cell_name" in df_tox_records.columns and "cell_assay" in df_tox_records.columns:
            pair_rows.extend(
                df_tox_records[["cell_name", "cell_assay"]].fillna("None").to_dict(orient="records")
            )

    return classify_cell_and_assay_pairs_with_llm(
        pair_rows=pair_rows,
        client=client,
        model=model,
        temperature=temperature,
        batch_size=batch_size,
        event_rows=event_rows,
        pdf_key=pdf_key,
        title=title,
        material=material,
    )


def apply_tox_cell_origin_type_from_assay_postprocess_map(
    df: pd.DataFrame,
    postprocess_map_df: pd.DataFrame
) -> pd.DataFrame:

    if df is None:
        return df
    if df.empty:
        return df
    if "cell_name" not in df.columns or "cell_assay" not in df.columns:
        return df

    out = df.copy()
    original_cols = out.columns.tolist()

    out["__cell_name_key__"] = out["cell_name"].map(_cell_name_key)
    out["__cell_assay_key__"] = out["cell_assay"].map(_cell_assay_key)

    if postprocess_map_df is None or postprocess_map_df.empty:
        out = out.drop(columns=["__cell_name_key__", "__cell_assay_key__"], errors="ignore")
        return out

    map_df = postprocess_map_df[
        [
            "__cell_name_key__",
            "__cell_assay_key__",
            "cell_origin",
            "cell_type",
        ]
    ].drop_duplicates(subset=["__cell_name_key__", "__cell_assay_key__"])

    out = out.drop(columns=["cell_origin", "cell_type"], errors="ignore")

    out = out.merge(
        map_df,
        on=["__cell_name_key__", "__cell_assay_key__"],
        how="left"
    )

    out["cell_origin"] = out["cell_origin"].fillna("None")
    out["cell_type"] = out["cell_type"].fillna("None")

    out = out.drop(
        columns=["__cell_name_key__", "__cell_assay_key__"],
        errors="ignore"
    )

    reordered_cols = [c for c in original_cols if c in out.columns]
    extra_cols = [c for c in out.columns if c not in reordered_cols]
    out = out[reordered_cols + extra_cols].reset_index(drop=True)

    return out


def apply_tox_cell_assay_postprocess_map(
    df: pd.DataFrame,
    postprocess_map_df: pd.DataFrame
) -> pd.DataFrame:
    if df is None:
        return df
    if df.empty:
        return df
    if "cell_name" not in df.columns or "cell_assay" not in df.columns:
        return df

    out = df.copy()
    original_cols = out.columns.tolist()

    out["__cell_name_key__"] = out["cell_name"].map(_cell_name_key)
    out["__cell_assay_key__"] = out["cell_assay"].map(_cell_assay_key)

    if postprocess_map_df is None or postprocess_map_df.empty:
        out = out.drop(columns=["__cell_name_key__", "__cell_assay_key__"], errors="ignore")
        return out

    map_df = postprocess_map_df[[
        "__cell_name_key__",
        "__cell_assay_key__",
        "keep_row",
        "normalized_cell_assay",
        "cell_origin",
        "cell_type",
        "confidence",
        "reason",
    ]].drop_duplicates(subset=["__cell_name_key__", "__cell_assay_key__"])

    out = out.drop(columns=["cell_origin", "cell_type"], errors="ignore")

    out = out.merge(
        map_df,
        on=["__cell_name_key__", "__cell_assay_key__"],
        how="left"
    )

    out["keep_row"] = out["keep_row"].fillna("No")
    out["normalized_cell_assay"] = out["normalized_cell_assay"].fillna("None")
    out["cell_origin"] = out["cell_origin"].fillna("None")
    out["cell_type"] = out["cell_type"].fillna("None")

    # Remove rows marked for deletion
    out = out[out["keep_row"] == "Yes"].copy()

    # Apply assay-name normalization
    norm_mask = out["normalized_cell_assay"].notna() & (out["normalized_cell_assay"] != "None")
    out.loc[norm_mask, "cell_assay"] = out.loc[norm_mask, "normalized_cell_assay"]

    out = out.drop(
        columns=[
            "__cell_name_key__",
            "__cell_assay_key__",
            "keep_row",
            "normalized_cell_assay",
            "confidence",
            "reason",
        ],
        errors="ignore"
    )

    reordered_cols = [c for c in original_cols if c in out.columns]
    extra_cols = [c for c in out.columns if c not in reordered_cols]
    out = out[reordered_cols + extra_cols].reset_index(drop=True)

    return out


def classify_cell_names_with_llm(
    cell_name_rows: list,
    client,
    model: str,
    temperature: float = 0,
    batch_size: int = 80,
    event_rows: list = None,
    pdf_key: str = "MULTI",
    title: str = "MULTI",
    material: str = "MULTI",
) -> pd.DataFrame:
    if event_rows is None:
        event_rows = []

    unique_names = []
    seen = set()

    for row in cell_name_rows:
        cell_name = _tox_norm_str(row.get("cell_name"))
        if _tox_is_missing(cell_name):
            continue

        key = _cell_name_key(cell_name)
        if key not in seen:
            seen.add(key)
            unique_names.append({"cell_name": cell_name})

    if not unique_names:
        return pd.DataFrame(columns=[
            "cell_name",
            "normalized_cell_name",
            "changed",
            "confidence",
            "reason",
            "__cell_name_key__",
        ])

    all_rows = []

    for i in range(0, len(unique_names), batch_size):
        batch = unique_names[i:i+batch_size]

        system_prompt = """
You are post-processing extracted cell names from nanotoxicology papers.

Your task:
- normalize each cell_name into the preferred abbreviated / standard form
- preserve biological identity
- preserve meaningful derivative / engineered modifiers
- do NOT collapse distinct engineered variants into the parent line

Main rule:
- If a standard abbreviation or standard cell-line label exists, prefer that abbreviated form.
- If the original includes a full name followed by an abbreviation in parentheses, use the abbreviation as the base name.
- If the original is already a concise standard cell-line label, keep it.
- Preserve meaningful suffixes/modifiers such as:
  - overexpressing
  - knockdown
  - antisense transfected
  - resistant
  - clone names
  - derivative / stable transfectant markers

Examples:
- human cardiac microvascular endothelial cells (HCMECs) -> HCMECs
- HCMECs -> HCMECs
- human bronchial epithelial cells (BEAS-2B) -> BEAS-2B
- alveolar epithelial cells (A549) -> A549
- A549 cells -> A549
- MCF-7 -> MCF-7
- MCF-7 MGST1-overexpressing -> MCF-7 MGST1-overexpressing
- MCF-7 antisense transfected -> MCF-7 antisense transfected
- RAW 264.7 macrophages -> RAW264.7
- THP-1 cells -> THP-1

Rules:
- Do NOT invent a new identity.
- Do NOT merge clearly distinct engineered variants into the parent line.
- If no safe simplification exists, keep the original.
- changed must be exactly "Yes" or "No".
- Return JSON only.
"""

        user_prompt = (
            "Normalize the following cell_name rows.\n\n"
            f"{json.dumps(batch, ensure_ascii=False, indent=2)}"
        )

        resp = _tox_logged_chat_completion_create(
            client=client,
            model=model,
            temperature=temperature,
            response_format={
                "type": "json_schema",
                "json_schema": TOX_CELL_NAME_POSTPROCESS_SCHEMA
            },
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],

            event_rows=event_rows,
            stage="tox_postprocess_cell_name_api",
            pdf_key=pdf_key,
            title=title,
            material=material,

            api_call_name="classify_cell_names_with_llm",
            candidate_row_id=-1,
            source_page=-1,
            caption_len=-1,
            n_images=0,
        )

        result = json.loads(resp.choices[0].message.content)
        all_rows.extend(result.get("items", []))

    df_map = pd.DataFrame(all_rows)

    if df_map.empty:
        return pd.DataFrame(columns=[
            "cell_name",
            "normalized_cell_name",
            "changed",
            "confidence",
            "reason",
            "__cell_name_key__",
        ])

    df_map["cell_name"] = df_map["cell_name"].map(_tox_norm_str)
    df_map["normalized_cell_name"] = df_map["normalized_cell_name"].map(_normalize_cell_name_final_label)
    df_map["changed"] = df_map["changed"].map(
        lambda x: "Yes" if _tox_norm_str(x).lower() in {"yes", "y", "true", "1"} else "No"
    )
    df_map["confidence"] = df_map["confidence"].map(_tox_norm_str)
    df_map["reason"] = df_map["reason"].map(_tox_norm_str)

    missing_mask = df_map["normalized_cell_name"].map(_tox_is_missing)
    df_map.loc[missing_mask, "normalized_cell_name"] = df_map.loc[missing_mask, "cell_name"]

    same_mask = (
        df_map["cell_name"].map(_cell_name_key) ==
        df_map["normalized_cell_name"].map(_cell_name_key)
    )
    df_map.loc[same_mask, "changed"] = "No"

    df_map["__cell_name_key__"] = df_map["cell_name"].map(_cell_name_key)

    df_map = (
        df_map
        .drop_duplicates(subset=["__cell_name_key__"])
        .sort_values(["cell_name"])
        .reset_index(drop=True)
    )

    return df_map


def build_tox_cell_name_postprocess_map_from_outputs(
    df_tox_records: pd.DataFrame,
    df_tox_claims: pd.DataFrame,
    client,
    model: str,
    temperature: float = 0,
    batch_size: int = 80,
    event_rows: list = None,
    pdf_key: str = "MULTI",
    title: str = "MULTI",
    material: str = "MULTI",
) -> pd.DataFrame:
    if event_rows is None:
        event_rows = []

    cell_name_rows = []

    if df_tox_records is not None and not df_tox_records.empty and "cell_name" in df_tox_records.columns:
        cell_name_rows.extend(
            df_tox_records[["cell_name"]].fillna("None").to_dict(orient="records")
        )

    return classify_cell_names_with_llm(
        cell_name_rows=cell_name_rows,
        client=client,
        model=model,
        temperature=temperature,
        batch_size=batch_size,
        event_rows=event_rows,
        pdf_key=pdf_key,
        title=title,
        material=material,
    )


def apply_tox_cell_name_postprocess_map(
    df: pd.DataFrame,
    cell_name_map_df: pd.DataFrame
) -> pd.DataFrame:
    if df is None:
        return df
    if df.empty:
        return df
    if "cell_name" not in df.columns:
        return df

    out = df.copy()
    original_cols = out.columns.tolist()

    out["__cell_name_key__"] = out["cell_name"].map(_cell_name_key)

    if cell_name_map_df is None or cell_name_map_df.empty:
        out = out.drop(columns=["__cell_name_key__"], errors="ignore")
        return out

    map_df = cell_name_map_df[[
        "__cell_name_key__",
        "normalized_cell_name",
        "changed",
        "confidence",
        "reason",
    ]].drop_duplicates(subset=["__cell_name_key__"])

    out = out.merge(
        map_df,
        on="__cell_name_key__",
        how="left"
    )

    out["normalized_cell_name"] = out["normalized_cell_name"].fillna("None")

    replace_mask = out["normalized_cell_name"].notna() & (out["normalized_cell_name"] != "None")
    out.loc[replace_mask, "cell_name"] = out.loc[replace_mask, "normalized_cell_name"]

    out = out.drop(
        columns=[
            "__cell_name_key__",
            "normalized_cell_name",
            "changed",
            "confidence",
            "reason",
        ],
        errors="ignore"
    )

    reordered_cols = [c for c in original_cols if c in out.columns]
    extra_cols = [c for c in out.columns if c not in reordered_cols]
    out = out[reordered_cols + extra_cols].reset_index(drop=True)

    return out

TOX_RESULT_NORMALIZATION_GROUP_SCHEMA = {
    "name": "tox_result_normalization_group_classifier",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "group_id": {"type": "string"},
                        "scale_type": {
                            "type": "string",
                            "enum": [
                                "fraction_0_1",
                                "percent_0_100",
                                "unclear",
                            ],
                        },
                        "semantic_type": {
                            "type": "string",
                            "enum": [
                                "cell_viability",
                                "cytotoxicity_or_cell_death",
                                "unclear",
                            ],
                        },
                        "conversion_rule": {
                            "type": "string",
                            "enum": [
                                "identity",
                                "multiply_100",
                                "hundred_minus_x",
                                "hundred_minus_100x",
                                "none",
                            ],
                        },
                        "confidence": {"type": "string"},
                        "reason": {"type": "string"},
                    },
                    "required": [
                        "group_id",
                        "scale_type",
                        "semantic_type",
                        "conversion_rule",
                        "confidence",
                        "reason",
                    ],
                },
            }
        },
        "required": ["items"],
    },
}

def classify_tox_result_groups_with_llm(
    df: pd.DataFrame,
    client,
    model: str,
    temperature: float = 0,
    batch_size: int = 40,
    event_rows: list = None,
    pdf_key: str = "MULTI",
    title: str = "MULTI",
    material: str = "MULTI",
) -> pd.DataFrame:
    if event_rows is None:
        event_rows = []

    if df is None or df.empty:
        return pd.DataFrame(
            columns=[
                "group_id",
                "scale_type",
                "semantic_type",
                "conversion_rule",
                "confidence",
                "reason",
            ]
        )

    _, group_payloads = _tox_build_result_normalization_groups(df)

    if not group_payloads:
        return pd.DataFrame(
            columns=[
                "group_id",
                "scale_type",
                "semantic_type",
                "conversion_rule",
                "confidence",
                "reason",
            ]
        )

    out_rows = []

    for i in range(0, len(group_payloads), batch_size):
        batch = group_payloads[i:i+batch_size]

        system_prompt = """
You are classifying toxicity assay result groups for post-processing normalization.

Each input group already represents a single coherent source context, typically one panel / one caption context / one endpoint context.
Do NOT decide per row.
Decide ONE interpretation for the WHOLE group.

Goal:
- determine whether the assay_results scale is:
  - fraction_0_1
  - percent_0_100
  - unclear
- determine whether the assay_results semantic meaning is:
  - cell_viability
  - cytotoxicity_or_cell_death
  - unclear
- determine the correct deterministic conversion rule to transform the group into:
  - 0~100 scale
  - cell viability meaning
- do NOT invent or modify assay_results values yourself

Important decision priority:
1. Use y_title first if informative.
2. Then use x_title, source_caption, panel_label, endpoint_type, cell_assay.
3. Then use the numeric pattern across rows.
4. The same assay name can appear with different semantics across panels, so do NOT rely on cell_assay alone.
5. If evidence is insufficient, use unclear / none conservatively.

Allowed outputs:
- scale_type:
  - fraction_0_1
  - percent_0_100
  - unclear
- semantic_type:
  - cell_viability
  - cytotoxicity_or_cell_death
  - unclear
- conversion_rule:
  - identity
  - multiply_100
  - hundred_minus_x
  - hundred_minus_100x
  - none

Interpretation examples:
- y_title = "Cell viability (%)", values = [72, 81, 95]
  -> percent_0_100 + cell_viability + identity
- y_title = "Relative viability", values = [0.72, 0.81, 0.95]
  -> fraction_0_1 + cell_viability + multiply_100
- y_title = "Cytotoxicity (%)", values = [5, 18, 42]
  -> percent_0_100 + cytotoxicity_or_cell_death + hundred_minus_x
- y_title = "LDH release", values = [0.05, 0.18, 0.42]
  -> fraction_0_1 + cytotoxicity_or_cell_death + hundred_minus_100x

Rules:
- If the group clearly expresses viability already in percent, use identity.
- If the group clearly expresses viability in 0~1 scale, use multiply_100.
- If the group clearly expresses cytotoxicity/cell death in percent, use hundred_minus_x.
- If the group clearly expresses cytotoxicity/cell death in 0~1 scale, use hundred_minus_100x.
- If unclear, use conversion_rule = none.
- Return one item per group_id.
- Keep reason short.
- Return JSON only.
"""

        user_prompt = (
            "Classify the following toxicity result groups.\n\n"
            f"{json.dumps(batch, ensure_ascii=False, indent=2)}"
        )

        resp = _tox_logged_chat_completion_create(
            client=client,
            model=model,
            temperature=temperature,
            response_format={
                "type": "json_schema",
                "json_schema": TOX_RESULT_NORMALIZATION_GROUP_SCHEMA
            },
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],

            event_rows=event_rows,
            stage="tox_result_normalization_api",
            pdf_key=pdf_key,
            title=title,
            material=material,

            api_call_name="classify_tox_result_groups_with_llm",
            candidate_row_id=-1,
            source_page=-1,
            caption_len=-1,
            n_images=0,
        )

        result = json.loads(resp.choices[0].message.content)
        out_rows.extend(result.get("items", []))

    df_map = pd.DataFrame(out_rows)

    if df_map.empty:
        return pd.DataFrame(
            columns=[
                "group_id",
                "scale_type",
                "semantic_type",
                "conversion_rule",
                "confidence",
                "reason",
            ]
        )

    df_map["group_id"] = df_map["group_id"].map(_tox_norm_str)
    df_map["scale_type"] = df_map["scale_type"].map(_tox_normalize_scale_type_label)
    df_map["semantic_type"] = df_map["semantic_type"].map(_tox_normalize_semantic_type_label)
    df_map["conversion_rule"] = df_map["conversion_rule"].map(_tox_normalize_conversion_rule_label)
    df_map["confidence"] = df_map["confidence"].map(_tox_norm_str)
    df_map["reason"] = df_map["reason"].map(_tox_norm_str)

    df_map = (
        df_map
        .drop_duplicates(subset=["group_id"])
        .sort_values("group_id")
        .reset_index(drop=True)
    )

    return df_map


def apply_tox_final_row_postprocess(
    df: pd.DataFrame,
    client,
    model: str,
    temperature: float = 0,
    batch_size: int = 40,
    event_rows: list = None,
    pdf_key: str = "MULTI",
    title: str = "MULTI",
    material: str = "MULTI",
) -> pd.DataFrame:
    """
    Final row-level cleanup.
    1) Drop rows if either exposure_time or exposure_dose is None (strict mode).
    2) Group rows into normalization groups centered on source, panel, and y_title.
    3) Let the LLM interpret the meaning of each group.
    4) Normalize assay_results according to the interpretation.
    5) Remove exact duplicate rows.
    """
    if event_rows is None:
        event_rows = []

    if df is None:
        return df
    if df.empty:
        return df

    out = df.copy().reset_index(drop=True)

    required_cols = ["exposure_time", "exposure_dose"]
    for col in required_cols:
        if col not in out.columns:
            out[col] = "None"

    keep_mask = (
        ~out["exposure_time"].map(_tox_is_missing) &
        ~out["exposure_dose"].map(_tox_is_missing)
    )
    out = out[keep_mask].copy().reset_index(drop=True)

    if out.empty:
        return out

    needed_cols = [
        "pdf_key",
        "material",
        "cell_assay",
        "cell_name",
        "source_page",
        "source_caption",
        "panel_label",
        "x_title",
        "y_title",
        "endpoint_type",
        "evidence_text",
        "exposure_time_unit",
        "exposure_dose_unit",
        "assay_results",
    ]
    for col in needed_cols:
        if col not in out.columns:
            out[col] = "None"

    grouped_df, _ = _tox_build_result_normalization_groups(out)

    group_map_df = classify_tox_result_groups_with_llm(
        df=grouped_df.drop(columns=["group_key_tmp"], errors="ignore"),
        client=client,
        model=model,
        temperature=temperature,
        batch_size=batch_size,
        event_rows=event_rows,
        pdf_key=pdf_key,
        title=title,
        material=material,
    )

    if group_map_df.empty:
        grouped_df["conversion_rule"] = "none"
    else:
        grouped_df = grouped_df.merge(
            group_map_df[
                [
                    "group_id",
                    "scale_type",
                    "semantic_type",
                    "conversion_rule",
                    "confidence",
                    "reason",
                ]
            ],
            left_on="group_id_tmp",
            right_on="group_id",
            how="left"
        )
        grouped_df["conversion_rule"] = grouped_df["conversion_rule"].fillna("none")
        grouped_df["scale_type"] = grouped_df["scale_type"].fillna("unclear")
        grouped_df["semantic_type"] = grouped_df["semantic_type"].fillna("unclear")

    grouped_df["assay_results"] = grouped_df.apply(
        lambda r: _tox_apply_conversion_rule_to_value(
            r.get("assay_results", "None"),
            r.get("conversion_rule", "none")
        ),
        axis=1
    )

    grouped_df["exposure_time_num"] = pd.to_numeric(grouped_df["exposure_time"], errors="coerce")
    grouped_df["exposure_dose_num"] = pd.to_numeric(grouped_df["exposure_dose"], errors="coerce")
    grouped_df["assay_results_num"] = pd.to_numeric(grouped_df["assay_results"], errors="coerce")

    grouped_df.drop(
        columns=[
            "row_id_tmp",
            "group_key_tmp",
            "group_id_tmp",
            "group_id",
            "time_sort_tmp",
            "dose_sort_tmp",
            "scale_type",
            "semantic_type",
            "conversion_rule",
            "confidence",
            "reason",
        ],
        inplace=True,
        errors="ignore"
    )

    grouped_df = grouped_df.drop_duplicates().reset_index(drop=True)

    return grouped_df


def run_tox_postprocess_cell_and_assay(
    df_tox_records: pd.DataFrame,
    df_tox_claims: pd.DataFrame,
    client,
    model: str,
    temperature: float = 0,
    batch_size: int = 80,
    cell_name_batch_size: int = 80,
    event_rows: list = None,
    pdf_key: str = "MULTI",
    title: str = "MULTI",
    material: str = "MULTI",
):
    """
    Run final Tox postprocessing in a fail-open manner.
    Prevent the entire batch from failing when an individual sub-step fails.
    """

    if event_rows is None:
        event_rows = []

    tox_claims_post = df_tox_claims.copy() if df_tox_claims is not None else pd.DataFrame()
    tox_records_post = df_tox_records.copy() if df_tox_records is not None else pd.DataFrame()

    assay_postprocess_map_df = pd.DataFrame()
    cell_name_postprocess_map_df = pd.DataFrame()

    # Postprocess stage 1: Apply shared final filters
    try:
        tox_records_post, tox_claims_post = run_tox_shared_final_filters(
            df_tox_records=tox_records_post,
            df_tox_claims=tox_claims_post,
            client=client,
            model=model,
            temperature=temperature,
            xtitle_batch_size=batch_size,
            extreme_threshold=150.0,
            event_rows=event_rows,
            pdf_key=pdf_key,
            title=title,
            material=material,
        )
    except Exception as e:
        _append_tox_event(
            event_rows=event_rows,
            level="batch",
            status="WARN",
            stage="tox_postprocess_shared_filters",
            pdf_key=pdf_key,
            title=title,
            material=material,
            extracted="No",
            error_type=type(e).__name__,
            message=str(e),
        )
        print(f"[WARN] run_tox_shared_final_filters failed | {e}")

    # Postprocess stage 2: Build assay and cell postprocess maps
    try:
        if tox_records_post is not None and not tox_records_post.empty:
            assay_postprocess_map_df = build_tox_cell_assay_postprocess_map_from_outputs(
                df_tox_records=tox_records_post,
                df_tox_claims=None,
                client=client,
                model=model,
                temperature=temperature,
                batch_size=batch_size,
                event_rows=event_rows,
                pdf_key=pdf_key,
                title=title,
                material=material,
            )
    except Exception as e:
        assay_postprocess_map_df = pd.DataFrame()
        _append_tox_event(
            event_rows=event_rows,
            level="batch",
            status="WARN",
            stage="tox_postprocess_assay_map",
            pdf_key=pdf_key,
            title=title,
            material=material,
            extracted="No",
            error_type=type(e).__name__,
            message=str(e),
        )
        print(f"[WARN] build_tox_cell_assay_postprocess_map_from_outputs failed | {e}")

    # Postprocess stage 3: Apply cell origin and cell type labels
    try:
        if (
            tox_records_post is not None
            and not tox_records_post.empty
            and assay_postprocess_map_df is not None
            and not assay_postprocess_map_df.empty
        ):
            tox_records_post = apply_tox_cell_origin_type_from_assay_postprocess_map(
                df=tox_records_post,
                postprocess_map_df=assay_postprocess_map_df
            )
    except Exception as e:
        _append_tox_event(
            event_rows=event_rows,
            level="batch",
            status="WARN",
            stage="tox_postprocess_apply_origin_type",
            pdf_key=pdf_key,
            title=title,
            material=material,
            extracted="No",
            error_type=type(e).__name__,
            message=str(e),
        )
        print(f"[WARN] apply_tox_cell_origin_type_from_assay_postprocess_map failed | {e}")

    # Postprocess stage 4: Build cell-name normalization map
    try:
        if tox_records_post is not None and not tox_records_post.empty:
            cell_name_postprocess_map_df = build_tox_cell_name_postprocess_map_from_outputs(
                df_tox_records=tox_records_post,
                df_tox_claims=None,
                client=client,
                model=model,
                temperature=temperature,
                batch_size=cell_name_batch_size,
                event_rows=event_rows,
                pdf_key=pdf_key,
                title=title,
                material=material,
            )
    except Exception as e:
        cell_name_postprocess_map_df = pd.DataFrame()
        _append_tox_event(
            event_rows=event_rows,
            level="batch",
            status="WARN",
            stage="tox_postprocess_cell_name_map",
            pdf_key=pdf_key,
            title=title,
            material=material,
            extracted="No",
            error_type=type(e).__name__,
            message=str(e),
        )
        print(f"[WARN] build_tox_cell_name_postprocess_map_from_outputs failed | {e}")

    # Postprocess stage 5: Apply cell-name normalization
    try:
        if (
            tox_records_post is not None
            and not tox_records_post.empty
            and cell_name_postprocess_map_df is not None
            and not cell_name_postprocess_map_df.empty
        ):
            tox_records_post = apply_tox_cell_name_postprocess_map(
                df=tox_records_post,
                cell_name_map_df=cell_name_postprocess_map_df
            )
    except Exception as e:
        _append_tox_event(
            event_rows=event_rows,
            level="batch",
            status="WARN",
            stage="tox_postprocess_apply_cell_name",
            pdf_key=pdf_key,
            title=title,
            material=material,
            extracted="No",
            error_type=type(e).__name__,
            message=str(e),
        )
        print(f"[WARN] apply_tox_cell_name_postprocess_map failed | {e}")

    # Postprocess stage 6: Apply final row-level normalization
    try:
        if tox_records_post is not None and not tox_records_post.empty:
            tox_records_post = apply_tox_final_row_postprocess(
                df=tox_records_post,
                client=client,
                model=model,
                temperature=temperature,
                batch_size=batch_size,
                event_rows=event_rows,
                pdf_key=pdf_key,
                title=title,
                material=material,
            )
    except Exception as e:
        _append_tox_event(
            event_rows=event_rows,
            level="batch",
            status="WARN",
            stage="tox_postprocess_final_row_normalization",
            pdf_key=pdf_key,
            title=title,
            material=material,
            extracted="No",
            error_type=type(e).__name__,
            message=str(e),
        )
        print(f"[WARN] apply_tox_final_row_postprocess failed | {e}")

    # Postprocess stage 7: Add synthetic controls
    try:
        if tox_records_post is not None and not tox_records_post.empty:
            tox_records_post = add_synthetic_controls_to_tox_records(
                df_tox_records=tox_records_post,
                group_cols=TOX_SYNTHETIC_CONTROL_GROUP_COLS,
                zero_dose_value="0",
                zero_assay_result_value="100",
            )
    except Exception as e:
        _append_tox_event(
            event_rows=event_rows,
            level="batch",
            status="WARN",
            stage="tox_postprocess_add_synthetic_control",
            pdf_key=pdf_key,
            title=title,
            material=material,
            extracted="No",
            error_type=type(e).__name__,
            message=str(e),
        )
        print(f"[WARN] add_synthetic_controls_to_tox_records failed | {e}")

    # Postprocess stage 8: Final cleanup
    try:
        if tox_records_post is not None and not tox_records_post.empty:
            tox_records_post = tox_records_post.drop_duplicates().reset_index(drop=True)
            tox_records_post = _tox_sort_records(tox_records_post)
    except Exception as e:
        _append_tox_event(
            event_rows=event_rows,
            level="batch",
            status="WARN",
            stage="tox_postprocess_sort_records",
            pdf_key=pdf_key,
            title=title,
            material=material,
            extracted="No",
            error_type=type(e).__name__,
            message=str(e),
        )
        print(f"[WARN] final tox record sort failed | {e}")

    try:
        if tox_claims_post is not None and not tox_claims_post.empty:
            tox_claims_post = tox_claims_post.drop_duplicates().reset_index(drop=True)
            tox_claims_post = _tox_sort_claims(tox_claims_post)
    except Exception as e:
        _append_tox_event(
            event_rows=event_rows,
            level="batch",
            status="WARN",
            stage="tox_postprocess_sort_claims",
            pdf_key=pdf_key,
            title=title,
            material=material,
            extracted="No",
            error_type=type(e).__name__,
            message=str(e),
        )
        print(f"[WARN] final tox claim sort failed | {e}")

    return (
        tox_records_post,
        tox_claims_post,
        assay_postprocess_map_df,
        cell_name_postprocess_map_df,
    )


def _tox_normalize_scale_type_label(x: str) -> str:
    s = _tox_norm_str(x).lower()
    if s in {"fraction_0_1", "fraction", "0_1", "0-1", "0 to 1"}:
        return "fraction_0_1"
    if s in {"percent_0_100", "percent", "percentage", "0_100", "0-100", "0 to 100"}:
        return "percent_0_100"
    return "unclear"


def _tox_normalize_semantic_type_label(x: str) -> str:
    s = _tox_norm_str(x).lower()
    if s in {"cell_viability", "viability", "cell viability"}:
        return "cell_viability"
    if s in {
        "cytotoxicity_or_cell_death",
        "cytotoxicity",
        "cell_death",
        "cell death",
        "death",
    }:
        return "cytotoxicity_or_cell_death"
    return "unclear"


def _tox_normalize_conversion_rule_label(x: str) -> str:
    s = _tox_norm_str(x).lower()
    if s in {"identity", "keep", "as_is", "asis"}:
        return "identity"
    if s in {"multiply_100", "x100", "times_100", "fraction_to_percent"}:
        return "multiply_100"
    if s in {"hundred_minus_x", "100-x", "100_minus_x"}:
        return "hundred_minus_x"
    if s in {"hundred_minus_100x", "100-(100x)", "100_minus_100x", "100-(x*100)"}:
        return "hundred_minus_100x"
    return "none"


def _tox_apply_conversion_rule_to_value(value, conversion_rule: str) -> str:
    val = _tox_clean_numeric_value(value)
    if val == "None":
        return "None"

    try:
        num = float(val)
    except Exception:
        return "None"

    rule = _tox_normalize_conversion_rule_label(conversion_rule)

    if rule == "identity":
        out = num
    elif rule == "multiply_100":
        out = num * 100.0
    elif rule == "hundred_minus_x":
        out = 100.0 - num
    elif rule == "hundred_minus_100x":
        out = 100.0 - (num * 100.0)
    else:
        # none / unclear -> keep the original value
        out = num

    return _format_float_str(out)


def _tox_group_sort_value(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce")


def _tox_build_result_normalization_group_key(row: pd.Series):
    def _page_int(x):
        try:
            return int(x)
        except Exception:
            return -1

    return (
        _tox_norm_str(row.get("pdf_key", "None")),
        _tox_norm_str(row.get("material", "None")),
        _tox_norm_str(row.get("cell_assay", "None")),
        _tox_norm_str(row.get("cell_name", "None")),
        _page_int(row.get("source_page", -1)),
        _tox_norm_str(row.get("source_caption", "None")),
        _tox_norm_str(row.get("panel_label", "None")) if "panel_label" in row.index else "None",
        _tox_norm_str(row.get("y_title", "None")),
        _tox_norm_str(row.get("x_title", "None")),
    )


def _tox_build_result_normalization_groups(df: pd.DataFrame) -> tuple[pd.DataFrame, list]:
    work = df.copy().reset_index(drop=True)
    work["row_id_tmp"] = work.index.astype(int)

    required_cols = [
        "pdf_key",
        "material",
        "cell_assay",
        "cell_name",
        "source_page",
        "source_caption",
        "panel_label",
        "x_title",
        "y_title",
        "exposure_time",
        "exposure_time_unit",
        "exposure_dose",
        "exposure_dose_unit",
        "assay_results",
        "endpoint_type",
        "evidence_text",
    ]
    for col in required_cols:
        if col not in work.columns:
            work[col] = "None"

    work["group_key_tmp"] = work.apply(_tox_build_result_normalization_group_key, axis=1)

    unique_keys = list(dict.fromkeys(work["group_key_tmp"].tolist()))
    group_id_map = {k: f"G{idx+1:05d}" for idx, k in enumerate(unique_keys)}
    work["group_id_tmp"] = work["group_key_tmp"].map(group_id_map)

    group_payloads = []
    for gkey in unique_keys:
        gid = group_id_map[gkey]
        gdf = work[work["group_key_tmp"] == gkey].copy()

        gdf["time_sort_tmp"] = _tox_group_sort_value(gdf["exposure_time"])
        gdf["dose_sort_tmp"] = _tox_group_sort_value(gdf["exposure_dose"])
        gdf = gdf.sort_values(
            by=["time_sort_tmp", "dose_sort_tmp", "row_id_tmp"],
            kind="stable"
        ).reset_index(drop=True)

        head = gdf.iloc[0]

        rows = []
        for r in gdf.itertuples(index=False):
            rows.append({
                "row_id": int(getattr(r, "row_id_tmp")),
                "exposure_time": _tox_norm_str(getattr(r, "exposure_time", "None")) or "None",
                "exposure_time_unit": _tox_norm_str(getattr(r, "exposure_time_unit", "None")) or "None",
                "exposure_dose": _tox_norm_str(getattr(r, "exposure_dose", "None")) or "None",
                "exposure_dose_unit": _tox_norm_str(getattr(r, "exposure_dose_unit", "None")) or "None",
                "assay_results": _tox_norm_str(getattr(r, "assay_results", "None")) or "None",
            })

        group_payloads.append({
            "group_id": gid,
            "material": _tox_norm_str(head["material"]) or "None",
            "cell_assay": _tox_norm_str(head["cell_assay"]) or "None",
            "cell_name": _tox_norm_str(head["cell_name"]) or "None",
            "source_page": int(head["source_page"]) if str(head["source_page"]).strip().lstrip("-").isdigit() else -1,
            "source_caption": _tox_norm_str(head["source_caption"]) or "None",
            "panel_label": _tox_norm_str(head["panel_label"]) or "None",
            "x_title": _tox_norm_str(head["x_title"]) or "None",
            "y_title": _tox_norm_str(head["y_title"]) or "None",
            "endpoint_type": _tox_norm_str(head["endpoint_type"]) or "None",
            "evidence_text": _tox_norm_str(head["evidence_text"]) or "None",
            "rows": rows,
        })

    return work, group_payloads

In [ ]:
# Step 8.15: Generate deterministic synthetic control rows

TOX_SYNTHETIC_CONTROL_GROUP_COLS = [
    "title",
    "material_name",
    "media",
    "cell_assay",
    "cell_name",
    "exposure_time",
]

TOX_SYNTHETIC_CONTROL_SOURCE_TYPE = "synthetic_control"
TOX_SYNTHETIC_CONTROL_VALUE_ORIGIN = "synthetic_control"
TOX_SYNTHETIC_CONTROL_SOURCE_CAPTION = (
    "Synthetic control added deterministically because no exposure_dose=0 row existed "
    "for the same title/material_name/media/cell_assay/cell_name/exposure_time group."
)
TOX_SYNTHETIC_CONTROL_EVIDENCE_TEXT = (
    "Synthetic control row added deterministically "
    "(exposure_dose=0, assay_results=100)."
)
TOX_SYNTHETIC_CONTROL_X_TITLE = "Synthetic"
TOX_SYNTHETIC_CONTROL_Y_TITLE = "Synthetic"


def _tox_first_non_missing_in_series(series, default="None"):
    for x in series:
        if not _tox_is_missing(x):
            return x
    return default


def _tox_normalize_synthetic_flag(x) -> int:
    s = _tox_norm_str(x).lower()
    if s in {"1", "1.0", "true", "yes", "y"}:
        return 1
    return 0


def _tox_is_zero_dose_value(x, tol: float = 1e-12) -> bool:
    dose = _tox_clean_dose_value(x)
    if dose == "None":
        return False
    try:
        return abs(float(dose)) <= tol
    except Exception:
        return False


def _tox_build_synthetic_control_row_from_group(
    group_df: pd.DataFrame,
    zero_dose_value: str = "0",
    zero_assay_result_value: str = "100",
) -> dict:
    """
    Within the same
    (title, material_name, media, cell_assay, cell_name, exposure_time)
    group, create a synthetic control row when no exposure_dose=0 row exists.
    """

    row = {}

    # Preserve existing group values as much as possible
    for col in group_df.columns:
        if col in {
            "exposure_dose",
            "assay_results",
            "is_synthetic_control",
            "source_row_id",
            "source_page",
            "source_type",
            "source_caption",
            "panel_label",
            "evidence_text",
            "value_origin",
            "x_title",
            "y_title",
            "assay_results_num",
            "exposure_dose_num",
        }:
            continue

        row[col] = _tox_first_non_missing_in_series(group_df[col], default="None")

    # Forced values for synthetic controls
    row["exposure_dose"] = str(zero_dose_value)
    row["assay_results"] = str(zero_assay_result_value)
    row["is_synthetic_control"] = 1

    # synthetic provenance
    row["source_row_id"] = -1
    row["source_page"] = -1
    row["source_type"] = TOX_SYNTHETIC_CONTROL_SOURCE_TYPE
    row["source_caption"] = TOX_SYNTHETIC_CONTROL_SOURCE_CAPTION
    row["panel_label"] = "Synthetic"
    row["evidence_text"] = TOX_SYNTHETIC_CONTROL_EVIDENCE_TEXT
    row["value_origin"] = TOX_SYNTHETIC_CONTROL_VALUE_ORIGIN

    # synthetic axis title
    row["x_title"] = TOX_SYNTHETIC_CONTROL_X_TITLE
    row["y_title"] = TOX_SYNTHETIC_CONTROL_Y_TITLE

    # Preserve the first non-missing unit-related value within the group
    if "exposure_dose_unit" in group_df.columns:
        row["exposure_dose_unit"] = _tox_first_non_missing_in_series(
            group_df["exposure_dose_unit"],
            default="None"
        )
    else:
        row["exposure_dose_unit"] = "None"

    if "exposure_time_unit" in group_df.columns:
        row["exposure_time_unit"] = _tox_first_non_missing_in_series(
            group_df["exposure_time_unit"],
            default="None"
        )
    else:
        row["exposure_time_unit"] = "None"

    # Adjust cell_assay_raw
    if _tox_is_missing(row.get("cell_assay_raw", "None")):
        row["cell_assay_raw"] = _tox_norm_str(row.get("cell_assay", "None")) or "None"

    # numeric helper
    row["assay_results_num"] = 100.0
    row["exposure_dose_num"] = 0.0

    return row


def add_synthetic_controls_to_tox_records(
    df_tox_records: pd.DataFrame,
    group_cols: list = None,
    zero_dose_value: str = "0",
    zero_assay_result_value: str = "100",
) -> pd.DataFrame:
    """
    For each title, material_name, media, cell_assay, cell_name, and exposure_time combination,
    add a synthetic control row when no exposure_dose=0 row exists.

    Notes:
    - Applies only to tox_records.
    - Does not apply to tox_claims.
    - Assumes application after final 0-100 scaling.
    """

    if group_cols is None:
        group_cols = TOX_SYNTHETIC_CONTROL_GROUP_COLS

    if df_tox_records is None:
        return df_tox_records

    out = df_tox_records.copy()

    if out.empty:
        if "is_synthetic_control" not in out.columns:
            out["is_synthetic_control"] = pd.Series(dtype="int64")
        return out

    required_cols = list(dict.fromkeys(group_cols + [
        "pdf_key",
        "title",
        "material",
        "material_name",
        "composition",
        "media",
        "core_size",
        "core_size_unit",
        "core_size_measurement_method",
        "core_size_source",
        "hydrodynamic_size",
        "hydrodynamic_size_unit",
        "hydrodynamic_size_measurement_method",
        "hydrodynamic_size_source",
        "surface_charge",
        "surface_charge_unit",
        "surface_charge_measurement_method",
        "surface_charge_source",
        "surface_area",
        "surface_area_unit",
        "surface_area_measurement_method",
        "surface_area_source",
        "cell_assay",
        "cell_name",
        "cell_species",
        "cell_origin",
        "cell_type",
        "exposure_time",
        "exposure_time_unit",
        "exposure_dose",
        "exposure_dose_unit",
        "assay_results",
        "x_title",
        "y_title",
        "endpoint_type",
        "value_origin",
        "evidence_text",
        "cell_assay_raw",
        "source_row_id",
        "source_page",
        "source_type",
        "source_caption",
        "panel_label",
        "is_synthetic_control",
    ]))

    for col in required_cols:
        if col not in out.columns:
            out[col] = "None"

    out["is_synthetic_control"] = out["is_synthetic_control"].map(_tox_normalize_synthetic_flag)

    new_rows = []

    grouped = out.groupby(group_cols, dropna=False, sort=False)

    for _, gdf in grouped:
        if gdf.empty:
            continue

        has_zero_dose = gdf["exposure_dose"].map(_tox_is_zero_dose_value).any()
        if has_zero_dose:
            continue

        synthetic_row = _tox_build_synthetic_control_row_from_group(
            group_df=gdf,
            zero_dose_value=zero_dose_value,
            zero_assay_result_value=zero_assay_result_value,
        )
        new_rows.append(synthetic_row)

    if new_rows:
        df_new = pd.DataFrame(new_rows)

        for col in out.columns:
            if col not in df_new.columns:
                df_new[col] = "None"
        for col in df_new.columns:
            if col not in out.columns:
                out[col] = "None"

        out = pd.concat(
            [out, df_new[out.columns]],
            ignore_index=True
        )

    out["is_synthetic_control"] = (
        out["is_synthetic_control"]
        .fillna(0)
        .map(_tox_normalize_synthetic_flag)
    )

    # Recompute numeric helper columns
    out = clean_tox_output(out)

    # Final cleanup
    out = out.drop_duplicates().reset_index(drop=True)
    out = _tox_sort_records(out)

    return out

# Step 9: Run the pipeline in batches

Process papers in batches, save intermediate outputs, maintain timing logs, and support targeted reruns through a manifest.

In [ ]:
# Step 9: Define batch runner utility functions
import os
import re
import gc
import math
import time
import json
import pandas as pd
from datetime import datetime
from pytz import timezone

BATCH_SIZE = 10

def _now_kst_str():
    return datetime.now(timezone("Asia/Seoul")).strftime("%Y%m%d_%H%M%S")

def _now_kst_dt_str():
    return datetime.now(timezone("Asia/Seoul")).strftime("%Y-%m-%d %H:%M:%S %Z")

def _safe_collection_name(name: str) -> str:
    return str(name).replace(" ", "_")

def make_stage_save_path(output_dir, collection_name, stage, run_ts, batch_id=None):
    safe_name = _safe_collection_name(collection_name)
    if batch_id is None:
        filename = f"{safe_name}_{stage}_{run_ts}.xlsx"
    else:
        filename = f"{safe_name}__Batch{batch_id}_{stage}_{run_ts}.xlsx"
    return os.path.join(output_dir, filename)

_illegal_characters_re = re.compile(r'[\x00-\x08\x0B-\x0C\x0E-\x1F]')

def clean_illegal_excel_chars(value):
    if isinstance(value, str):
        value = _illegal_characters_re.sub(" ", value)
        value = re.sub(r"\s+", " ", value).strip()
    return value

def clean_df_for_excel(df: pd.DataFrame) -> pd.DataFrame:
    if df is None:
        return pd.DataFrame()
    df = df.copy()
    if df.empty:
        return df
    obj_cols = df.select_dtypes(include=["object"]).columns
    for col in obj_cols:
        df[col] = df[col].map(clean_illegal_excel_chars)
    return df

def concat_or_empty(df_list, columns=None):
    valid = [x for x in df_list if isinstance(x, pd.DataFrame) and not x.empty]
    if valid:
        return pd.concat(valid, ignore_index=True)
    return pd.DataFrame(columns=columns)

def chunk_list(xs, size):
    for i in range(0, len(xs), size):
        yield xs[i:i+size]

def record_time(
    timing_rows: list,
    batch_id,
    stage: str,
    start_dt: str,
    start_ts: float,
    n_papers: int = 0,
    status: str = "SUCCESS",
    note: str = ""
):
    end_dt = _now_kst_dt_str()
    end_ts = time.perf_counter()
    timing_rows.append({
        "batch_id": batch_id,
        "stage": stage,
        "start_dt": start_dt,
        "end_dt": end_dt,
        "elapsed_sec": round(end_ts - start_ts, 3),
        "n_papers": int(n_papers),
        "status": status,
        "note": str(note),
    })

def save_time_workbook(timing_rows, output_dir, selected_collection, run_ts):
    df_time = pd.DataFrame(timing_rows)
    if df_time.empty:
        df_time = pd.DataFrame(columns=[
            "batch_id", "stage", "start_dt", "end_dt", "elapsed_sec", "n_papers", "status", "note"
        ])

    # batch summary pivot
    if not df_time.empty:
        df_summary = (
            df_time.pivot_table(
                index="batch_id",
                columns="stage",
                values="elapsed_sec",
                aggfunc="sum",
                fill_value=0
            )
            .reset_index()
        )
    else:
        df_summary = pd.DataFrame()

    save_path = make_stage_save_path(
        output_dir=output_dir,
        collection_name=selected_collection,
        stage="time",
        run_ts=run_ts,
        batch_id=None
    )

    with pd.ExcelWriter(save_path, engine="openpyxl") as writer:
        clean_df_for_excel(df_time).to_excel(writer, sheet_name="batch_stage_times", index=False)
        clean_df_for_excel(df_summary).to_excel(writer, sheet_name="batch_summary", index=False)

    print(f"[TIME SAVED] {save_path}")
    return save_path

def save_mat_batch(output_dir, selected_collection, batch_id, run_ts, df_mat):
    save_path = make_stage_save_path(output_dir, selected_collection, "mat", run_ts, batch_id=batch_id)
    clean_df_for_excel(df_mat).to_excel(save_path, index=False)
    print(f"[BATCH {batch_id}] Mat saved: {save_path}")
    return save_path

def save_pchem_batch(output_dir, selected_collection, batch_id, run_ts, df_pchem, df_pchem_claims):
    save_path = make_stage_save_path(output_dir, selected_collection, "pchem", run_ts, batch_id=batch_id)
    with pd.ExcelWriter(save_path, engine="openpyxl") as writer:
        clean_df_for_excel(df_pchem).to_excel(writer, sheet_name="records", index=False)
        clean_df_for_excel(df_pchem_claims).to_excel(writer, sheet_name="claims", index=False)
    print(f"[BATCH {batch_id}] PChem saved: {save_path}")
    return save_path

def save_tox_batch(
    output_dir,
    selected_collection,
    batch_id,
    run_ts,
    df_tox,
    df_tox_claims,
    df_tox_event_log=None,
    tox_postprocess_map_df=None,
    tox_cell_name_postprocess_map_df=None
):
    save_path = make_stage_save_path(output_dir, selected_collection, "tox", run_ts, batch_id=batch_id)

    if df_tox_event_log is None or df_tox_event_log.empty:
        df_tox_event_log = pd.DataFrame(columns=TOX_EVENT_LOG_COLUMNS)

    df_tox_save = _tox_sort_records(df_tox.copy()) if df_tox is not None else pd.DataFrame(columns=TOX_RECORD_COLUMNS)
    df_tox_claims_save = _tox_sort_claims(df_tox_claims.copy()) if df_tox_claims is not None else pd.DataFrame(columns=TOX_CLAIM_COLUMNS)

    with pd.ExcelWriter(save_path, engine="openpyxl") as writer:
        clean_df_for_excel(df_tox_save).to_excel(writer, sheet_name="tox_records", index=False)
        clean_df_for_excel(df_tox_claims_save).to_excel(writer, sheet_name="tox_claims", index=False)
        clean_df_for_excel(df_tox_event_log).to_excel(writer, sheet_name="tox_event_log", index=False)

        if tox_postprocess_map_df is not None and not tox_postprocess_map_df.empty:
            clean_df_for_excel(tox_postprocess_map_df).to_excel(
                writer,
                sheet_name="assay_postprocess_map",
                index=False
            )

        if tox_cell_name_postprocess_map_df is not None and not tox_cell_name_postprocess_map_df.empty:
            clean_df_for_excel(tox_cell_name_postprocess_map_df).to_excel(
                writer,
                sheet_name="cell_name_postprocess_map",
                index=False
            )

    print(f"[BATCH {batch_id}] Tox saved: {save_path}")
    return save_path

def save_final_mat(output_dir, selected_collection, run_ts, df_mat):
    save_path = make_stage_save_path(output_dir, selected_collection, "mat", run_ts, batch_id=None)
    clean_df_for_excel(df_mat).to_excel(save_path, index=False)
    print(f"[FINAL] Mat saved: {save_path}")
    return save_path

def save_final_pchem(output_dir, selected_collection, run_ts, df_pchem, df_pchem_claims):
    save_path = make_stage_save_path(output_dir, selected_collection, "pchem", run_ts, batch_id=None)
    with pd.ExcelWriter(save_path, engine="openpyxl") as writer:
        clean_df_for_excel(df_pchem).to_excel(writer, sheet_name="records", index=False)
        clean_df_for_excel(df_pchem_claims).to_excel(writer, sheet_name="claims", index=False)
    print(f"[FINAL] PChem saved: {save_path}")
    return save_path

def save_final_tox(
    output_dir,
    selected_collection,
    run_ts,
    df_tox,
    df_tox_claims,
    df_tox_event_log=None
):
    save_path = make_stage_save_path(output_dir, selected_collection, "tox", run_ts, batch_id=None)

    if df_tox_event_log is None or df_tox_event_log.empty:
        df_tox_event_log = pd.DataFrame(columns=TOX_EVENT_LOG_COLUMNS)

    df_tox_save = _tox_sort_records(df_tox.copy()) if df_tox is not None else pd.DataFrame(columns=TOX_RECORD_COLUMNS)
    df_tox_claims_save = _tox_sort_claims(df_tox_claims.copy()) if df_tox_claims is not None else pd.DataFrame(columns=TOX_CLAIM_COLUMNS)

    with pd.ExcelWriter(save_path, engine="openpyxl") as writer:
        clean_df_for_excel(df_tox_save).to_excel(writer, sheet_name="tox_records", index=False)
        clean_df_for_excel(df_tox_claims_save).to_excel(writer, sheet_name="tox_claims", index=False)
        clean_df_for_excel(df_tox_event_log).to_excel(writer, sheet_name="tox_event_log", index=False)

    print(f"[FINAL] Tox saved: {save_path}")
    return save_path

In [ ]:
# Step 9: Define per-paper processors and the batch execution wrapper

def extract_one_paper_text_bundle(paper, client, chat_model_name):
    full_text_raw = extract_full_text_from_pdf(paper["pdf_path"])

    full_text_clean, repair_log = repair_full_text_selectively(
        full_text_raw=full_text_raw,
        client=client,
        model=chat_model_name,
        temperature=0,
        batch_size=8,
        max_chunk_chars=1400
    )

    methods_text_clean = extract_methods_section_from_full_text(full_text_clean)

    paper_text_one = {
        "title": paper["title"],
        "pdf_key": paper["pdf_key"],
        "pdf_path": paper["pdf_path"],
        "full_text": full_text_clean,
        "methods": methods_text_clean,
    }

    repair_row = {
        "pdf_key": paper["pdf_key"],
        "title": paper["title"],
        "repair_log": json.dumps(repair_log, ensure_ascii=False)
    }

    return paper_text_one, repair_row

def extract_one_paper_materials(paper_text_one):
    result = extract_assay_materials_from_full_paper(paper_text_one["full_text"])
    df_materials_one = pd.DataFrame(result["materials"])

    if df_materials_one.empty:
        return pd.DataFrame(
            columns=["pdf_key", "title", "material", "material_name", "composition", "media"]
        )

    df_materials_one.insert(0, "pdf_key", paper_text_one["pdf_key"])
    df_materials_one.insert(1, "title", paper_text_one["title"])

    df_materials_one["material"] = df_materials_one.apply(
        lambda row: (
            f"{row['material_name']} [{row['media']}]"
            if pd.notna(row["media"]) and str(row["media"]).strip() not in ["", "None"]
            else row["material_name"]
        ),
        axis=1
    )

    cols = ["pdf_key", "title", "material", "material_name", "composition", "media"]
    return df_materials_one[[c for c in cols if c in df_materials_one.columns]]

def extract_one_paper_images(paper, layout_model, img_dir):
    df_images_one = extract_figures_tables_with_caption_merging(
        pdf_file_path=paper["pdf_path"],
        key=paper["pdf_key"],
        img_dir=img_dir,
        model=layout_model,
        dpi=300,
        debug=False
    )

    if df_images_one.empty:
        df_images_one = pd.DataFrame(columns=["image_path", "caption", "image_property", "page"])

    df_images_one["pdf_key"] = paper["pdf_key"]
    df_images_one["title"] = paper["title"]
    return df_images_one

def extract_one_paper_pchem(
    paper_text_one,
    df_materials_one,
    df_images_one,
    client,
    chat_model_name
):
    if df_materials_one is None or df_materials_one.empty:
        return (
            pd.DataFrame(columns=["material", "row_id", "page", "image_property", "caption", "image_path", "pdf_key"]),
            pd.DataFrame(),
            pd.DataFrame()
        )

    df_pchem_candidates_one = build_pchem_candidates_df(
        df_materials=df_materials_one,
        df_images=df_images_one,
        client=client,
        model=chat_model_name
    )

    if df_pchem_candidates_one.empty:
        df_pchem_candidates_one = pd.DataFrame(
            columns=["material", "row_id", "page", "image_property", "caption", "image_path"]
        )

    df_pchem_candidates_one["pdf_key"] = paper_text_one["pdf_key"]

    df_pchem_one, df_pchem_claims_one = extract_pchem_data_for_materials_in_one_paper(
        df_materials_one=df_materials_one,
        df_pchem_candidates_one=df_pchem_candidates_one,
        full_text_one=paper_text_one["full_text"],
        client=client,
        model=chat_model_name,
        temperature=0,
        top_n_candidates=6,
        max_snippets=12,
        page_window=1,
        return_claims=True
    )

    df_pchem_one = clean_pchem_output(df_pchem_one)

    if not df_pchem_one.empty and "title" not in df_pchem_one.columns:
        df_pchem_one.insert(1, "title", paper_text_one["title"])

    if not df_pchem_claims_one.empty and "title" not in df_pchem_claims_one.columns:
        df_pchem_claims_one["title"] = paper_text_one["title"]

    return df_pchem_candidates_one, df_pchem_one, df_pchem_claims_one

def extract_one_paper_tox(
    paper_text_one,
    df_pchem_one,
    df_images_one,
    client,
    chat_model_name
):
    if df_pchem_one is None or df_pchem_one.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    tox_event_rows = []

    df_tox_targets_one = build_tox_targets_from_pchem_for_one_paper(df_pchem_one)
    if df_tox_targets_one.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    selector_result_one = select_tox_figures_tables_from_captions(
        df_tox_targets_one=df_tox_targets_one,
        df_images_one=df_images_one,
        client=client,
        model=chat_model_name,
        temperature=0,
        event_rows=tox_event_rows,
        pdf_key=paper_text_one["pdf_key"],
        title=paper_text_one["title"],
        material="MULTI_TARGETS",
    )

    df_tox_selector_one = tox_selector_result_to_df(selector_result_one)

    df_tox_candidates_one = build_tox_candidates_df(
        df_tox_selector_one=df_tox_selector_one,
        df_images_one=df_images_one
    )

    if not df_tox_candidates_one.empty:
        df_tox_candidates_one = ensure_material_in_tox_candidates(
            df_tox_candidates_one=df_tox_candidates_one,
            df_tox_targets_one=df_tox_targets_one
        )
        df_tox_candidates_one["pdf_key"] = paper_text_one["pdf_key"]
        df_tox_candidates_one["title"] = paper_text_one["title"]

    df_tox_one, df_tox_claims_one = extract_tox_data_for_materials_in_one_paper(
        df_tox_targets_one=df_tox_targets_one,
        df_tox_candidates_one=df_tox_candidates_one,
        full_text_one=paper_text_one["full_text"],
        methods_text_one=paper_text_one["methods"],
        client=client,
        model=chat_model_name,
        temperature=0,
        top_n_candidates=4,
        max_snippets=28,
        page_window=1,
        return_claims=True,
        event_rows=tox_event_rows,
        title=paper_text_one["title"],
    )

    df_tox_one = clean_tox_output(df_tox_one)

    if not df_tox_one.empty:
        if "title" in df_tox_one.columns:
            df_tox_one["title"] = paper_text_one["title"]
        else:
            df_tox_one.insert(1, "title", paper_text_one["title"])

    if not df_tox_claims_one.empty:
        df_tox_claims_one["title"] = paper_text_one["title"]

    tox_event_log_df = _build_tox_event_log_df(tox_event_rows)

    return df_tox_candidates_one, df_tox_one, df_tox_claims_one, tox_event_log_df

def run_pipeline_in_batches(
    papers,
    selected_collection,
    output_dir,
    img_dir,
    tmp_dir,
    client,
    chat_model_name,
    layout_model,
    batch_size=10,
):
    run_ts = _now_kst_str()
    timing_rows = []

    # Fix the execution order first
    # - Main run: sort by pdf_key
    # - Rerun: prefer existing paper_idx when available
    papers_df = pd.DataFrame(papers).copy()

    if papers_df.empty:
        return {
            "final_mat": pd.DataFrame(),
            "final_pchem": pd.DataFrame(),
            "final_pchem_claims": pd.DataFrame(),
            "final_tox": pd.DataFrame(columns=TOX_RECORD_COLUMNS),
            "final_tox_claims": pd.DataFrame(columns=TOX_CLAIM_COLUMNS),
            "final_tox_event_log": pd.DataFrame(columns=TOX_EVENT_LOG_COLUMNS),
            "timing_rows": pd.DataFrame(),
            "run_ts": run_ts,
        }

    # For reruns, prefer the manifest-based paper_idx order when available
    if "paper_idx" in papers_df.columns:
        papers_df["paper_idx"] = pd.to_numeric(papers_df["paper_idx"], errors="coerce")
        if papers_df["paper_idx"].notna().all():
            papers_df = papers_df.sort_values(["paper_idx", "pdf_key"], kind="stable").reset_index(drop=True)
        else:
            papers_df = papers_df.sort_values("pdf_key", kind="stable").reset_index(drop=True)
    else:
        papers_df = papers_df.sort_values("pdf_key", kind="stable").reset_index(drop=True)

    # Papers actually used for execution
    papers_run = papers_df.to_dict(orient="records")

    # Batch stage 0: Save the manifest
    # - Record papers in the same order as execution
    manifest_df = papers_df.copy()
    manifest_df["paper_idx"] = manifest_df.index + 1
    manifest_df["batch_id"] = (manifest_df.index // batch_size) + 1

    manifest_path = os.path.join(
        output_dir,
        f"{_safe_collection_name(selected_collection)}_manifest_{run_ts}.xlsx"
    )
    clean_df_for_excel(manifest_df).to_excel(manifest_path, index=False)
    print(f"[MANIFEST SAVED] {manifest_path}")

    # For accumulating all results
    all_mat_list = []
    all_pchem_list = []
    all_pchem_claims_list = []
    all_tox_list = []
    all_tox_claims_list = []
    all_tox_event_log_list = []

    total_batches = int(math.ceil(len(papers_run) / batch_size))

    for batch_id, batch_papers in enumerate(chunk_list(papers_run, batch_size), start=1):
        print(f"\n========== Batch {batch_id}/{total_batches} START ==========")

        # batch containers
        batch_paper_texts = []
        batch_repair_rows = []
        batch_mat_list = []
        batch_images_map = {}
        batch_pchem_candidates_list = []
        batch_pchem_list = []
        batch_pchem_claims_list = []
        batch_tox_candidates_list = []
        batch_tox_list = []
        batch_tox_claims_list = []
        batch_tox_event_log_list = []

        batch_total_start_dt = _now_kst_dt_str()
        batch_total_start_ts = time.perf_counter()

        # Batch stage 1: Extract full text and Methods section
        stage_start_dt = _now_kst_dt_str()
        stage_start_ts = time.perf_counter()

        for i, paper in enumerate(batch_papers, start=1):
            print(f"[Batch {batch_id}] FullText/Methods {i}/{len(batch_papers)} | {paper['pdf_key']}")
            try:
                paper_text_one, repair_row = extract_one_paper_text_bundle(
                    paper=paper,
                    client=client,
                    chat_model_name=chat_model_name
                )
                batch_paper_texts.append(paper_text_one)
                batch_repair_rows.append(repair_row)
            except Exception as e:
                print(f"[ERROR] FullText/Methods failed | {paper['pdf_key']} | {e}")

        record_time(
            timing_rows=timing_rows,
            batch_id=batch_id,
            stage="Full text/Methods extraction",
            start_dt=stage_start_dt,
            start_ts=stage_start_ts,
            n_papers=len(batch_paper_texts)
        )

        # Batch stage 2: Extract assayed nanomaterials
        stage_start_dt = _now_kst_dt_str()
        stage_start_ts = time.perf_counter()

        for i, paper_text_one in enumerate(batch_paper_texts, start=1):
            print(f"[Batch {batch_id}] Mat {i}/{len(batch_paper_texts)} | {paper_text_one['pdf_key']}")
            try:
                df_materials_one = extract_one_paper_materials(paper_text_one)
                if not df_materials_one.empty:
                    batch_mat_list.append(df_materials_one)
            except Exception as e:
                print(f"[ERROR] Mat failed | {paper_text_one['pdf_key']} | {e}")

        df_batch_mat = concat_or_empty(
            batch_mat_list,
            columns=["pdf_key", "title", "material", "material_name", "composition", "media"]
        )

        record_time(
            timing_rows=timing_rows,
            batch_id=batch_id,
            stage="Material extraction",
            start_dt=stage_start_dt,
            start_ts=stage_start_ts,
            n_papers=len(batch_paper_texts)
        )

        # Batch stage 3: Save material extraction results
        stage_start_dt = _now_kst_dt_str()
        stage_start_ts = time.perf_counter()

        save_mat_batch(
            output_dir=output_dir,
            selected_collection=selected_collection,
            batch_id=batch_id,
            run_ts=run_ts,
            df_mat=df_batch_mat
        )

        record_time(
            timing_rows=timing_rows,
            batch_id=batch_id,
            stage="Material save",
            start_dt=stage_start_dt,
            start_ts=stage_start_ts,
            n_papers=len(batch_paper_texts)
        )

        # Batch stage 4: Extract figures, tables, and captions
        stage_start_dt = _now_kst_dt_str()
        stage_start_ts = time.perf_counter()

        for i, paper in enumerate(batch_papers, start=1):
            print(f"[Batch {batch_id}] Image/Table {i}/{len(batch_papers)} | {paper['pdf_key']}")
            try:
                df_images_one = extract_one_paper_images(
                    paper=paper,
                    layout_model=layout_model,
                    img_dir=img_dir
                )
                batch_images_map[paper["pdf_key"]] = df_images_one
            except Exception as e:
                print(f"[ERROR] Image/Table failed | {paper['pdf_key']} | {e}")
                batch_images_map[paper["pdf_key"]] = pd.DataFrame(
                    columns=["image_path", "caption", "image_property", "page", "pdf_key", "title"]
                )

        record_time(
            timing_rows=timing_rows,
            batch_id=batch_id,
            stage="Figure/Table, Caption extraction",
            start_dt=stage_start_dt,
            start_ts=stage_start_ts,
            n_papers=len(batch_paper_texts)
        )

        # Batch stage 5: Extract PChem data
        stage_start_dt = _now_kst_dt_str()
        stage_start_ts = time.perf_counter()

        for i, paper_text_one in enumerate(batch_paper_texts, start=1):
            pdf_key = paper_text_one["pdf_key"]
            print(f"[Batch {batch_id}] PChem {i}/{len(batch_paper_texts)} | {pdf_key}")

            df_materials_one = df_batch_mat[df_batch_mat["pdf_key"] == pdf_key].copy()
            df_images_one = batch_images_map.get(pdf_key, pd.DataFrame())

            try:
                df_pchem_candidates_one, df_pchem_one, df_pchem_claims_one = extract_one_paper_pchem(
                    paper_text_one=paper_text_one,
                    df_materials_one=df_materials_one,
                    df_images_one=df_images_one,
                    client=client,
                    chat_model_name=chat_model_name
                )

                if not df_pchem_candidates_one.empty:
                    batch_pchem_candidates_list.append(df_pchem_candidates_one)
                if not df_pchem_one.empty:
                    batch_pchem_list.append(df_pchem_one)
                if not df_pchem_claims_one.empty:
                    batch_pchem_claims_list.append(df_pchem_claims_one)

            except Exception as e:
                print(f"[ERROR] PChem failed | {pdf_key} | {e}")

        df_batch_pchem_candidates = concat_or_empty(batch_pchem_candidates_list)
        df_batch_pchem = concat_or_empty(batch_pchem_list)
        df_batch_pchem_claims = concat_or_empty(batch_pchem_claims_list)

        record_time(
            timing_rows=timing_rows,
            batch_id=batch_id,
            stage="PChem extraction",
            start_dt=stage_start_dt,
            start_ts=stage_start_ts,
            n_papers=len(batch_paper_texts)
        )

        # Batch stage 6: Save PChem extraction results
        stage_start_dt = _now_kst_dt_str()
        stage_start_ts = time.perf_counter()

        save_pchem_batch(
            output_dir=output_dir,
            selected_collection=selected_collection,
            batch_id=batch_id,
            run_ts=run_ts,
            df_pchem=df_batch_pchem,
            df_pchem_claims=df_batch_pchem_claims
        )

        record_time(
            timing_rows=timing_rows,
            batch_id=batch_id,
            stage="PChem save",
            start_dt=stage_start_dt,
            start_ts=stage_start_ts,
            n_papers=len(batch_paper_texts)
        )

        # Batch stage 7: Extract toxicity data
        stage_start_dt = _now_kst_dt_str()
        stage_start_ts = time.perf_counter()

        for i, paper_text_one in enumerate(batch_paper_texts, start=1):
            pdf_key = paper_text_one["pdf_key"]
            print(f"[Batch {batch_id}] Tox {i}/{len(batch_paper_texts)} | {pdf_key}")

            df_pchem_one = df_batch_pchem[df_batch_pchem["pdf_key"] == pdf_key].copy() if not df_batch_pchem.empty else pd.DataFrame()
            df_images_one = batch_images_map.get(pdf_key, pd.DataFrame())

            try:
                df_tox_candidates_one, df_tox_one, df_tox_claims_one, tox_event_log_one = extract_one_paper_tox(
                    paper_text_one=paper_text_one,
                    df_pchem_one=df_pchem_one,
                    df_images_one=df_images_one,
                    client=client,
                    chat_model_name=chat_model_name
                )

                if not df_tox_candidates_one.empty:
                    batch_tox_candidates_list.append(df_tox_candidates_one)
                if not df_tox_one.empty:
                    batch_tox_list.append(df_tox_one)
                if not df_tox_claims_one.empty:
                    batch_tox_claims_list.append(df_tox_claims_one)
                if not tox_event_log_one.empty:
                    batch_tox_event_log_list.append(tox_event_log_one)

            except Exception as e:
                print(f"[ERROR] Tox failed | {pdf_key} | {e}")

        df_batch_tox_candidates = concat_or_empty(batch_tox_candidates_list)
        df_batch_tox = concat_or_empty(batch_tox_list, columns=TOX_RECORD_COLUMNS)
        df_batch_tox_claims = concat_or_empty(batch_tox_claims_list, columns=TOX_CLAIM_COLUMNS)
        df_batch_tox_event_log = concat_or_empty(batch_tox_event_log_list, columns=TOX_EVENT_LOG_COLUMNS)

        record_time(
            timing_rows=timing_rows,
            batch_id=batch_id,
            stage="Tox extraction",
            start_dt=stage_start_dt,
            start_ts=stage_start_ts,
            n_papers=len(batch_paper_texts)
        )

        # Batch stage 8: Postprocess toxicity outputs
        stage_start_dt = _now_kst_dt_str()
        stage_start_ts = time.perf_counter()

        tox_postprocess_map_df = pd.DataFrame()
        tox_cell_name_postprocess_map_df = pd.DataFrame()

        if not df_batch_tox.empty:
            batch_tox_event_rows = (
                df_batch_tox_event_log.to_dict(orient="records")
                if df_batch_tox_event_log is not None and not df_batch_tox_event_log.empty
                else []
            )
            
            df_batch_tox, df_batch_tox_claims, tox_postprocess_map_df, tox_cell_name_postprocess_map_df = run_tox_postprocess_cell_and_assay(
                df_tox_records=df_batch_tox,
                df_tox_claims=df_batch_tox_claims,
                client=client,
                model=chat_model_name,
                temperature=0,
                batch_size=80,
                cell_name_batch_size=80,
                event_rows=batch_tox_event_rows,
                pdf_key="MULTI",
                title=f"Batch {batch_id}",
                material="MULTI",
            )
            
            df_batch_tox_event_log = _build_tox_event_log_df(batch_tox_event_rows)

        record_time(
            timing_rows=timing_rows,
            batch_id=batch_id,
            stage="Tox Post-process",
            start_dt=stage_start_dt,
            start_ts=stage_start_ts,
            n_papers=len(batch_paper_texts)
        )

        # Batch stage 9: Save toxicity extraction results
        stage_start_dt = _now_kst_dt_str()
        stage_start_ts = time.perf_counter()

        save_tox_batch(
            output_dir=output_dir,
            selected_collection=selected_collection,
            batch_id=batch_id,
            run_ts=run_ts,
            df_tox=df_batch_tox,
            df_tox_claims=df_batch_tox_claims,
            df_tox_event_log=df_batch_tox_event_log,
            tox_postprocess_map_df=tox_postprocess_map_df,
            tox_cell_name_postprocess_map_df=tox_cell_name_postprocess_map_df
        )

        record_time(
            timing_rows=timing_rows,
            batch_id=batch_id,
            stage="Tox save",
            start_dt=stage_start_dt,
            start_ts=stage_start_ts,
            n_papers=len(batch_paper_texts)
        )

        # Batch stage 10: Record total batch runtime
        record_time(
            timing_rows=timing_rows,
            batch_id=batch_id,
            stage="Batch total",
            start_dt=batch_total_start_dt,
            start_ts=batch_total_start_ts,
            n_papers=len(batch_paper_texts)
        )

        # Update and save the per-batch timing workbook after each batch
        save_time_workbook(
            timing_rows=timing_rows,
            output_dir=output_dir,
            selected_collection=selected_collection,
            run_ts=run_ts
        )

        # Accumulate rows for final merge
        if not df_batch_mat.empty:
            all_mat_list.append(df_batch_mat)

        if not df_batch_pchem.empty:
            all_pchem_list.append(df_batch_pchem)
        if not df_batch_pchem_claims.empty:
            all_pchem_claims_list.append(df_batch_pchem_claims)

        if not df_batch_tox.empty:
            all_tox_list.append(df_batch_tox)
        if not df_batch_tox_claims.empty:
            all_tox_claims_list.append(df_batch_tox_claims)
        if not df_batch_tox_event_log.empty:
            all_tox_event_log_list.append(df_batch_tox_event_log)

        # Clean up memory
        gc.collect()
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass

        print(f"========== Batch {batch_id}/{total_batches} END ==========")

    # Final merge
    final_mat = concat_or_empty(all_mat_list)
    final_pchem = concat_or_empty(all_pchem_list)
    final_pchem_claims = concat_or_empty(all_pchem_claims_list)
    final_tox = concat_or_empty(all_tox_list, columns=TOX_RECORD_COLUMNS)
    final_tox_claims = concat_or_empty(all_tox_claims_list, columns=TOX_CLAIM_COLUMNS)
    final_tox_event_log = concat_or_empty(all_tox_event_log_list, columns=TOX_EVENT_LOG_COLUMNS)

    if not final_mat.empty:
        save_final_mat(output_dir, selected_collection, run_ts, final_mat)

    if not final_pchem.empty:
        save_final_pchem(output_dir, selected_collection, run_ts, final_pchem, final_pchem_claims)

    if not final_tox.empty or not final_tox_event_log.empty:
        save_final_tox(
            output_dir=output_dir,
            selected_collection=selected_collection,
            run_ts=run_ts,
            df_tox=final_tox,
            df_tox_claims=final_tox_claims,
            df_tox_event_log=final_tox_event_log
        )

    # Save the final timing workbook one more time
    save_time_workbook(
        timing_rows=timing_rows,
        output_dir=output_dir,
        selected_collection=selected_collection,
        run_ts=run_ts
    )

    return {
        "final_mat": final_mat,
        "final_pchem": final_pchem,
        "final_pchem_claims": final_pchem_claims,
        "final_tox": final_tox,
        "final_tox_claims": final_tox_claims,
        "final_tox_event_log": final_tox_event_log,
        "timing_rows": pd.DataFrame(timing_rows),
        "run_ts": run_ts,
    }

In [ ]:
# Step 9: Define selected-paper rerun helper functions

def parse_paper_idx_spec(idx_spec):
    """
    Convert a paper_idx selection string/list into a list of integers.
    Example:
      "3" -> [3]
      "3,5,8-10" -> [3,5,8,9,10]
      [3,5,8] -> [3,5,8]
    """
    if idx_spec is None:
        return []

    if isinstance(idx_spec, (list, tuple, set)):
        out = []
        for x in idx_spec:
            try:
                out.append(int(x))
            except Exception:
                pass
        return sorted(set(out))

    if isinstance(idx_spec, int):
        return [idx_spec]

    s = str(idx_spec).strip()
    if s == "":
        return []

    out = []
    for token in s.split(","):
        token = token.strip()
        if token == "":
            continue

        if "-" in token:
            a, b = token.split("-", 1)
            try:
                a = int(a.strip())
                b = int(b.strip())
                lo, hi = min(a, b), max(a, b)
                out.extend(range(lo, hi + 1))
            except Exception:
                pass
        else:
            try:
                out.append(int(token))
            except Exception:
                pass

    return sorted(set(out))


def load_manifest_df(manifest_path: str) -> pd.DataFrame:
    """
    Load the manifest xlsx file.
    """
    df = pd.read_excel(manifest_path)

    required_cols = ["paper_idx", "pdf_key", "title", "pdf_path"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Required columns are missing from the manifest: {missing}")

    df["paper_idx"] = pd.to_numeric(df["paper_idx"], errors="coerce")
    df = df[df["paper_idx"].notna()].copy()
    df["paper_idx"] = df["paper_idx"].astype(int)

    return df


def select_papers_from_manifest_by_idx(
    manifest_path: str,
    paper_idxs
):
    """
    Create a papers subset based on paper_idx values in the manifest.
    The return value is a list[dict] that can be passed directly to run_pipeline_in_batches().
    """
    target_idxs = parse_paper_idx_spec(paper_idxs)
    if not target_idxs:
        raise ValueError("No paper_idx values were selected.")

    df_manifest = load_manifest_df(manifest_path)

    df_sel = (
        df_manifest[df_manifest["paper_idx"].isin(target_idxs)]
        .sort_values("paper_idx")
        .reset_index(drop=True)
        .copy()
    )

    if df_sel.empty:
        raise ValueError(f"Selected paper_idx values were not found in the manifest: {target_idxs}")

    print("[SELECTED paper_idx]")
    print(df_sel[["paper_idx", "pdf_key", "title"]])

    return df_sel.to_dict(orient="records")


def select_papers_from_manifest_by_batch_id(
    manifest_path: str,
    batch_ids
):
    """
    Create a papers subset based on batch_id values in the manifest.
    Example: batch_ids=2 or "1,3"
    """
    if isinstance(batch_ids, int):
        batch_ids = [batch_ids]
    elif isinstance(batch_ids, str):
        batch_ids = [int(x.strip()) for x in batch_ids.split(",") if x.strip()]
    else:
        batch_ids = [int(x) for x in batch_ids]

    df_manifest = load_manifest_df(manifest_path)

    if "batch_id" not in df_manifest.columns:
        raise ValueError("The manifest does not contain a batch_id column.")

    df_sel = (
        df_manifest[df_manifest["batch_id"].isin(batch_ids)]
        .sort_values(["batch_id", "paper_idx"])
        .reset_index(drop=True)
        .copy()
    )

    if df_sel.empty:
        raise ValueError(f"Selected batch_id values were not found in the manifest: {batch_ids}")

    print("[SELECTED batch_id / paper_idx]")
    print(df_sel[["batch_id", "paper_idx", "pdf_key", "title"]])

    return df_sel.to_dict(orient="records")

## Step 9.1: Standard batch execution

Download PDFs once, run the full batch pipeline, and expose merged output tables in the notebook.

In [ ]:
# Step 9.1: Execute the standard batch pipeline
overall_start_dt = _now_kst_dt_str()
overall_start_ts = time.perf_counter()
print("[Pipeline Batch Runner] Overall start:", overall_start_dt)

# Step 9.1a: Download PDFs once for the standard batch run
pdf_download_start_dt = _now_kst_dt_str()
pdf_download_start_ts = time.perf_counter()

papers = get_all_pdfs_from_zotero(collection_name=selected_collection)
#papers = papers[50:53]

pdf_download_elapsed = round(time.perf_counter() - pdf_download_start_ts, 3)
print(f"[PDF DOWNLOAD DONE] {len(papers)} papers | {pdf_download_elapsed} sec")

# Step 9.1b: Run the standard batch pipeline
BATCH_SIZE = 10

batch_results = run_pipeline_in_batches(
    papers=papers,
    selected_collection=selected_collection,
    output_dir=output_dir,
    img_dir=img_dir,
    tmp_dir=tmp_dir,
    client=client,
    chat_model_name=CHAT_MODEL_NAME,
    layout_model=model,   # layoutparser model
    batch_size=BATCH_SIZE
)

overall_elapsed = round(time.perf_counter() - overall_start_ts, 3)
print("[Pipeline Batch Runner] Overall end  :", _now_kst_dt_str())
print("[Pipeline Batch Runner] Overall elapsed (sec):", overall_elapsed)

# For checking final results inside the notebook
final_mat = batch_results["final_mat"]
final_pchem = batch_results["final_pchem"]
final_pchem_claims = batch_results["final_pchem_claims"]
final_tox = batch_results["final_tox"]
final_tox_claims = batch_results["final_tox_claims"]
df_batch_timing = batch_results["timing_rows"]

## Step 9.2: Re-run selected papers from a saved manifest

Use this optional rerun template only after updating the project name, collection name, manifest path, and selected paper or batch IDs.

The values in this section are examples from a prior run. Before executing this cell, replace `project_name`, `selected_collection`, `manifest_path`, and `rerun_papers` or `batch_ids` with values from your own output manifest.


In [ ]:
# Step 9.2: Execute a selected-paper rerun
rerun_start_dt = _now_kst_dt_str()
rerun_start_ts = time.perf_counter()
print("[Pipeline Batch Runner] Rerun start:", rerun_start_dt)

# Step 9.2: Configure selected-paper rerun from a saved manifest
# Replace the example values below before running this optional cell.
# Example project label from a previous run. Replace with your own project label.
project_name = "01 GPT-5.4_Oxide_A2018_5_5"
# Example Zotero collection name from a previous run. Replace with your own collection name.
selected_collection = "Oxide_A2018_5_5"

# Example manifest path from a previous run. Replace with the manifest generated by your own batch run.
manifest_path = "../output/Oxide_A2018_5_5_01 GPT-5.4_Oxide_A2018_5_5_20260417_234936/Oxide_A2018_5_5_manifest_20260417_235613.xlsx"

# Parent output directory containing the manifest
output_manifest_dir = os.path.dirname(manifest_path)
run_folder_name = os.path.basename(output_manifest_dir)

# Example 1: directly specify paper_idx values
rerun_papers = select_papers_from_manifest_by_idx(
    manifest_path=manifest_path,
    paper_idxs="16,66"
)

# Example 2: use this option to specify batch_id values
# rerun_papers = select_papers_from_manifest_by_batch_id(
#     manifest_path=manifest_path,
#     batch_ids=2
# )

rerun_tag = f"RERUN_{_now_kst_str()}"
rerun_output_dir = os.path.join(output_manifest_dir, rerun_tag)
rerun_img_dir = os.path.join("../image/", run_folder_name, rerun_tag)
rerun_tmp_dir = os.path.join("../tmp/", run_folder_name, rerun_tag)
#rerun_output_dir = os.path.join(output_dir, rerun_tag)
#rerun_img_dir = os.path.join(img_dir, rerun_tag)
#rerun_tmp_dir = os.path.join(tmp_dir, rerun_tag)

os.makedirs(rerun_output_dir, exist_ok=True)
os.makedirs(rerun_img_dir, exist_ok=True)
os.makedirs(rerun_tmp_dir, exist_ok=True)

BATCH_SIZE = 10

rerun_results = run_pipeline_in_batches(
    papers=rerun_papers,
    selected_collection=selected_collection,
    output_dir=rerun_output_dir,
    img_dir=rerun_img_dir,
    tmp_dir=rerun_tmp_dir,
    client=client,
    chat_model_name=CHAT_MODEL_NAME,
    layout_model=model,
    batch_size=min(BATCH_SIZE, len(rerun_papers))
)

rerun_elapsed = round(time.perf_counter() - rerun_start_ts, 3)
print("[Pipeline Batch Runner] Rerun end  :", _now_kst_dt_str())
print("[Pipeline Batch Runner] Rerun elapsed (sec):", rerun_elapsed)

# For checking final results inside the notebook
rerun_mat = rerun_results["final_mat"]
rerun_pchem = rerun_results["final_pchem"]
rerun_pchem_claims = rerun_results["final_pchem_claims"]
rerun_tox = rerun_results["final_tox"]
rerun_tox_claims = rerun_results["final_tox_claims"]
rerun_tox_event_log = rerun_results["final_tox_event_log"]
df_rerun_timing = rerun_results["timing_rows"]